# Randomized Implementations of the Platonic-Solid POVMs — Walkthrough

A companion walkthrough for `code/randomized_implementations.py` and the seven sibling modules it
binds (`randomized_{core,field,scalars,twojobs,obstruction,decker,fragments}.py`) — the
verification suite behind Section 5.2.3, Appendix D, and Appendix F.3 of the thesis.

The thesis distinguishes two protocols that both carried the name *randomized implementation*:

- **R1 — randomized-projective.** Draw a rotation $g$ uniformly from a finite group $G$, apply
  $U_g$, apply one fixed *alignment* $A$ (vertex axis $\to \hat z$), read out in the $Z$ basis.
  No ancillas; needs antipodal vertex pairs, so the tetrahedral SIC is excluded;
  estimator-channel factor $T_{zz}$. (The literature's "randomized measurements" primitive.)
- **R2 — twirled-native.** Draw $g$, apply $U_g$, then run the *fixed* native
  (Decker/Naimark) circuit and relabel outcomes by $g$ in classical post-processing. Ancillas;
  works for all five solids, SIC included; estimator-channel factor $\operatorname{tr}T/3$.
  (The literature's measurement/readout twirling.)

**One reframing carries the whole suite: a protocol is *which maps you average*.** It is applied
five times below.

1. **Two estimator channels (§1).** R1's draw sits in the outcome probability *and* in the snapshot, so the
   average twirls the composition readout$\,\circ\,$noise — the rank-one $\hat z\hat z^\top T$
   conjugated into the solid's frame. R2's readout never depends on the draw, so the average
   twirls the noise *alone*. One lemma, two conjugated objects, two scalars — the
   estimator-channel factor identifies the protocol.
2. **Two jobs (§2).** The drawn set must *be* the POVM (its orbit sweeps the vertex axes
   uniformly: realization) and must *average to a scalar* (irreducibility: the twirl). Two
   independent properties of the same set of maps, and they come apart — the bars cross at the
   icosahedron.
3. **Exactness (§3).** Every map has to be compiled from the gate set, so it carries the gate
   field with it. Direction (vertices outside the gates' real field) and weight (effect traces
   outside the dyadic ring) between them convict all five solids; what survives for the
   octahedron is a classical coin, not a circuit.
4. **A wrong list (§3 cont.).** Believe one vertex list while the device measures another, and
   the same average launders the entire mismatch into a single overlap $\kappa$ — zero offset,
   never a bias, a $1/\kappa^2$ shot premium. Decker's outcome order, priced per solid.
5. **Gate noise (§1c).** Put the noise *inside* the drawn word and there is no longer one fixed
   map to average — the noise is correlated with $g$ and Schur is silent. What decides the
   residual's order in $\gamma$ is the word set's prefix multiset: the reframing earns its keep
   by failing informatively.

**Zero RNG anywhere — and that is the pedagogy.** Nothing here samples. Every "randomized"
quantity is a finite group average computed as an explicit sum: float64 linear algebra asserted
at $10^{-9}$ (the algebraic numbers involved are well separated), SymPy exactly where field
membership, or a claim quantified over *arbitrary* noise, gate noise or state, *is* the claim.
Where a verdict is an identity (which
element a product is, whether two vertices are antipodes) it is decided a second time by
canonical form over a small number field, and the two answers are required to agree.

**References:**
- Decker, Janzing & Beth, *Quantum circuits for single-qubit measurements corresponding to platonic solids* (2004); Decker, *Implementation of group-covariant POVMs* (2005) — the native circuits, rebuilt in §3 cont.
- Elben, Flammia, Huang, Kueng, Preskill, Vermersch & Zoller, *The randomized measurement toolbox* (2022) — randomized-projective's literature home
- Chen, Yu, Zeng & Flammia, *Robust shadow estimation* (2021) — calibration under a measurement twirl, twirled-native's literature home
- Nguyen, Bönsel, Steinberg & Gühne, *Optimising shadow tomography with generalised measurements* (2022) — Platonic-solid POVM shadows (the closest adjacent work)
- Gross, Audenaert & Eisert, *Evenly distributed unitaries* (2007) — $2T$ as the minimal group 2-design in $d=2$
- Bannai, Navarro, Rizo & Tiep, *Unitary t-groups* (2018); Roy & Scott, *Unitary designs and codes* (2009) — the design ladder
- Hirao, Nozaki & Tasaka (2025) — the spherical-design side of the same group orbits
- Conway & Smith, *On Quaternions and Octonions* (2003) — the $\tau$/$\sigma$ convention

## The map, and the contract

`randomized_implementations.py` is the entry point: its `main()` *is* the run order, and the
banners it prints are the suite's Sections 0–5. The bindings live in seven flat siblings, a DAG
with tools below checks:

| notebook | suite section | finding | modules doing the work |
|---|---|---|---|
| §0 canonical data | 0 | the npz pins | `core`, check in `scalars` |
| §1 two protocols, two estimator channels | 1 | 1 + 2 | `core` (channels), `field` (exact kit), `scalars` |
| §1b calibration mismatch | 1 | 1 | `scalars` (+ a lazy, write-free `shadow_experiments` import) |
| §1c gate noise | 1 | 6 | `scalars` |
| §1d the alignment | 1 | — | `scalars` |
| §2 the two jobs | 2 | 4 | `twojobs` (+ sweep machinery in `core`) |
| §3 the exactness obstruction | 3 | 3 | `obstruction` (+ symbolic layer in `core`, theorem in `field`) |
| §3 cont. Decker's circuits | 3 cont. | 5 | `decker` |
| §4 the ledger | 4 | — | `fragments` |
| §5 the design ladder | 5 | remark | `twojobs` |
| §6 the receipts | — | — | the entry point's `main()` |

**The contract of this notebook.**

- Every module definition shown below is lifted *verbatim, mechanically* (`ast`, at build time)
  from whichever of the eight files owns it — nothing is hand-copied, so the cells track module
  edits by construction. Hand-written code is only glue: imports, one `DATA` adaptation, small
  demos and comparisons.
- One adaptation: the module resolves its data directory next to itself via `__file__`, which a
  notebook lacks. The setup cell redefines the same trailing-slash string relative to the cwd
  (`code/`). Everything else is untouched.
- The notebook **writes nothing** — and enforces it. The first code cell arms a runtime
  tripwire (an audit hook) refusing any write-mode open under the repository, so the claim
  holds for every idiom and every call into the imported suite, not just the patterns a build
  lint can see. Importing any module of the suite — or calling `main()` — writes nothing; the
  six thesis fragments are emitted only when the entry point runs as a script.
- The cells below select for depth on the through-line rather than mirroring every check; the
  final cell closes the gap four ways: it re-hashes the modules (and the builder) against
  build-time sha256 pins, so a committed notebook that has drifted behind a later edit cannot
  execute quietly; it spot-checks lifted primitives against the imported production modules;
  it re-derives the ledger's `spec_sheet()`; and it runs the suite's `main()` end to end, so
  every check not staged here still runs and still passes, in this very notebook. Only the
  fragment writers themselves stay unexecuted.

In [1]:
# === Setup (notebook glue): imports + the one adaptation ===

import itertools
import math
from pathlib import Path

import numpy as np
import sympy as sp
from sympy import I as sI
from sympy import Matrix, Rational, sqrt

# The one adaptation: randomized_core resolves DATA next to itself via
# __file__, which a notebook lacks. Same trailing-slash string semantics,
# with the notebook executing from code/.
DATA = f"{Path('data')}/"

# The exact-rotation layer rebuilds T/O/I from main.py's canonical
# quaternions; importing main is write-free (its writers sit under __main__).
from main import geometric_group

In [2]:
# === The write tripwire (glue): "writes nothing", enforced at runtime ===

# Every Python-level file write funnels through the `open` audit event, so
# one hook turns the header's claim into a runtime guarantee covering every
# idiom (open, np.savez, Path.write_text, savefig, ...) and every call into
# the imported suite -- not just patterns a static lint can see. Scope:
# refuse write-mode opens under the repository; __pycache__ is exempt (a
# fresh clone's first import compiles bytecode); reads stay free.
import os
import sys

_REPO = Path.cwd().resolve().parent        # the notebook executes in code/


def _no_tree_writes(event, args):
    if event != "open":
        return
    path, mode, flags = args
    if isinstance(mode, str):
        writey = bool(set(mode) & set("wxa+"))
    else:                                  # os.open passes mode=None
        writey = bool(flags & (os.O_WRONLY | os.O_RDWR))
    if not writey or path is None:
        return
    try:
        p = Path(os.fsdecode(path))
    except TypeError:                      # e.g. an integer fd: not a path
        return
    p = p if p.is_absolute() else Path.cwd() / p
    if "__pycache__" not in p.parts and p.resolve().is_relative_to(_REPO):
        raise RuntimeError(f"write tripwire: refusing write-mode open of {p}")


sys.addaudithook(_no_tree_writes)

# ...and the proof it is armed, in this committed output: a write-mode open
# under code/ must die, refused before the file is created.
try:
    open("data/_tripwire_probe", "w").close()
except RuntimeError:
    print("write tripwire armed: a write-mode open under code/ was refused")
else:
    raise AssertionError("tripwire failed to arm")
assert not Path("data/_tripwire_probe").exists()

write tripwire armed: a write-mode open under code/ was refused


## 0. The canonical data, and the generic probe

Inputs are the thesis's own symbolic exports in `code/data/`: `povm_*.npz` (Bloch vertices and
effects, in the published numbering of the POVM vertex table), `group_{T,O,I}.npz` (the rotation
groups) and `group_2{T,O,I}.npz` (the binary groups with their synthesized circuits — the rows of
Appendix A, i.e. the *atlas words* the coin and the draw will be priced in).

One fixed measurement-side noise $r \mapsto Tr + t$ serves as the probe for every float check.
It is chosen *generic* — the two candidate scalars distinct, offset nonzero, $T$ anisotropic —
so no depolarizing verdict below can pass by accident; the next cell asserts exactly that
(the same three asserts `main()` opens with). Section 1a will then remove the probe from the
load-bearing path altogether by quantifying over every $(T, t)$.

In [3]:
# === Canonical constants and loaders (lifted from randomized_core.py) ===

SOLIDS = ("tetrahedron", "octahedron", "cube", "icosahedron", "dodecahedron")

TAU_SYM = (1 + sqrt(5)) / 2          # golden ratio (Conway & Smith tau)

SIG_SYM = (sqrt(5) - 1) / 2          # inverse golden ratio sigma

# covariance rotation group of each solid's POVM
COVARIANCE = {"tetrahedron": "T", "octahedron": "O", "cube": "O",
              "icosahedron": "I", "dodecahedron": "I"}

# A fixed, generic measurement-side affine noise r -> T r + t, shared verbatim
# with shadow_experiments.py's two-protocol study, whose own channels are
# checked value for value against ours, up to the canonical dual's factor 3.
# Genericity is asserted in main() -- the two candidate scalars differ, the
# offset is nonzero, T is anisotropic -- so no depolarizing check can pass
# vacuously.
T_NOISE = np.array([[0.83, 0.06, -0.11],
                    [-0.04, 0.71, 0.09],
                    [0.12, -0.07, 0.62]])

t_NOISE = np.array([0.05, -0.03, 0.17])

PAULI = np.array([[[0, 1], [1, 0]],
                  [[0, -1j], [1j, 0]],
                  [[1, 0], [0, -1]]], dtype=complex)


def load_vertices(solid):
    """Bloch vertices (V, 3) of the solid's POVM from the canonical npz."""
    return np.load(DATA + f"povm_{solid}.npz")["vertices"]


def load_elements(solid):
    """POVM elements (V, 2, 2) from the canonical npz."""
    return np.load(DATA + f"povm_{solid}.npz")["elements"]


def load_rotations(g):
    """SO(3) rotations (|G|, 3, 3) of the polyhedral group g in {T, O, I}."""
    return np.load(DATA + f"group_{g}.npz")["rotations"]


def load_atlas(g):
    """The binary group 2g's atlas: unitaries + synthesized circuits."""
    d = np.load(DATA + f"group_2{g}.npz")
    return {k: d[k] for k in d.files}


def rotation_from_unitary(U):
    """SO(3) rotation of a 2x2 unitary: R_ij = tr(sigma_i U sigma_j U+)/2."""
    return np.real(np.einsum("iab,bc,jcd,ad->ij", PAULI, U, PAULI, U.conj())) / 2


def rot_key(R, decimals=9):
    """Hashable rounded key of a rotation matrix: the float-grid identity trick.

    What binds is the distance from an exact entry to a rounding BOUNDARY, not
    the grid spacing.  T's and O's entries are {0, +-1} and land on the grid, so
    they get the full half-spacing, 5e-10.  I is the only tight group:
    tau/2 = 0.809016994|3749... and sigma/2 = 0.309016994|3749... sit 1.25e-10
    from a boundary -- four of I's nine entries, tied because the two differ by
    exactly 1/2, itself a grid multiple.  Measured error over every call site
    (raw npz, rotation_from_unitary, products R_i R_j, conjugates R_h R_i R_h^T)
    is at most 4.4e-16, so the margin is 2.8e5x and I is what binds it.

    exact_rotations is the companion that needs no grid at all; it is a second
    opinion on the float sweep, never a replacement for it.
    """
    return tuple(np.round(np.asarray(R), decimals).flatten().tolist())

In [4]:
# === The cast, and the probe's genericity (glue) ===

print(f"{'solid':14s} {'V':>3s} {'grp':>4s} {'|G|':>4s} {'|2G|':>5s}")
for solid in SOLIDS:
    s, g = load_vertices(solid), COVARIANCE[solid]
    print(f"{solid:14s} {len(s):3d} {g:>4s} {len(load_rotations(g)):4d}"
          f" {len(load_atlas(g)['unitaries']):5d}")

print(f"\nprobe noise r -> T r + t:  T_zz = {T_NOISE[2, 2]:.6f}"
      f"   tr(T)/3 = {np.trace(T_NOISE) / 3:.6f}")

# main()'s own preamble asserts, restated: the probe is generic
assert abs(T_NOISE[2, 2] - np.trace(T_NOISE) / 3) > 0.05   # scalars distinct
assert np.linalg.norm(t_NOISE) > 0.05                      # offset nonzero
assert np.linalg.norm(T_NOISE - np.trace(T_NOISE) / 3 * np.eye(3)) > 0.05
print("generic: the candidate scalars differ, the offset is nonzero, T is")
print("anisotropic -- no depolarizing verdict below can pass by accident")

solid            V  grp  |G|  |2G|
tetrahedron      4    T   12    24
octahedron       6    O   24    48
cube             8    O   24    48
icosahedron     12    I   60   120
dodecahedron    20    I   60   120

probe noise r -> T r + t:  T_zz = 0.620000   tr(T)/3 = 0.720000
generic: the candidate scalars differ, the offset is nonzero, T is
anisotropic -- no depolarizing verdict below can pass by accident


### 0a. The symbolic twins

Alongside the float layer the suite carries exact twins: the five vertex sets as SymPy vectors,
the four gates whose rotation axes the solids will turn out to inherit ($X$, $Z$, the face gate
$F = HS^\dagger$, the golden gate $\Phi$), and exact Bloch axes/actions. Two subtleties both
live here:

- **Two numberings.** `symbolic_solids()` and the published npz order agree as *sets* (same
  solids, same pose) but number the tetrahedron, cube and icosahedron differently.
  `atlas_vertices()` permutes the symbolic vertices into the published order — the one a reader
  can look up — and anything the thesis prints *with a vertex index* runs on it.
- **Identity questions decided twice.** Wherever a verdict is an identity, a canonical-form
  companion re-decides it and the two must agree; the float pipeline stays in place (each exact
  companion pairs with a value-for-value numeric agreement check, because a boolean exact test
  does not test its own transcription).

`check_canonical_data` then pins the whole data layer: npz vertices $=$ exact solids (with
margin), effects $= \frac1V(\mathrm{Id} + \hat n_k\cdot\vec\sigma)$, gates $=$ `gates.npz`
projectively, the $\pm U$ pairs of each binary group projecting onto exactly the rotation group,
and $T < O$, $T < I$.

In [5]:
# === The symbolic layer (lifted from randomized_core.py) ===

PAULI_SYM = [Matrix([[0, 1], [1, 0]]),
             Matrix([[0, -sI], [sI, 0]]),
             Matrix([[1, 0], [0, -1]])]


def symbolic_solids():
    """The five Platonic vertex sets, exact, in the atlas orientation."""
    s_ico = 1 / sqrt(2 + TAU_SYM)
    c_dod = 1 / sqrt(3)
    S = {}
    S["tetrahedron"] = [Matrix(v) / sqrt(3)
                        for v in [(1, 1, 1), (1, -1, -1), (-1, 1, -1), (-1, -1, 1)]]
    S["octahedron"] = [Matrix(v) for v in
                       [(1, 0, 0), (-1, 0, 0), (0, 1, 0), (0, -1, 0), (0, 0, 1), (0, 0, -1)]]
    S["cube"] = [Matrix(v) / sqrt(3) for v in itertools.product([1, -1], repeat=3)]
    S["icosahedron"] = (
        [Matrix([a * s_ico * TAU_SYM, b * s_ico, 0]) for a in (1, -1) for b in (1, -1)]
        + [Matrix([0, a * s_ico * TAU_SYM, b * s_ico]) for a in (1, -1) for b in (1, -1)]
        + [Matrix([a * s_ico, 0, b * s_ico * TAU_SYM]) for a in (1, -1) for b in (1, -1)])
    S["dodecahedron"] = (
        [Matrix(v) / sqrt(3) for v in itertools.product([1, -1], repeat=3)]
        + [Matrix([0, a * c_dod * SIG_SYM, b * c_dod * TAU_SYM]) for a in (1, -1) for b in (1, -1)]
        + [Matrix([a * c_dod * SIG_SYM, b * c_dod * TAU_SYM, 0]) for a in (1, -1) for b in (1, -1)]
        + [Matrix([a * c_dod * TAU_SYM, 0, b * c_dod * SIG_SYM]) for a in (1, -1) for b in (1, -1)])
    return S


def atlas_gates():
    """The thesis gates whose rotation axes the solids inherit (SymPy)."""
    H = (1 / sqrt(2)) * Matrix([[1, 1], [1, -1]])
    S = Matrix([[1, 0], [0, sI]])
    return {
        "X": Matrix([[0, 1], [1, 0]]),
        "Z": Matrix([[1, 0], [0, -1]]),
        "F": H * S.conjugate().T,                    # face gate F = H S+
        "Phi": Rational(1, 2) * Matrix([[TAU_SYM + sI * SIG_SYM, 1],
                                        [-1, TAU_SYM - sI * SIG_SYM]]),
    }


def state_from_bloch(n):
    """Density matrix (I + n . sigma)/2 of a Bloch vector, exact."""
    rho = (sp.eye(2) + sum((nj * P for nj, P in zip(n, PAULI_SYM)),
                           sp.zeros(2, 2))) / 2
    return sp.simplify(rho)


def on_solid(n, verts):
    """Is the exact vector n one of the vertices?"""
    return any(sp.simplify(n - v) == Matrix([0, 0, 0]) for v in verts)


def bloch_axis(U):
    """Rotation axis of a 2x2 unitary, exact and normalized.

    In SU(2) form U = cos(t/2) I - i sin(t/2) n.sigma, so
    tr(sigma_j U) = -2i sin(t/2) n_j: n is parallel to (i/2) tr(sigma_j U).
    """
    U = sp.simplify(U / sp.sqrt(U.det()))
    n = Matrix([sp.simplify(sI * sp.trace(P * U) / 2) for P in PAULI_SYM])
    n = sp.simplify(n / sp.sqrt(n.dot(n)))
    return sp.Matrix([sp.radsimp(sp.nsimplify(c)) for c in n])


def bloch_matrix(U):
    """SO(3) action of a 2x2 unitary, exact: R_ij = tr(sigma_i U sigma_j U+)/2.

    Quadratic in the entries of U, which is why the 1/sqrt2 of H and F never
    reaches SO(3) (it pairs off) and why global phase cancels outright -- the
    two facts that make Q(sqrt5), not K_R, the field of the ATLAS-GENERATED
    rotations, those of every thesis gate set with no T in it. Not of the
    others: Bloch(T) = Rz(45 deg) leaves Q(sqrt5) and stays in K_R, which is
    the whole of check_reorientation_obstruction's second half.
    """
    Ud = U.conjugate().T
    return Matrix(3, 3, lambda i, j:
                  sp.radsimp(sp.nsimplify(sp.simplify(
                      sp.trace(PAULI_SYM[i] * U * PAULI_SYM[j] * Ud) / 2))))


def atlas_vertices(solid):
    """symbolic_solids()[solid], permuted into povm_{solid}.npz order.

    That npz order IS the published numbering: povm_properties.py emits it as
    tab:povm-atlas, and paper/figures/platonic_solid_povms.tex draws the same
    indices.  symbolic_solids() does NOT match it -- same solids, same pose,
    same vertex sets, different numbering for the tetrahedron, cube and
    icosahedron.  So anything the thesis prints WITH a vertex index has to run
    here, on the numbering a reader can look up; see exact_vertices() below for
    why the permutation is safe, and det_witness() for what depends on it.
    """
    sym = symbolic_solids()[solid]
    num = np.array([[float(c) for c in v] for v in sym])
    out = []
    for row in load_vertices(solid):
        dist = np.linalg.norm(num - row, axis=1)
        near = np.sort(dist)[:2]
        assert near[0] < 1e-12 and near[1] > 0.1, (solid, row, near)
        out.append(sym[int(np.argmin(dist))])
    return out

In [6]:
# === Section 0: pin the claims to the canonical data (check_canonical_data) ===

solids_sym = symbolic_solids()
for solid in SOLIDS:
    s = load_vertices(solid)
    E = load_elements(solid)
    sym = np.array([[float(c) for c in v] for v in solids_sym[solid]])
    assert len(s) == len(sym)
    for v in s:                  # vertex sets agree as sets, with margin:
        near = np.sort(np.linalg.norm(sym - v, axis=1))[:2]
        assert near[0] < 1e-12 and near[1] > 0.1, (solid, v, near)
    V = len(s)                   # elements are (1/V)(I + n . sigma)
    expected = (np.eye(2)[None] + np.einsum("kn,nab->kab", s, PAULI)) / V
    d_E = np.abs(E - expected).max()
    assert d_E < 1e-12, (solid, d_E)
print("[ok] npz vertices match the exact symbolic solids; elements = (1/V)(I + n.sigma)")

gates_npz = np.load(DATA + "gates.npz", allow_pickle=True)
names = [str(n) for n in gates_npz["names"]]
for name, U in atlas_gates().items():
    A = np.array(U.evalf(), dtype=complex)
    B = gates_npz["su2"][names.index(name)]
    # |tr(A^dag B)| = 2 iff A = +-B: the ONLY guard that the symbolic gates
    # and gates.npz agree. Deviation is 0.0 exactly.
    d_tr = abs(abs(np.trace(A.conj().T @ B)) - 2)
    assert d_tr < 1e-12, f"gate {name} != +-gates.npz (|dev| = {d_tr:.2e})"
print("[ok] symbolic gates match gates.npz projectively (X, Z, F, Phi)")

for g in ("T", "O", "I"):
    R = load_rotations(g)
    U2 = load_atlas(g)["unitaries"]
    keys_R = {rot_key(Rg) for Rg in R}
    keys_U = {rot_key(rotation_from_unitary(U)) for U in U2}
    assert keys_R == keys_U and len(keys_R) == len(R)
print("[ok] group_2X unitaries project onto exactly the group_X rotations (+-U pair up)")

keys_T = {rot_key(Rg) for Rg in load_rotations("T")}
for g in ("O", "I"):
    assert keys_T <= {rot_key(Rg) for Rg in load_rotations(g)}
print("[ok] T < O and T < I as rotation groups (2T sits inside both 2O and 2I)")

[ok] npz vertices match the exact symbolic solids; elements = (1/V)(I + n.sigma)
[ok] symbolic gates match gates.npz projectively (X, Z, F, Phi)
[ok] group_2X unitaries project onto exactly the group_X rotations (+-U pair up)
[ok] T < O and T < I as rotation groups (2T sits inside both 2O and 2I)


## 1. Two protocols, two estimator channels — the reframing, first application

A protocol's *estimator channel* is the affine map $r \mapsto Mr + \mathrm{off}$ from the
state's Bloch vector to the mean reconstructed snapshot ($3\times$ the mean sampled vertex —
the canonical dual's factor). Both protocols below compute it as an **exact group average** —
a finite sum over draws and outcomes, no sampling — under the shared probe noise.

- In R1 the drawn $g$ sits in the outcome probability *and* in the snapshot $3b\,R_g^\top v_0$,
  so the average twirls the composition readout$\,\circ\,$noise;
- in R2 the readout ranges over all $V$ effects whatever the draw, so summing outcomes first
  contracts the vertex covariance $\sum_k \hat n_k\hat n_k^\top = \frac V3\,\mathrm{Id}$ and the
  average twirls the noise alone.

Same lemma, different object handed to it — hence different scalars, printed below. R1 needs
antipodal vertex pairs (its coin measures $\{v, -v\}$ bases), so the tetrahedron is out; R2
keeps it — *the SIC is not the price of the twirl; the ancilla is* (finding 2). And sharpness:
with **no** draw the channel is the noise map itself, offset and all — what the twirl removes is
really there.

**Notation, pinned before any number appears.** Throughout the suite $\kappa$ is the *estimator
channel's* multiplier, ideal $1$ — equivalently the overlap of the believed measurement with the
performed one, so any fixed misspecification costs a $1/\kappa^2$ shot premium. The thesis's
Appendix F.3.1 and `shadow_experiments.py` write $\eta$ for the *calibration scalar*, the
shrinkage of the twirled measurement channel **before** the canonical dual's factor $3$. The
dictionary is $\kappa = 3\eta$: ideal $\kappa = 1$ is noiseless $\eta = 1/3$. Premia are ratios
and read the same in either convention; a bare $1/\kappa^2$ carried into $\eta$'s units is
$1/(9\eta^2)$.

In [7]:
# === The two protocols as exact estimator channels (lifted from randomized_core.py) ===

def is_decomposable(s):
    """Antipodal decomposability: every vertex has its antipode in the set."""
    return all(any(np.allclose(a, -b, atol=1e-9) for b in s) for a in s)


def alignment(s):
    """The fixed rotation A of R1, taking the vertex v0 nearest +z to zhat.

    Rodrigues' formula about the axis v0 x zhat. For the octahedron v0 is
    zhat itself and A = I -- the one solid whose R1 needs no alignment.
    """
    v = s[np.argmax(s[:, 2])]
    zhat = np.array([0.0, 0.0, 1.0])
    axis = np.cross(v, zhat)
    if np.linalg.norm(axis) < 1e-12:
        return np.eye(3), v
    axis /= np.linalg.norm(axis)
    ang = np.arccos(np.clip(v @ zhat, -1, 1))
    K = np.array([[0, -axis[2], axis[1]],
                  [axis[2], 0, -axis[0]],
                  [-axis[1], axis[0], 0]])
    A = np.eye(3) + np.sin(ang) * K + (1 - np.cos(ang)) * K @ K
    # np.allclose/isclose keep rtol=1e-5 alongside any atol, so where the atol
    # has to be the whole bound the deviation is measured and compared
    # explicitly; allclose stands where the reference is 0, and in the float
    # searches, which the exact layer re-decides. Worst here 1.2e-16.
    d_A = np.abs(A @ v - zhat).max()
    assert d_A < 1e-12, f"alignment: A v0 != zhat, V = {len(s)} (max |dev| = {d_A:.2e})"
    return A, v


def channel_R1(s, R, T, t):
    """Exact estimator channel of R1 (randomized-projective) under noise
    r -> T r + t acting measurement-side.

    Protocol: draw g uniformly from the rotations R, apply U_g, apply the
    fixed alignment A (v0 -> zhat), read out Z. Snapshot = 3 b R_g^T v0
    (the canonical dual of a projective measurement). The outcome
    probability p(b) = (1 + b [T (A R_g r) + t]_z)/2 is linear in the input
    Bloch vector r, so E[snapshot] = M r + off with M and off computed
    below as exact group averages -- no sampling anywhere.
    """
    A, v = alignment(s)
    M = np.zeros((3, 3))
    off = np.zeros(3)
    for Rg in R:
        pre = A @ Rg
        for b in (+1, -1):
            lin = b * (T @ pre)[2, :] / 2.0          # coefficient of r in p(b)
            const = (1 + b * t[2]) / 2.0
            snap = 3.0 * b * (Rg.T @ v)
            M += np.outer(snap, lin)
            off += snap * const
    return M / len(R), off / len(R)


def channel_R2(s, R, T, t, s_actual=None):
    """Exact estimator channel of R2 (twirled-native) under the same noise.

    Protocol: draw g, apply U_g, measure the NATIVE (Naimark) POVM -- all V
    effects (1/V)(I + n_k . sigma) -- and relabel by g in post-processing.
    Snapshot = 3 R_g^T n_k; p(k) = (1/V)(1 + n_k . (T R_g r + t)). Again an
    exact average, all solids admitted (no decomposability needed).

    s_actual, if given, is the vertex list the device MEASURES, index for
    index, where s stays the list the estimator BELIEVES: snapshots are
    built from s, outcome probabilities from s_actual. This is the
    two-list channel of the price-of-inexactness theorem, which the thesis
    states in Appendix F.3.2. Both hypotheses matter.
    For an IRREDUCIBLE draw R any fixed mismatch twirls to exactly

        M = tr(B T)/V Id,  off = 0,   B = sum_k b_k a_k^T,

    b the believed vertex, a the measured one. Two readings of the same
    scalar: at T = Id it is the frame overlap (1/V) sum_k b_k . a_k, the
    kappa the callers price; with no mismatch B = (V/3) Id and it is
    Finding 1's tr(T)/3. The bare overlap is NOT the answer under noise --
    at T_NOISE the nine SIC derangements, every one -1/3 noiselessly,
    spread to nine DISTINCT prices (check_decker_outcome_order asserts
    both halves). And the zero offset is the DRAW's: off = 3 sum_k
    const_k [<R_g^T>] b_k = (3/V) [<R_g^T>] B t, so it vanishes for any
    belief list whatever and reports nothing about the mismatch -- drop
    irreducibility (R = [Id]) and it is (3/V) B t, the t itself of
    check_two_protocols only when the lists agree. Default: no mismatch.
    """
    a = s if s_actual is None else s_actual
    V = len(s)
    M = np.zeros((3, 3))
    off = np.zeros(3)
    for Rg in R:
        # strict: V normalizes the probabilities, so a short s_actual would
        # silently return a plausible depolarizing kappa off V - 1 terms
        for bk, ak in zip(s, a, strict=True):
            lin = (ak @ (T @ Rg)) / V                # coefficient of r in p(k)
            const = (1 + ak @ t) / V
            snap = 3.0 * (Rg.T @ bk)
            M += np.outer(snap, lin)
            off += snap * const
    return M / len(R), off / len(R)


def orbit_counts(s, R):
    """How often the R1 snapshot direction b R_g^T v0 hits each vertex.

    R1's coin: draw g and outcome b land the snapshot on b R_g^T v0.
    Uniform counts = the draw samples the POVM's vertices uniformly (the
    coin realizes the POVM); a zero = that vertex is never measured at all.
    """
    _, v = alignment(s)
    hits = np.zeros(len(s), dtype=int)
    for Rg in R:
        for b in (+1, -1):
            w = b * (Rg.T @ v)
            d = np.linalg.norm(s - w, axis=1)
            k = int(np.argmin(d))
            assert d[k] < 1e-9, "snapshot direction is not a vertex"
            hits[k] += 1
    return hits

In [8]:
# === Section 1: findings 1 + 2 at the probe (check_two_protocols) ===

kappa_R1 = T_NOISE[2, 2]
kappa_R2 = np.trace(T_NOISE) / 3
print(f"generic probe noise:  T_zz = {kappa_R1:.6f}   tr(T)/3 = {kappa_R2:.6f}")
print(f"\n{'solid':14s} {'grp':4s} {'R1 (randomized-projective)':>27s}   {'R2 (twirled-native)':>21s}")
print("-" * 72)
for solid in SOLIDS:
    s = load_vertices(solid)
    R = load_rotations(COVARIANCE[solid])
    # R2 exists for every solid: unbiased, exactly depolarizing at tr(T)/3.
    # Worst M deviation over all five solids and both protocols is 1.3e-14
    # (dodecahedron, R2 at zero noise).
    M0, o0 = channel_R2(s, R, np.eye(3), np.zeros(3))
    d_M0 = np.abs(M0 - np.eye(3)).max()
    assert d_M0 < 1e-10, f"{solid}: R2 biased at zero noise (max |dev| = {d_M0:.2e})"
    assert np.allclose(o0, 0, atol=1e-9)
    M2, o2 = channel_R2(s, R, T_NOISE, t_NOISE)
    d_M2 = np.abs(M2 - kappa_R2 * np.eye(3)).max()
    assert d_M2 < 1e-10, \
        f"{solid}: R2 not depol at tr(T)/3 (max |dev| = {d_M2:.2e})"
    assert np.allclose(o2, 0, atol=1e-9)
    # R1 exists only for the antipodal solids
    if is_decomposable(s):
        hits = orbit_counts(s, R)    # the coin: uniform over the vertices
        assert hits.min() == hits.max() == 2 * len(R) // len(s)
        M0, o0 = channel_R1(s, R, np.eye(3), np.zeros(3))
        d_M0 = np.abs(M0 - np.eye(3)).max()
        assert d_M0 < 1e-10, \
            f"{solid}: R1 biased at zero noise (max |dev| = {d_M0:.2e})"
        assert np.allclose(o0, 0, atol=1e-9)
        M1, o1 = channel_R1(s, R, T_NOISE, t_NOISE)
        d_M1 = np.abs(M1 - kappa_R1 * np.eye(3)).max()
        assert d_M1 < 1e-10, \
            f"{solid}: R1 not depol at T_zz (max |dev| = {d_M1:.2e})"
        assert np.allclose(o1, 0, atol=1e-9)
        r1 = f"depol, kappa = {M1[0, 0]:.6f}"
    else:
        assert solid == "tetrahedron"
        r1 = "undefined (no antipodes)"
    r2 = f"depol, kappa = {M2[0, 0]:.6f}"
    print(f"{solid:14s} {COVARIANCE[solid]:4s} {r1:>27s}   {r2:>21s}")
print("\n[ok] R1: exactly depolarizing at kappa = T_zz for the four antipodal solids;")
print("[ok] R2: exactly depolarizing at kappa = tr(T)/3 for ALL FIVE (SIC included);")
print("     offsets vanish, both estimators unbiased at zero noise")
# The sharpness half of the laundering: with NO draw the channel is the
# noise map ITSELF -- M = T and off = t exactly, by the frame condition
# (3/V) sum n n^T = Id and sum n = 0 -- so a fixed coherent error
# survives whole (an offset, and a tilt: bias FIRST order in the error)
# until a draw is layered on. The laundering is the draw's doing, not
# the POVM's. Asserted nowhere else; an identity, not a tolerance.
# It doubles as the two-list theorem's irreducibility hypothesis failing
# visibly: R = [Id] is reducible, and the offset that vanishes for every
# irreducible draw and every belief list is here (3/V) B t -- t itself
# in this no-mismatch case, where B = (V/3) Id.
for solid in SOLIDS:
    s = load_vertices(solid)
    M, off = channel_R2(s, [np.eye(3)], T_NOISE, t_NOISE)
    d_M = np.abs(M - T_NOISE).max()          # worst over the five: 2.2e-16
    d_o = np.abs(off - t_NOISE).max()
    assert d_M < 1e-12, f"{solid}: undrawn channel != T (max |dev| = {d_M:.2e})"
    assert d_o < 1e-12, f"{solid}: undrawn offset != t (max |dev| = {d_o:.2e})"
print("[ok] sharpness: with NO draw the channel is T itself, offset t, exactly")
print("     (the frame condition) -- what the twirl removes is really there")

generic probe noise:  T_zz = 0.620000   tr(T)/3 = 0.720000

solid          grp   R1 (randomized-projective)     R2 (twirled-native)
------------------------------------------------------------------------
tetrahedron    T       undefined (no antipodes)   depol, kappa = 0.720000
octahedron     O        depol, kappa = 0.620000   depol, kappa = 0.720000
cube           O        depol, kappa = 0.620000   depol, kappa = 0.720000
icosahedron    I        depol, kappa = 0.620000   depol, kappa = 0.720000
dodecahedron   I        depol, kappa = 0.620000   depol, kappa = 0.720000

[ok] R1: exactly depolarizing at kappa = T_zz for the four antipodal solids;
[ok] R2: exactly depolarizing at kappa = tr(T)/3 for ALL FIVE (SIC included);
     offsets vanish, both estimators unbiased at zero noise
[ok] sharpness: with NO draw the channel is T itself, offset t, exactly
     (the frame condition) -- what the twirl removes is really there


### The mechanism: who decides the readout axis

The two protocols differ in exactly one place. In R1 the same draw that rotates the state also
decides — through the fixed $\hat z$ readout — which lab direction of the noise gets probed:
draw and readout are perfectly correlated, the correlation sits *inside* the average, and what
survives is the noise seen along the readout axis, $T_{zz}$. In R2 the readout is the same fixed
POVM every shot; there is nothing for the draw to correlate with, the noise is seen whole, and
only its isotropic mean survives, $\operatorname{tr}T/3$.

The module's own reduction says this in one line: R1's channel depends on the noise only through
the rank-one $v_0 w^\top$ with $w = A^\top T^\top \hat z$ — and since $A$ is a rotation,
$v_0 \cdot w = \hat z^\top T^\top \hat z = T_{zz}$, the alignment cancelling between snapshot
and readout. The demo verifies the reduction against `channel_R1` and runs R2's contraction
explicitly.

The moral — the thesis's own — is **calibrate the protocol you run**: fit a
randomized-projective experiment with $\operatorname{tr}T/3$, or a twirled-native one with
$T_{zz}$, and the miscalibration is exactly the anisotropy of the noise — the gap between the
two printed factors above.

In [9]:
# === R1 reduced to a rank-one twirl (lifted), plus R2's contraction (glue) ===

def rank_one_twirl(Rs, v, w):
    """(3 E_g R_g^T v w^T R_g,  3 E_g R_g^T v): the R1 channel, reduced.

    R1's estimator channel depends on the noise only through the rank-one
    v w^T, with v = A^T zhat the seed vertex and w = A^T T^T zhat -- the
    alignment cancels between snapshot and readout, and the twirl never sees
    the noise map itself. Works over SymPy or numpy alike; the reduction is
    verified against channel_R1 subgroup by subgroup in check_coin_group.
    """
    n = len(Rs)
    if isinstance(v, Matrix):
        return (3 * sum((R.T * (v * w.T) * R for R in Rs), sp.zeros(3, 3)) / n,
                3 * sum((R.T * v for R in Rs), sp.zeros(3, 1)) / n)
    return (3 * np.mean([R.T @ np.outer(v, w) @ R for R in Rs], axis=0),
            3 * np.mean([R.T @ v for R in Rs], axis=0))


zhat = np.array([0.0, 0.0, 1.0])
s = load_vertices("icosahedron")
R = load_rotations("I")
A, v0 = alignment(s)

# R1: the average twirls the composition readout . noise -- the rank-one
# v0 w^T, w = A^T T^T zhat. The reduction must BE channel_R1 -- asserted,
# not just printed: executing green must mean the identity held.
M1, off1 = rank_one_twirl(R, v0, A.T @ T_NOISE.T @ zhat)
Mf, of_ = channel_R1(s, R, T_NOISE, t_NOISE)
assert np.abs(M1 - Mf).max() < 1e-12
assert np.abs(t_NOISE[2] * off1 - of_).max() < 1e-12
print(f"reduction vs channel_R1:  max|dM| = {np.abs(M1 - Mf).max():.1e}, "
      f"max|doff| = {np.abs(t_NOISE[2] * off1 - of_).max():.1e}")
print(f"kappa = v0 . w = zhat^T T^T zhat = T_zz = "
      f"{v0 @ (A.T @ T_NOISE.T @ zhat):.6f}")

# ...and once with a draw that does NOT twirl (the trivial group), where the
# offsets are far from zero. On the irreducible draw above both offsets
# vanish, so this control is what keeps the offset comparison from passing
# vacuously; the module's own bridge asserts the identity on all 178
# (subgroup, solid) pairs of both lattices (check_coin_group).
R_triv = np.eye(3)[None]
M1t, off1t = rank_one_twirl(R_triv, v0, A.T @ T_NOISE.T @ zhat)
Mft, offt = channel_R1(s, R_triv, T_NOISE, t_NOISE)
assert np.abs(offt).max() > 0.1            # genuinely nonzero
assert np.abs(M1t - Mft).max() < 1e-12
assert np.abs(t_NOISE[2] * off1t - offt).max() < 1e-12
print(f"\ntrivial-draw control: max|off| = {np.abs(offt).max():.3f} != 0,"
      f" and the reduction is still exact")

# R2: summing the V outcomes first contracts sum_k n_k n_k^T = (V/3) Id,
# so the average twirls the noise ALONE and the trace is tr(T)/3.
print(f"\nvertex covariance: max|s^T s - (V/3) Id| = "
      f"{np.abs(s.T @ s - (len(s) / 3) * np.eye(3)).max():.1e}")
G = 3 * np.mean([Rg.T @ (T_NOISE / 3) @ Rg for Rg in R], axis=0)
assert np.abs(G - (np.trace(T_NOISE) / 3) * np.eye(3)).max() < 1e-12
print(f"twirl of the noise alone: max|G - (tr T/3) Id| = "
      f"{np.abs(G - (np.trace(T_NOISE) / 3) * np.eye(3)).max():.1e}")

reduction vs channel_R1:  max|dM| = 5.6e-16, max|doff| = 7.5e-18
kappa = v0 . w = zhat^T T^T zhat = T_zz = 0.620000

trivial-draw control: max|off| = 0.434 != 0, and the reduction is still exact

vertex covariance: max|s^T s - (V/3) Id| = 1.3e-15
twirl of the noise alone: max|G - (tr T/3) Id| = 2.2e-16


### 1a. The exact upgrade: both factors are identities in the noise

`check_two_protocols` decided both factors with `allclose` at the single probe. Here each is
quantified over **every** measurement-side $(T, t)$: the channels are linear in the noise, so a
spanning set of probes decides the whole probe space — four probes for R1 (only row $z$ of $T$
and $t_z$ can enter at all), thirteen for R2 (all of $T$ enters; the offset is affine in $t$).
The verdicts run over each solid's own small number field — no tolerance — and, per the
canonical-form discipline, each boolean is paired with a value-for-value float agreement at the
probe, because an exact check does not test its own transcription.

Machinery, in two lifts: the exact rotation layer (T/O/I rebuilt from `main.py`'s canonical
quaternions, in npz row order, no grid and no matching step), then the number-field kit — the
per-solid fields, fail-loud coercion, exact alignment (radical- and transcendental-free:
$A = I + K_w + K_w^2/(1+v_z)$, rational in $v_0$), and literal transcriptions of both channels
over an arbitrary coefficient domain $K$.

The two negative controls at the end are the point: at $T = E_{00}$ (so
$\operatorname{tr}T = 1$ but $T_{zz} = 0$) R2 returns $\mathrm{Id}_3/3$ exactly where R1 returns
$0$ — finding 1 as an identity, not a gap between two decimals — and a reducible draw fails both
tests outright.

In [10]:
# === The exact rotation layer (lifted from randomized_core.py) ===

# Every entry of every polyhedral rotation matrix in the atlas pose, exactly:
# T and O are signed permutations, I adds the golden half-integers. Nine
# values, no two closer than 0.19.
#
# The list is a CENSUS, not a decision procedure: exact_rotations builds the
# matrices from main.py's canonical quaternions and checks their entries
# against these nine values. Lifting float rotations onto the list instead --
# nearest entry within 1e-9, one coordinate at a time -- would be rot_key's
# rounding grid again, a nine-value codebook in place of nine decimals, and
# would put a tolerance back in front of everything downstream.
ROT_ENTRIES = [sp.Integer(0), sp.Integer(1), sp.Integer(-1),
               Rational(1, 2), -Rational(1, 2),
               TAU_SYM / 2, -TAU_SYM / 2, SIG_SYM / 2, -SIG_SYM / 2]


def _quat_to_rotation_sym(q):
    """Symbolic twin of export_numpy.quat_to_rotation -- same (w, -z, -y, -x)."""
    w, x, y, z = q.w, -q.z, -q.y, -q.x
    return Matrix([
        [1 - 2*(y*y + z*z),  2*(x*y - w*z),      2*(x*z + w*y)],
        [2*(x*y + w*z),      1 - 2*(x*x + z*z),  2*(y*z - w*x)],
        [2*(x*z - w*y),      2*(y*z + w*x),      1 - 2*(x*x + y*y)],
    ])


_EXACT_ROT = {}


def exact_rotations(g):
    """Rotations of T/O/I as exact matrices, in data/group_{g}.npz ROW ORDER.

    No matching step and no tolerance.  The npz is written by export_numpy's
    rotation_group_to_numpy(), which sorts the binary group's quaternions and
    keeps one per proj_hash class; re-running that path without the float cast
    returns the same elements in the same order BY CONSTRUCTION, so index i
    here is index i there.  proj_hash is main.py's canonical form, so the
    correspondence is decided, not searched for.

    Three checks, none of them load-bearing for the construction: SO(3)
    membership, and that the entries fall in the ROT_ENTRIES census.  Order
    against the npz is verified in check_exact_two_bars.

    Memoized: rebuilding I costs 0.5 s apiece.  The list is copied out so a
    caller cannot grow the cached one.
    """
    if g in _EXACT_ROT:
        return list(_EXACT_ROT[g])
    seen, reps = set(), []
    for q in sorted(geometric_group("2" + g)):
        ph = q.proj_hash()
        if ph not in seen:
            seen.add(ph)
            reps.append(q)
    out = []
    for q in reps:
        M = _quat_to_rotation_sym(q)
        assert sp.expand(M.T * M - sp.eye(3)) == sp.zeros(3, 3)
        assert sp.expand(M.det()) == 1
        assert all(any(sp.expand(M[i, j] - e) == 0 for e in ROT_ENTRIES)
                   for i in range(3) for j in range(3)), (g, M)
        out.append(M)
    _EXACT_ROT[g] = out
    return list(out)

In [11]:
# === The number-field kit (lifted from randomized_field.py) ===

# Rotation entries of T, O and I lie in Q(sqrt2, sqrt5) -- main.py's own field,
# the one _to_basis canonicalises.  Vertices need more, and each solid gets a
# separate small field rather than one global degree-16 field (150x faster,
# every field degree <= 4).  Each must ALSO contain the covariance group's
# entries, so the orbit b Rg^T v0 cannot leave it; that is checked by
# coercion, which raises rather than approximating.
FIELD_GENS = {"tetrahedron": (sqrt(3),), "octahedron": (), "cube": (sqrt(3),),
              "icosahedron": (sqrt(2 + TAU_SYM),),
              "dodecahedron": (sqrt(3), sqrt(5))}

FIELD_NAME = {"tetrahedron": "Q(sqrt3)", "octahedron": "Q", "cube": "Q(sqrt3)",
              "icosahedron": "Q(sqrt(2+tau))", "dodecahedron": "Q(sqrt3, sqrt5)"}


def solid_field(solid):
    """The solid's coefficient field as a SymPy domain (QQ when trivial)."""
    gens = FIELD_GENS[solid]
    return sp.QQ.algebraic_field(*gens) if gens else sp.QQ


def to_field(M, K):
    """Coerce a symbolic 3x3 into K as nested lists -- fail-loud, no simplify.

    from_sympy raises CoercionFailed outside K, so a mis-declared field or a
    re-posed solid cannot slip through as a wrong answer.  Deliberately no
    nsimplify/radsimp in front: every rotation entry and every vertex coerces
    raw, so nothing here rests on a normalising heuristic.
    """
    return [[K.from_sympy(M[i, j]) for j in range(3)] for i in range(3)]


def _mm(A, B, K):
    return [[sum((A[i][k] * B[k][j] for k in range(3)), K.zero) for j in range(3)]
            for i in range(3)]


def _mv(A, v, K):
    return [sum((A[i][k] * v[k] for k in range(3)), K.zero) for i in range(3)]


def _tr(A):
    return [[A[j][i] for j in range(3)] for i in range(3)]


def _eye(K):
    return [[K.one if i == j else K.zero for j in range(3)] for i in range(3)]


def exact_vertices(solid, K):
    """symbolic_solids()[solid] over K, permuted into povm_{solid}.npz order.

    Order is load-bearing, not cosmetic: alignment() takes v0 = argmax(s[:,2]),
    i.e. the FIRST vertex at the top latitude, and the top latitude is a tie on
    three of the four solids (cube 4, icosahedron 2, dodecahedron 2; only the
    octahedron has a unique v0).  A set-level correspondence would silently
    pick a different seed.  The permutation used is the one
    check_canonical_data() already asserts unique, not a new one.

    The npz is float64, so this is the one place the exact path must meet a
    float with no exact object on the other side to meet.  It cannot be removed
    -- only bounded, and the bound is enormous: the symbolic vertices
    reproduce the npz BIT-IDENTICALLY (max error 0.0) while the closest two
    distinct vertices of any solid are 0.71 apart.  So the assertion is on the
    MARGIN rather than on the threshold -- match inside 1e-12, runner-up beyond
    0.1 -- which leaves twelve orders of slack on each side and, unlike a bare
    threshold, cannot rot quietly.
    """
    return [[K.from_sympy(c) for c in v] for v in atlas_vertices(solid)]


def exact_alignment(s, K):
    """alignment(), radical-free and transcendental-free.

    The float version normalises the axis and goes through arccos/sin/cos.  It
    need not: with w = v0 x zhat left UNNORMALISED, |w| = sin(ang), so
    sin(ang) K_unit = K_w exactly, and (1 - cos(ang)) K_unit^2 = K_w^2/(1 + v_z)
    since |w|^2 = (1 - cos)(1 + cos).  So

        A = I + K_w + K_w^2 / (1 + v0_z),

    rational in v0's coordinates -- no square root enters and A stays in the
    solid's own field.  The octahedron's v0 = zhat gives K_w = 0, hence A = I,
    with no special case needed.
    """
    zs = [K.to_sympy(v[2]) for v in s]
    i0 = 0
    for i, z in enumerate(zs):                # np.argmax: FIRST strict maximum
        if sp.simplify(z - zs[i0]).is_positive:
            i0 = i
    v = s[i0]
    Kw = [[K.zero, K.zero, -v[0]], [K.zero, K.zero, -v[1]], [v[0], v[1], K.zero]]
    Kw2 = _mm(Kw, Kw, K)
    d = K.one + v[2]
    A = [[_eye(K)[i][j] + Kw[i][j] + Kw2[i][j] / d for j in range(3)]
         for i in range(3)]
    assert _mv(A, v, K) == [K.zero, K.zero, K.one], "A does not send v0 to zhat"
    return A, v


def exact_is_decomposable(s, K):
    """is_decomposable() over K -- antipodal pairing by field equality.

    This is the verdict that admits or bars R1 (finding 2): the tetrahedral SIC
    has no antipodal pairs, the other four solids are centrally symmetric.  It
    decides whether two vectors are negatives of each other, so it is an
    identity test, decided in canonical form rather than by a tolerance.
    """
    return all(any([-c for c in a] == list(u) for u in s) for a in s)


def exact_orbit_directions(R, v, K):
    """The DISTINCT directions R_g^T v, deduped by field equality.

    The float twin rounds to 9 decimals and calls np.unique; here two orbit
    points are the same point or they are not.  Used for the dodecahedron's
    inscribed-cube split, where the count itself is the claim.
    """
    out = []
    for Rg in R:
        w = _mv(_tr(Rg), v, K)
        if w not in out:
            out.append(w)
    return out


def exact_orbit_counts(s, R, K):
    """orbit_counts(), with argmin-plus-tolerance replaced by field equality."""
    _, v = exact_alignment(s, K)
    hits = [0] * len(s)
    for Rg in R:
        Rgt = _tr(Rg)
        for b in (K.one, -K.one):
            w = [b * c for c in _mv(Rgt, v, K)]
            hit = [k for k, u in enumerate(s) if list(u) == w]
            assert len(hit) == 1, "snapshot direction is not a vertex"
            hits[hit[0]] += 1
    return hits


def exact_channel_R1(s, R, T, t, K):
    """channel_R1() over K -- a literal transcription, same loop, same terms."""
    A, v = exact_alignment(s, K)
    M = [[K.zero] * 3 for _ in range(3)]
    off = [K.zero] * 3
    half = K.one / 2
    for Rg in R:
        TP = _mm(T, _mm(A, Rg, K), K)
        snap0 = _mv(_tr(Rg), v, K)
        for b in (K.one, -K.one):
            lin = [b * TP[2][j] * half for j in range(3)]
            const = (K.one + b * t[2]) * half
            snap = [3 * b * c for c in snap0]
            for i in range(3):
                off[i] += snap[i] * const
                for j in range(3):
                    M[i][j] += snap[i] * lin[j]
    n = K.convert(len(R))
    return [[M[i][j] / n for j in range(3)] for i in range(3)], [o / n for o in off]


def _probe(K, entry=None, t_at=None):
    """A basis probe: T with a single 1 at `entry`, t with a single 1 at `t_at`."""
    T = [[K.one if entry == (i, j) else K.zero for j in range(3)] for i in range(3)]
    return T, [K.one if t_at == a else K.zero for a in range(3)]


def exact_twirls(s, R, K):
    """Does R twirl for EVERY measurement-side noise (T, t) -- not just T_NOISE?

    M and off are LINEAR in (T, t): T enters exactly once, through
    (T @ pre)[2, :], and t exactly once, through (1 + b t_z)/2.  So the verdict
    on a spanning set of probes is the verdict on the whole probe space, and
    only row 2 of T and the single component t_z can enter at all.  Four probes
    span what survives: T = E_{2,k} for k = 0,1,2, and t = zhat.

    The condition itself is that M = T_zz I identically, so on the probes:
    E_{2,2} must give I and E_{2,0}, E_{2,1} must give 0.  The offset must
    vanish for every t, which on t = zhat reads sum_g Rg^T v0 = 0 (at t = 0 it
    vanishes for free, the two outcomes cancelling).

    exact_probe_span_ok() checks the span claim itself on the full group rather
    than leaving it resting on a reading of the code above.
    """
    Z = [[K.zero] * 3 for _ in range(3)]
    for k in range(3):
        M, _ = exact_channel_R1(s, R, *_probe(K, entry=(2, k)), K)
        if M != (_eye(K) if k == 2 else Z):
            return False
    _, off = exact_channel_R1(s, R, *_probe(K, t_at=2), K)
    return off == [K.zero] * 3


def exact_channel_R2(s, R, T, t, K):
    """channel_R2() over K -- a literal transcription of its one-list path,
    same loops, same terms. The two-list path (s_actual) has NO companion
    here: exact_reposed_twirl_R2 is the DIAGONAL case, belief = device =
    sC, priced at tr(C^T C T)/3 = tr(T)/3 for every rotation -- a pose,
    not a mismatch -- so the mismatch law tr(B T)/V rests on the float
    layer alone, at four float sites: the coset scan's noisy leg
    (check_reorientation_obstruction) and the anchors' T_NOISE legs,
    Table D.4 pairing and, for how T enters the two-list path rather than
    for the law itself, the second moment's probe leg (all three in
    check_decker_outcome_order). An exact two-list companion over
    Q(sqrt5)[C1, C2, T, t] would subsume the reposed theorem; it is
    deliberately not built -- the float sites above already guard the law,
    and an exact-layer addition is never free here: the entry module's
    charter requires every exact companion to be paired with a
    value-for-value numeric agreement check, so it would be a decision in
    its own right, not an extension of this function.
    """
    V = K.convert(len(s))
    M = [[K.zero] * 3 for _ in range(3)]
    off = [K.zero] * 3
    for Rg in R:
        TR = _mm(T, Rg, K)
        Rgt = _tr(Rg)
        for n in s:
            lin = [sum((n[a] * TR[a][j] for a in range(3)), K.zero) / V
                   for j in range(3)]
            const = (K.one + sum((n[a] * t[a] for a in range(3)), K.zero)) / V
            snap = [3 * c for c in _mv(Rgt, n, K)]
            for i in range(3):
                off[i] += snap[i] * const
                for j in range(3):
                    M[i][j] += snap[i] * lin[j]
    m = K.convert(len(R))
    return [[M[i][j] / m for j in range(3)] for i in range(3)], [o / m for o in off]


def exact_twirls_R2(s, R, K):
    """Is R2's channel (tr T/3) Id and unbiased for EVERY (T, t) -- not just T_NOISE?

    M is linear in T and does not involve t at all; off is AFFINE in t and does
    not involve T.  So nine probes decide M -- the claim M = (tr T/3) Id reads
    E_{i,i} -> Id/3 on the diagonal and E_{i,j} -> 0 off it -- and four decide
    off: t = 0 for the constant term, which must vanish on its own, then
    t = e_k for the linear part.

    Two differences from R1, both structural rather than incidental.  The WHOLE
    of T enters, through n^T T R_g summed over every vertex, so nothing reduces
    to a single row the way R1's readout does.  And there is no alignment: R2
    measures the native POVM, which is exactly why it admits the tetrahedron,
    where R1 is undefined.
    """
    Z = [[K.zero] * 3 for _ in range(3)]
    third = [[K.one / 3 if i == j else K.zero for j in range(3)] for i in range(3)]
    for i in range(3):
        for j in range(3):
            M, off = exact_channel_R2(s, R, *_probe(K, entry=(i, j)), K)
            if M != (third if i == j else Z) or off != [K.zero] * 3:
                return False
    return all(exact_channel_R2(s, R, *_probe(K, t_at=k), K)[1] == [K.zero] * 3
               for k in range(3))


def _exact_probe_noise(K):
    """T_NOISE / t_NOISE as exact rationals in K (two decimals, so exactly so)."""
    return ([[K.from_sympy(Rational(str(float(T_NOISE[i, j])))) for j in range(3)]
             for i in range(3)],
            [K.from_sympy(Rational(str(float(x)))) for x in t_NOISE])


def _as_float(M, K):
    return np.array([[float(K.to_sympy(M[i][j])) for j in range(3)] for i in range(3)])

In [12]:
# === Findings 1 + 2, exactly and generically (check_exact_scalars) ===

# Findings 1 + 2, exactly and generically. check_two_protocols decides both
# scalars with allclose at the single probe T_NOISE; each is quantified here
# over EVERY measurement-side (T, t), so T_NOISE stops being load-bearing.
#
# R1's half is not new mathematics -- rank_one_twirl already proves
# M = (v.w) Id_3 over six free symbols in check_coin_group -- but it is
# established here against channel_R1 itself rather than against a reduction
# of it, and over the same probe basis as R2, so the two protocols are
# finally checked by one argument instead of two.
print(f"  {'solid':14s} {'grp':4s} {'field':16s} {'R2 -> tr(T)/3':>14s}"
      f" {'R1 -> T_zz':>18s}   exact vs float at T_NOISE")
print("  " + "-" * 100)
for solid in SOLIDS:
    K, g = solid_field(solid), COVARIANCE[solid]
    R = [to_field(M, K) for M in exact_rotations(g)]
    s = exact_vertices(solid, K)
    s_f, R_f = load_vertices(solid), load_rotations(g)
    assert exact_twirls_R2(s, R, K), solid

    # Transcription guard. The generic verdicts above are booleans, and a
    # slip inside these loops (a dropped b, a swapped index) moves numbers
    # while only sometimes flipping a boolean.
    # So require the exact channels to reproduce the float ones at T_NOISE,
    # which is a value-for-value test of the transcription itself.
    Te, te = _exact_probe_noise(K)
    d = np.abs(_as_float(exact_channel_R2(s, R, Te, te, K)[0], K)
               - channel_R2(s_f, R_f, T_NOISE, t_NOISE)[0]).max()
    r1 = "n/a (not antipodal)"
    # which solids admit R1 at all is itself an identity question, so the
    # branch below is taken on field equality and the float verdict is
    # required to agree rather than trusted
    assert exact_is_decomposable(s, K) == is_decomposable(s_f), solid
    if exact_is_decomposable(s, K):
        assert exact_twirls(s, R, K), solid
        d = max(d, np.abs(_as_float(exact_channel_R1(s, R, Te, te, K)[0], K)
                          - channel_R1(s_f, R_f, T_NOISE, t_NOISE)[0]).max())
        r1 = "identity in T"
    assert d < 1e-12, (solid, d)
    print(f"  {solid:14s} {g:4s} {FIELD_NAME[solid]:16s} {'identity in T':>14s}"
          f" {r1:>18s}   max|diff| = {d:.1e}")
# Two negative controls, so neither verdict can be passing vacuously.
K = solid_field("octahedron")
s = exact_vertices("octahedron", K)
R = [to_field(M, K) for M in exact_rotations("O")]
# (i) The two scalars are DIFFERENT identities, not two readings of one
# number -- finding 1 itself. At the probe T = E_{0,0} we have tr T = 1 but
# T_zz = 0, so R2 must return Id_3/3 exactly where R1 returns 0.
P = _probe(K, entry=(0, 0))
third = [[K.one / 3 if i == j else K.zero for j in range(3)] for i in range(3)]
assert exact_channel_R2(s, R, *P, K)[0] == third
assert exact_channel_R1(s, R, *P, K)[0] == [[K.zero] * 3 for _ in range(3)]
# (ii) A reducible draw fails both tests outright: the C_3 coin of
# check_coin_group, which realizes the octahedral POVM and twirls nothing.
R_F = bloch_matrix(atlas_gates()["F"])
C3 = [to_field(M, K) for M in (sp.eye(3), R_F, R_F * R_F)]
assert not exact_twirls_R2(s, C3, K) and not exact_twirls(s, C3, K)
print("  negative controls: at T = E_00 (tr T = 1, T_zz = 0) R2 gives Id_3/3 where")
print("  R1 gives 0 -- finding 1 as an identity, not as a gap between two decimals;")
print("  and the reducible C_3 coin fails both tests")

print("[ok] both scalars are IDENTITIES in the noise, not values at a probe: R2 gives")
print("     (tr T/3) Id_3 and zero offset on all five solids -- the tetrahedron")
print("     included, where R1 is undefined -- and R1 gives T_zz Id_3 on the four")
print("     antipodal ones. So neither protocol's scalar rests on T_NOISE any more,")
print("     and the float<->float cross-check against shadow_experiments.py now has an")
print("     exact anchor on this side of it")

  solid          grp  field             R2 -> tr(T)/3         R1 -> T_zz   exact vs float at T_NOISE
  ----------------------------------------------------------------------------------------------------


  tetrahedron    T    Q(sqrt3)          identity in T n/a (not antipodal)   max|diff| = 2.2e-16


  octahedron     O    Q                 identity in T      identity in T   max|diff| = 5.6e-16


  cube           O    Q(sqrt3)          identity in T      identity in T   max|diff| = 7.8e-16


  icosahedron    I    Q(sqrt(2+tau))    identity in T      identity in T   max|diff| = 6.7e-16


  dodecahedron   I    Q(sqrt3, sqrt5)   identity in T      identity in T   max|diff| = 8.9e-16
  negative controls: at T = E_00 (tr T = 1, T_zz = 0) R2 gives Id_3/3 where
  R1 gives 0 -- finding 1 as an identity, not as a gap between two decimals;
  and the reducible C_3 coin fails both tests
[ok] both scalars are IDENTITIES in the noise, not values at a probe: R2 gives
     (tr T/3) Id_3 and zero offset on all five solids -- the tetrahedron
     included, where R1 is undefined -- and R1 gives T_zz Id_3 on the four
     antipodal ones. So neither protocol's scalar rests on T_NOISE any more,
     and the float<->float cross-check against shadow_experiments.py now has an
     exact anchor on this side of it


### 1b. The one misspecification that is a bias, not a premium

Everything above (and everything in §3 cont.) prices a fixed misspecification of the
*measurement* as a $1/\kappa^2$ shot premium. One mistake escapes that pricing, and it turns on
the same two scalars: **a calibration constant carried over from the other protocol.**

The hinge is that a calibration is *empirical*. Learned on the run it reconstructs, it absorbs
whatever the apparatus actually did — an inexact gate, a wrong list, any fixed $(T, t)$ — which
is why those cost shots and never truth. A constant carried across protocols was never learned
on the run being reconstructed. The estimator divides by the believed constant once per touched
site, so a weight-$w$ Pauli term is multiplied by exactly $(\kappa_{\rm run}/\kappa_{\rm cal})^w$
— a bias no shot count removes.

The check proves the law on Appendix F's own estimator — `shadow_experiments.py` imported
lazily and write-free, nothing read from its npz; every number recomputed through its
`noisy_effects` $\to$ `born_tensor` $\to$ `exact_estimator_mean` pipeline on its critical TFIM
ground state — and pins the appendix's printed numbers, both swap directions, plus the control
that separates the two regimes: reconstructed with the constant of the run itself, every
observable is exact.

In [13]:
# === The weight-w mismatch law (check_calibration_mismatch) ===

import shadow_experiments as se

rho = se.density(se.tfim_ground_state(se.G_CRIT))
truth = {on: se.exact_value(rho, obs) for on, obs in se.OBSERVABLES.items()}
e_true = truth["E_TFIM"]
# the appendix's state, pinned against the Jordan-Wigner closed form --
# the same independent witness shadow_experiments.main() runs
assert abs(e_true - se.tfim_ground_energy_exact(se.N_QUBITS, se.G_CRIT)) \
    < 1e-9, e_true

def estimates(s, kappa_run, kappa_cal):
    """Exact estimator means of a run whose channel multiplier is
    kappa_run, reconstructed believing kappa_cal (eta = kappa_cal/3)."""
    p = se.born_tensor(rho, se.noisy_effects(
        s, kappa_run * np.eye(3), np.zeros(3)))
    lut = se.lut_robust_canonical(s, kappa_cal / 3.0)
    return {on: se.exact_estimator_mean(p, obs, lut)
            for on, obs in se.OBSERVABLES.items()}

def law(kappa_run, kappa_cal, obs):
    ratio = kappa_run / kappa_cal
    return sum(c * ratio ** len(sites)
               * se.exact_value(rho, [(1.0, sites)]) for c, sites in obs)

s_ico, s_sic = load_vertices("icosahedron"), load_vertices("tetrahedron")
kappa_r1, kappa_r2 = T_NOISE[2, 2], np.trace(T_NOISE) / 3

# the per-site operator identity behind everything below, at a generic
# nonzero mismatch ratio
E_eff = se.noisy_effects(s_ico, kappa_r1 * np.eye(3), np.zeros(3))
single = np.einsum("ak,kij->aij", 3.0 * s_ico.T / kappa_r2, E_eff)
d_op = np.abs(single - (kappa_r1 / kappa_r2) * PAULI).max()
assert d_op < 1e-12, f"per-site operator identity fails ({d_op:.2e})"

# F.3.1's two constants, taken from the protocol channels
# rather than from the closed forms: dephasing 0.1 twirls to kappa = 1
# exactly under R1 (T_zz = 1: removed outright) and to 13/15 under R2
# (eta = 0.2889, the appendix's printed 0.289)
T_d, t_d = se.chan_dephasing(0.1)
R_I = load_rotations("I")
M1, o1 = channel_R1(s_ico, R_I, T_d, t_d)
d1 = max(np.abs(M1 - np.eye(3)).max(), np.abs(o1).max())
assert d1 < 1e-10, f"R1 does not remove dephasing (max |dev| = {d1:.2e})"
kappa_deph = np.trace(T_d) / 3
M2, o2 = channel_R2(s_ico, R_I, T_d, t_d)
d2 = max(np.abs(M2 - kappa_deph * np.eye(3)).max(), np.abs(o2).max())
assert d2 < 1e-10, f"R2 not at tr(T)/3 on dephasing (max |dev| = {d2:.2e})"

# F.3.1's printed decimals -- "(eta-hat = 0.289 and 0.311 at rate 0.1)",
# dephasing then amplitude damping under twirled-native -- pinned as the
# STRINGS the sentence prints, through the :.3f it rounds by, off the
# channel's own scalar rather than the closed forms 13/45 and
# (2 sqrt(0.9) + 0.9)/9 (0.28889 and 0.31082: neither within 3e-4 of a
# rounding boundary, against channels exact to 1e-16). The maps are
# Section 2.6's: dephasing diag(1-2p, 1-2p, 1), t = 0, from
# shadow_experiments.chan_dephasing above; amplitude damping
# diag(sqrt(1-g), sqrt(1-g), 1-g), t = (0, 0, g), built inline and required
# to match its float twin chan_amp_damping (gate_noise_channel's docstring
# names the twinning). R2 is solid-blind (finding 1), so the octahedron
# under its O draw stands in for the icosahedron above. The sentence's R1
# leg -- randomized-projective sees amplitude damping as T_zz = 1 - g, so
# eta = (1-g)/3 exactly -- on the octahedron, whose alignment is the
# identity, so the readout axis is Bloch z itself; 1e-12 against a
# measured 1.1e-16, both rates.
assert f"{M2[0, 0] / 3:.3f}" == "0.289", M2[0, 0] / 3
# ...and the probe sentence beside them, which types the same channel's
# diagonal and its two readings
assert [f"{T_NOISE[i, i]:.2f}" for i in range(3)] == ["0.83", "0.71", "0.62"]
assert f"{np.trace(T_NOISE) / 3:.2f}" == "0.72", np.trace(T_NOISE) / 3
s_o6, R_O6 = load_vertices("octahedron"), load_rotations("O")
eta_ad = {}
for gam_ad in (0.1, 0.05):
    T_ad = np.diag([np.sqrt(1 - gam_ad), np.sqrt(1 - gam_ad), 1 - gam_ad])
    t_ad = np.array([0.0, 0.0, gam_ad])
    T_se, t_se = se.chan_amp_damping(gam_ad)
    assert np.abs(T_ad - T_se).max() < 1e-15 and np.abs(t_ad - t_se).max() < 1e-15
    M2a, o2a = channel_R2(s_o6, R_O6, T_ad, t_ad)
    d2a = max(np.abs(M2a - np.trace(T_ad) / 3 * np.eye(3)).max(), np.abs(o2a).max())
    assert d2a < 1e-10, f"R2 not at tr(T)/3 on damping {gam_ad} ({d2a:.2e})"
    eta_ad[gam_ad] = M2a[0, 0] / 3
    M1a, o1a = channel_R1(s_o6, R_O6, T_ad, t_ad)
    d1a = max(np.abs(M1a - (1 - gam_ad) * np.eye(3)).max(), np.abs(o1a).max())
    assert d1a < 1e-12, f"R1 not at (1-g) Id on damping {gam_ad} ({d1a:.2e})"
    assert abs(M1a[2, 2] / 3 - (1 - gam_ad) / 3) < 1e-12
assert f"{eta_ad[0.1]:.3f}" == "0.311", eta_ad[0.1]

# six directed pairs: Experiment 3's anchor both ways, F.3.1's
# dephasing swap both ways, the probe's swap both ways
pairs = (("depol 0.1 run, noiseless constant", 0.9, 1.0),
         ("noiseless run, depol 0.1 constant", 1.0, 0.9),
         ("R1 run, R2 constant (dephasing 0.1)", 1.0, kappa_deph),
         ("R2 run, R1 constant (dephasing 0.1)", kappa_deph, 1.0),
         ("R1 run, R2 constant (probe)", kappa_r1, kappa_r2),
         ("R2 run, R1 constant (probe)", kappa_r2, kappa_r1))
e_est = {}
worst_law = worst_solid = 0.0
for label, k_run, k_cal in pairs:
    m_i = estimates(s_ico, k_run, k_cal)
    m_s = estimates(s_sic, k_run, k_cal)
    e_est[label] = m_i["E_TFIM"]
    for on, obs in se.OBSERVABLES.items():
        want = law(k_run, k_cal, obs)
        worst_law = max(worst_law, abs(m_i[on] - want), abs(m_s[on] - want))
        worst_solid = max(worst_solid, abs(m_i[on] - m_s[on]))
assert worst_law < 1e-9, f"weight-w law fails (max |dev| = {worst_law:.2e})"
assert worst_solid < 1e-12, \
    f"mismatch bias sees the solid (max |dev| = {worst_solid:.2e})"

# the control that IS the separator: reconstructed with the constant of
# the run itself, every observable is exact -- the empirical calibration
# absorbed the noise, whatever it was
worst_own = 0.0
for k in (0.9, kappa_deph, kappa_r1, kappa_r2):
    m = estimates(s_ico, k, k)
    worst_own = max(worst_own, max(abs(m[on] - truth[on]) for on in truth))
assert worst_own < 1e-9, \
    f"own-constant reconstruction biased (max |dev| = {worst_own:.2e})"

# the appendix's worked example, pinned at the precision it prints, with
# the three companion biases the check reports beside it
bias_a = e_est["depol 0.1 run, noiseless constant"] - e_true
assert abs(bias_a - 0.7578) < 5e-5, bias_a       # the depolarizing anchor
e_swap = e_est["R1 run, R2 constant (dephasing 0.1)"]
assert abs(e_swap + 6.49) < 5e-3, e_swap         # F.3.1's worked example
assert abs(e_true + 5.23) < 5e-3, e_true
assert abs((e_swap - e_true) + 1.27) < 5e-3, e_swap - e_true
b_12 = e_est["R1 run, R2 constant (probe)"] - e_true
b_21 = e_est["R2 run, R1 constant (probe)"] - e_true
assert abs(b_12 - 1.04) < 5e-3 and abs(b_21 + 1.33) < 5e-3, (b_12, b_21)

print("calibration constant mismatch (Appendix F.3.1's mismatch paragraph), on that")
print("appendix's own exact estimator -- its TFIM ground state, "
      f"E = {e_true:.4f}:")
print("  per-site operator identity sum_k (s_ka/eta_cal) E~_k = "
      "(kappa_run/kappa_cal)")
print(f"  sigma_a: max |dev| = {d_op:.1e}")
print("  weight-w law, term-exact on every observable, six directed "
      "pairs, SIC and")
print(f"  icosahedron: max |dev| = {worst_law:.1e}; solid-blind at "
      f"{worst_solid:.1e}")
print(f"  own constant -> exact (max |dev| = {worst_own:.1e}); the other "
      "protocol's -> a bias:")
print(f"    depol 0.1 under the noiseless dual:  E_TFIM bias "
      f"{bias_a:+.4f}  (exact, not sampled)")
print(f"    dephasing 0.1, R1 run / R2 constant (eta = "
      f"{kappa_deph / 3:.4f}):  E reads {e_swap:.4f}")
print(f"      against the true {e_true:.4f} -- bias "
      f"{e_swap - e_true:+.4f}, F.3.1's worked example")
print(f"    probe noise, the two directions:  {b_12:+.4f} / {b_21:+.4f}")
print("  F.3.1's decimals, off the channels: twirled-native reads eta = "
      f"{M2[0, 0] / 3:.3f} (dephasing 0.1)")
print(f"  and {eta_ad[0.1]:.3f} (amplitude damping 0.1); randomized-projective "
      "reads the damping at")
print("  (1-g)/3 exactly, both rates")
print("[ok] a constant carried across protocols is a bias, exactly")
print("     (kappa_run/kappa_cal)^w per weight-w term; learned on the run")
print("     itself it is no bias at all -- the separator between this "
      "check")
print("     and every premium above")

calibration constant mismatch (Appendix F.3.1's mismatch paragraph), on that
appendix's own exact estimator -- its TFIM ground state, E = -5.2263:
  per-site operator identity sum_k (s_ka/eta_cal) E~_k = (kappa_run/kappa_cal)
  sigma_a: max |dev| = 2.2e-16
  weight-w law, term-exact on every observable, six directed pairs, SIC and
  icosahedron: max |dev| = 5.9e-15; solid-blind at 5.2e-15
  own constant -> exact (max |dev| = 1.7e-14); the other protocol's -> a bias:
    depol 0.1 under the noiseless dual:  E_TFIM bias +0.7578  (exact, not sampled)
    dephasing 0.1, R1 run / R2 constant (eta = 0.2889):  E reads -6.4942
      against the true -5.2263 -- bias -1.2679, F.3.1's worked example
    probe noise, the two directions:  +1.0384 / -1.3324
  F.3.1's decimals, off the channels: twirled-native reads eta = 0.289 (dephasing 0.1)
  and 0.311 (amplitude damping 0.1); randomized-projective reads the damping at
  (1-g)/3 exactly, both rates
[ok] a constant carried across protocols is a bia

### 1c. Finding 6 — gate noise separates the protocols a second time, as an *order* — the reframing, fifth application

Measurement-side noise is one channel shared by every draw: Schur applies, and §1a is exact.
Per-gate noise arrives *inside* the drawn circuit, in an amount and orientation correlated with
$g$ — there is no longer one fixed map to average, and Schur is silent. This is where Appendix
F's numerical study found the twirled-native $Z_0$ residual second order in the damping
$\gamma$ where the projective one is linear; the check below proves that claim as a theorem —
over a *generic* symbolic state, a *generic* dilation strength, and **every** choice of atlas
representative — so it covers the study rather than reproducing it (nothing is read from its
npz).

The mechanism: to first order every insertion reaches the estimator through its word *prefix*
alone — post-processing hits the insertion with $R_g^\top = (S_iP_i)^\top$ and the suffix
cancels, $R_g^\top S_i = P_i^\top$ — so the first-order displacement is a sum over the prefix
multiset of the drawn words. Over the $T$ draw that sum has no $z$ component for any of the
$2^{12}$ representative choices, and the first-order matrix term is *diagonal* — scalar on the
bare row alone — so the $|0\rangle$ calibration, one scalar off the $zz$ entry, absorbs what
the $Z_0$ column sees; the $Z_0$ residual therefore starts at $\gamma^2$. The verdict needs
both the protocol *and* the draw — R1 on the same words is linear, and R2 over the
$O$ draw is linear again, for all $2^{24}$ representative choices.

(The two estimator helpers are deliberately written in the shadow study's normalisation — the
calibration scalar $\eta$, noiseless $1/3$ — rather than $\kappa$; the calibrated residual is a
ratio and cannot see the convention.)

In [14]:
# === Gate-noise machinery (lifted from randomized_scalars.py) ===

# The BFS alphabets of the two atlases used here: 2T is <X, Z, F>, 2O adds H
# and S.  (2I adds Phi at depth <= 4; the theorem needs only the T and O
# draws, so it is left out rather than paid for.)  atlas_gates() is left
# alone deliberately -- four checks iterate it and would change if it grew.
NOISE_GATES = ("X", "Z", "F", "H", "S")

_NOISE_ROT = {}


def _noise_gate_rotations():
    """Exact SO(3) matrices of NOISE_GATES, memoized (bloch_matrix simplifies)."""
    if not _NOISE_ROT:
        G = dict(atlas_gates())
        G["H"] = (1 / sqrt(2)) * Matrix([[1, 1], [1, -1]])
        G["S"] = Matrix([[1, 0], [0, sI]])
        _NOISE_ROT.update({n: bloch_matrix(G[n]) for n in NOISE_GATES})
    return _NOISE_ROT


def _parse_word(seq):
    """Atlas sequence string -> [(gate, dagger), ...] in operator order.

    'F X' means F.X, so X acts first; a trailing dagger marks the SU(2)
    inverse.  main.py's convention, and the one shadow_experiments.py reads.
    """
    return [] if str(seq) == "I" else [(t.rstrip("†"), t.endswith("†"))
                                       for t in str(seq).split()]


def exact_word_rotation(tokens, ROT):
    """SO(3) matrix of a word, exactly -- leftmost token outermost."""
    M = sp.eye(3)
    for base, dag in tokens:
        M = M * (ROT[base].T if dag else ROT[base])
    return M


def exact_draw(g, mode="bfs"):
    """One representative word per SO(3) element of g, with its exact rotation.

    The +-q pair collapses by EXACT rotation equality: T's and O's rotations
    are integer signed permutations, so the key is a tuple of integers --
    canonical and hashable with no field needed.  Shallowest word wins, first
    atlas row breaks ties.  That is shadow_experiments.load_series's selection
    rule with its 9-decimal rounding key removed, and the rotations are built
    from the WORDS rather than lifted from floats, so no grid enters at all.
    Both directions are then checked against the atlas.
    """
    atlas = load_atlas(g)
    toks = [_parse_word(s) for s in atlas[f"{mode}_sequences"]]
    if mode == "bfs":                    # the stored depth IS the word length
        assert all(len(t) == d for t, d in zip(toks, atlas["bfs_depths"])), g
    ROT = _noise_gate_rotations()
    reps = {}
    for i, tk in enumerate(toks):
        k = tuple(exact_word_rotation(tk, ROT))
        if k not in reps or len(tk) < len(toks[reps[k]]):
            reps[k] = i
    idx = sorted(reps.values())
    Rs = [exact_word_rotation(toks[i], ROT) for i in idx]
    assert {tuple(M) for M in Rs} == {tuple(M) for M in exact_rotations(g)}, g
    for i, M in zip(idx, Rs):            # ... and each word IS its atlas row
        f = rotation_from_unitary(atlas["unitaries"][i])
        assert np.abs(np.array(M, dtype=float) - f).max() < 1e-9, (g, mode, i)
    return [toks[i] for i in idx], Rs


def gate_noise_channel(tokens, gam, ROT):
    """Affine Bloch channel of the word with amplitude damping after each gate.

    Damping is r -> diag(c, c, c^2) r + (0, 0, 1-c^2) with c = sqrt(1-gamma),
    exactly; shadow_experiments.chan_amp_damping is its float twin.  Tokens
    act rightmost-first, so the leftmost gate is applied last.
    """
    Tn, tn = sp.diag(sqrt(1 - gam), sqrt(1 - gam), 1 - gam), Matrix([0, 0, gam])
    T, t = sp.eye(3), sp.zeros(3, 1)
    for base, dag in reversed(tokens):
        R = ROT[base].T if dag else ROT[base]
        T, t = Tn * (R * T), Tn * (R * t) + tn
    return T, t


def _estimator_R1(Rs, chans, v):
    """R1's estimator channel under g-correlated noise, in eta units.

    The same average channel_R1 forms -- readout probability times snapshot
    vertex -- but with the noise INSIDE the word rather than after it.  Written
    in the shadow module's normalisation (ideal eta = 1/3, not kappa = 1); the
    calibrated residual below is a ratio and cannot see the convention.
    """
    n = len(Rs)
    M = sum((Matrix(3, 3, lambda a, b: (R.T * v)[a] * (T.T * v)[b])
             for R, (T, _) in zip(Rs, chans)), sp.zeros(3, 3)) / n
    m = sum(((R.T * v) * v.dot(t) for R, (_, t) in zip(Rs, chans)),
            sp.zeros(3, 1)) / n
    return M, m


def _estimator_R2(Rs, chans, TN=None, tN=None):
    """R2's estimator channel under the same g-correlated noise.

    The solid enters only through (1/V) sum_k n_k n_k^T, which is Id_3/3 for
    all five (each is at least a 2-design) -- so it contracts away and the
    channel is solid-independent, the fact the shadow study asserts row by row
    at 1e-12 and this makes structural.  (TN, tN) model the fixed dilation as
    one g-independent pre-measurement channel.
    """
    n = len(Rs)
    if TN is None:
        TN, tN = sp.eye(3), sp.zeros(3, 1)
    M = sum((R.T * TN * T for R, (T, _) in zip(Rs, chans)), sp.zeros(3, 3)) / (3 * n)
    m = sum((R.T * (TN * t + tN) for R, (_, t) in zip(Rs, chans)),
            sp.zeros(3, 1)) / (3 * n)
    return M, m


def calibrated_residual(M, m, r):
    """(M r + m)/eta - r: the bias surviving the scalar |0> calibration."""
    return (M * r + m) / (M[2, 2] + m[2]) - r


def _taylor(X, var, k):
    """Exact k-th Taylor coefficient of a matrix/vector in var at var = 0."""
    return X.applyfunc(lambda e: sp.simplify(sp.diff(e, var, k).subs(var, 0))
                       / sp.factorial(k))


def _prefix_sum(tokens, ROT):
    """sum over the word's prefixes of P^T zhat -- one word's share of m1.

    The derivation this checks: to first order the insertion after gate i
    contributes S_i zhat to t_g, and post-processing hits it with R_g^T =
    (S_i P_i)^T, so R_g^T S_i = P_i^T S_i^T S_i = P_i^T.  Every insertion
    reaches the estimator through its PREFIX alone; the suffix cancels.
    """
    tot = Matrix([0, 0, 0])
    P = sp.eye(3)
    for base, dag in reversed(tokens):
        P = (ROT[base].T if dag else ROT[base]) * P
        tot += P.T * Matrix([0, 0, 1])
    return tot


def _mz_reachable(g, mode):
    """{(m1) over EVERY choice of atlas representative}, by Minkowski sum.

    Each SO(3) element offers two atlas words (the +-q pair), so there are
    2^|G| representative sets -- 2^24 for O.  m1 is a SUM of independent
    per-element contributions, so the reachable set is a Minkowski sum and a
    running set computes it without enumerating the choices.
    """
    atlas = load_atlas(g)
    toks = [_parse_word(s) for s in atlas[f"{mode}_sequences"]]
    ROT = _noise_gate_rotations()
    by_rot = {}
    for i, tk in enumerate(toks):
        by_rot.setdefault(tuple(exact_word_rotation(tk, ROT)), []).append(i)
    assert all(len(v) == 2 for v in by_rot.values()), g   # exactly the +-q pair
    reach = {(0, 0, 0)}
    for k in sorted(by_rot, key=str):
        opts = {tuple(_prefix_sum(toks[i], ROT)) for i in by_rot[k]}
        reach = {tuple(a + b for a, b in zip(r, o)) for r in reach for o in opts}
    scale = Rational(1, 3 * len(by_rot))
    return {tuple(sp.nsimplify(c) * scale for c in r) for r in reach}


def _exact_seed(solid):
    """The alignment vertex v0, exactly -- via the field, never via nsimplify.

    A float alignment() lifted by nsimplify would be a normalising heuristic in
    front of an exact computation, which the exact layer never admits;
    exact_alignment already supplies the seed canonically, tie-break and all
    (the vertex order it breaks the tie on is load-bearing).
    """
    K = solid_field(solid)
    _, v = exact_alignment(exact_vertices(solid, K), K)
    return Matrix([K.to_sympy(c) for c in v])

In [15]:
# === The gate-noise theorem (check_gate_noise_residual) ===

# Appendix F's gate-noise finding, as a theorem: the twirled-native Z0
# residual starts at gamma^2 while the projective one is linear. Proved
# over a GENERIC state r = (x, y, z), so it is a statement about the
# protocol rather than about the study's TFIM test vector.
gam, dil = sp.symbols("gamma delta", nonnegative=True)
r = Matrix(sp.symbols("x y z", real=True))
ROT = _noise_gate_rotations()
third = sp.eye(3) / 3

draws, chans = {}, {}
for g in ("T", "O"):
    toks, Rs = exact_draw(g)
    draws[g] = (toks, Rs)
    chans[g] = [gate_noise_channel(tk, gam, ROT) for tk in toks]

# the dilation modeled generically: damping of arbitrary strength delta,
# so the study's GAMMA_DIL = 0.05 is one point of a proved family
TN = sp.diag(sqrt(1 - dil), sqrt(1 - dil), 1 - dil)

rows = []
for label, M, m in (
        ("R1 projective  O draw / octahedron",
         *_estimator_R1(draws["O"][1], chans["O"], Matrix([0, 0, 1]))),
        ("R1 projective  T draw / icosahedron",
         *_estimator_R1(draws["T"][1], chans["T"], _exact_seed("icosahedron"))),
        ("R2 native      T draw / bare",
         *_estimator_R2(draws["T"][1], chans["T"])),
        ("R2 native      T draw / dilation delta",
         *_estimator_R2(draws["T"][1], chans["T"], TN, Matrix([0, 0, dil]))),
        ("R2 native      O draw / bare (control)",
         *_estimator_R2(draws["O"][1], chans["O"])),
        # the like-for-like control of the row Appendix F prints: the O
        # words in front of the same modeled dilation (asserted after the
        # dilated T row's constants; appended last so rows[2], rows[3] hold)
        ("R2 native      O draw / dilation delta",
         *_estimator_R2(draws["O"][1], chans["O"], TN, Matrix([0, 0, dil])))):
    M0, m0 = _taylor(M, gam, 0), _taylor(m, gam, 0)
    M1, m1 = _taylor(M, gam, 1), _taylor(m, gam, 1)
    assert sp.simplify(M0 - M0[2, 2] * sp.eye(3)) == sp.zeros(3, 3), label
    assert m0 == sp.zeros(3, 1), label       # gamma = 0 twirls exactly
    c1 = _taylor(calibrated_residual(M, m, r), gam, 1)
    rows.append((label, M1, m1, c1, M, m))

lin = {label: sp.simplify(c1[2]) for label, _, _, c1, _, _ in rows}
print(f"  {'draw / protocol':38s} {'M1 diag':>7s} {'(m1)_z':>22s}"
      f"  Z0 residual")
print("  " + "-" * 88)
for label, M1, m1, _, _, _ in rows:
    diag = all(M1[a, b] == 0 for a in range(3) for b in range(3) if a != b)
    print(f"  {label:38s} {('yes' if diag else 'NO'):>7s} {str(m1[2]):>22s}"
          f"  {'SECOND ORDER' if lin[label] == 0 else 'linear'}")
print("\n  d/dgamma of the |0>-calibrated Z0 residual, exact and for a generic state:")
for label in lin:
    print(f"    {label:38s} {sp.collect(sp.expand(lin[label]), sqrt(5))}")

# the two R2 rows on the T draw vanish identically in (x, y, z) -- and the
# dilated one for EVERY delta, not just the study's 0.05
assert lin["R2 native      T draw / bare"] == 0
assert lin["R2 native      T draw / dilation delta"] == 0
# ... and the three others do not, so the vanishing is not vacuous
assert lin["R1 projective  O draw / octahedron"] == (1 - r[2]) / 4
assert lin["R2 native      O draw / bare (control)"] == (1 - r[2]) / 6
assert lin["R1 projective  T draw / icosahedron"].has(r[1])   # not even diagonal

# the exact constants behind those verdicts, and the sharpness of gamma^2.
# They are the BARE row's; the DILATED row Appendix F prints has its own,
# asserted straight after. The VANISHING survives the dilation (asserted
# above, over every delta); the constants do not.
label_b, M1b, m1b, _, Mb, mb = rows[2]
assert label_b.endswith("/ bare")      # not the dilated row Appendix F prints
assert M1b == -third and m1b == Matrix([1, 1, 0]) / 18
quad = _taylor(calibrated_residual(Mb, mb, r), gam, 2)
assert sp.simplify(quad[2]) == (r[2] - 1) / 12          # zero only at |0>
assert sp.simplify(_taylor(calibrated_residual(Mb, mb, r), gam, 1)[0]) == Rational(1, 6)
print("\n  twirled-native on the T draw, bare row: M1 = -Id_3/3, m1 = (1,1,0)/18.")
print("  M1 is SCALAR, so the scalar |0> calibration absorbs it entirely and the")
print("  linear Z0 coefficient collapses to (m1)_z (1-z)/eta_0 -- which is 0. What")
print("  survives at first order is transverse and state-independent: the X0 slope")
print("  is (1/18)/(1/3) = 1/6 exactly. Z0 resumes at gamma^2 with coefficient")
print("  (z-1)/12, vanishing only at z = 1, the calibration state itself.")

# The same two constants for the DILATED row -- the one Appendix F prints,
# so they belong in an assert rather than in a sentence about one. delta is
# carried symbolically through the derivation, exactly as above, and fixed
# at the study's 1/20 only here, where the constants live.
label_d, _, _, _, Md, md = rows[3]
assert label_d.endswith("T draw / dilation delta")
res_d = calibrated_residual(Md.subs(dil, Rational(1, 20)),
                            md.subs(dil, Rational(1, 20)), r)
quad_d, slope_d = _taylor(res_d, gam, 2)[2], _taylor(res_d, gam, 1)[0]
assert sp.simplify(quad_d - (4 * sqrt(95) - 19) * (r[2] - 1) / 244) == 0
assert sp.simplify(slope_d - (Rational(295, 488) - 15 * sqrt(95) / 244) * r[0]
                   - (4 * sqrt(95) - 19) / 122) == 0
print("\n  the dilation moves both constants and neither verdict. At delta = 1/20")
print("  the Z0 gamma^2 coefficient is (4 sqrt(95) - 19)(z - 1)/244, which is")
print(f"  {float(quad_d.subs(r[2], 0)):.8f} at z = 0, and the X0 slope gains an x:")
print(f"    {slope_d}")

# F.3.4's mechanism sentence -- "to first order in gamma the 2T words'
# estimator channel is a diagonal matrix, whose zz-entry the calibration
# absorbs, plus an offset with no z-component, where the 2O words' offset
# has one" -- on the DILATED row Appendix F prints, delta still symbolic,
# so "for every state and modeled dilation strength" is the theorem and
# not the study's 1/20. Exact identities, no tolerance.
# (a) M1 of the dilated T row is DIAGONAL -- and diagonal is the word, not
#     scalar: transverse entries 13 delta/72 - 11 sqrt(1-delta)/72 - 13/72
#     against a zz-entry delta/9 - 2 sqrt(1-delta)/9 - 1/9, apart by
#     5 sqrt(1-delta) (1 - sqrt(1-delta))/72 > 0 on all of (0, 1), so on
#     [0, 1) it is scalar at delta = 0 alone (the bare row's -Id_3/3). The
#     calibration divides the Z0 column by M[2,2] + m[2] and nothing else,
#     so a diagonal M1 is exactly what it absorbs there; scalar was never
#     what the vanishing needed.
# (b) (m1)_z of that row is 0 identically in delta; the transverse entries
#     are (1 - delta)/18, the dilation shrinking them without tilting.
# (c) the bare O row's (m1)_z is 1/18 -- the z-component "the 2O words'
#     offset has", and the reason its lin[...] above is (1 - z)/6.
M1_dil, m1_dil = _taylor(Md, gam, 1), _taylor(md, gam, 1)
assert all(M1_dil[a, b] == 0 for a in range(3) for b in range(3) if a != b)
xx_dil = 13 * dil / 72 - 11 * sqrt(1 - dil) / 72 - Rational(13, 72)
zz_dil = dil / 9 - 2 * sqrt(1 - dil) / 9 - Rational(1, 9)
assert sp.simplify(M1_dil[0, 0] - xx_dil) == 0
assert sp.simplify(M1_dil[1, 1] - xx_dil) == 0
assert sp.simplify(M1_dil[2, 2] - zz_dil) == 0
assert sp.simplify(xx_dil - zz_dil
                   - 5 * sqrt(1 - dil) * (1 - sqrt(1 - dil)) / 72) == 0
assert sp.simplify(M1_dil.subs(dil, 0) + third) == sp.zeros(3, 3)
assert m1_dil[2] == 0
assert sp.simplify(m1_dil[0] - (1 - dil) / 18) == 0
assert sp.simplify(m1_dil[1] - (1 - dil) / 18) == 0
label_o, _, m1_o, _, _, _ = rows[4]
assert label_o.endswith("O draw / bare (control)")
assert m1_o[2] == Rational(1, 18)
# (d) The sentence's other restoration, "taking the twirled-native average
#     over 2O restores [the linear term]", was asserted above on the BARE O
#     row alone. On the dilated O row -- the like-for-like control of the
#     row F prints -- the linear Z0 coefficient is
#     (1 - z)(1 - delta + sqrt(1-delta)) / (4 (1 - delta + 2 sqrt(1-delta)))
#     exactly: (1 - z)/6 at delta = 0, and nonzero for every z < 1 and
#     every delta in [0, 1), both factors positive there; at the study's
#     1/20 on the maximally mixed state it reads sqrt(95)/122 + 21/244 =
#     0.166. Its (m1)_z is (1 - delta + sqrt(1-delta))/36: the 2O offset
#     keeps its z-component under the dilation; the 2T offset never had one.
label_od, _, m1_od, _, _, _ = rows[5]
assert label_od.endswith("O draw / dilation delta")
lin_od = ((1 - r[2]) * (1 - dil + sqrt(1 - dil))
          / (4 * (1 - dil + 2 * sqrt(1 - dil))))
assert sp.simplify(lin[label_od] - lin_od) == 0
assert sp.simplify(lin_od.subs(dil, 0) - (1 - r[2]) / 6) == 0
v_od = lin[label_od].subs({dil: Rational(1, 20), r[0]: 0, r[1]: 0, r[2]: 0})
assert sp.simplify(v_od - (sqrt(95) / 122 + Rational(21, 244))) == 0
assert float(v_od) > 0.1                      # 0.166: linear, not vanishing
assert sp.simplify(m1_od[2] - (1 - dil + sqrt(1 - dil)) / 36) == 0
print("\n  on the dilated T row M1 is DIAGONAL but not scalar -- transverse entries")
print(f"    {xx_dil}")
print(f"  against a zz-entry {zz_dil}, apart by")
print("  5 sqrt(1-delta)(1 - sqrt(1-delta))/72 -- and its (m1)_z is 0 identically in")
print("  delta (transverse (1-delta)/18). The bare O row's (m1)_z is 1/18, the dilated")
print("  O row's (1 - delta + sqrt(1-delta))/36, whose linear Z0 coefficient")
print("  (1-z)(1 - delta + sqrt(1-delta))/(4(1 - delta + 2 sqrt(1-delta))) reads")
print(f"  {float(v_od):.6f} at delta = 1/20 on the maximally mixed state: linear again")

# The mechanism, checked rather than asserted: R_g^T S_i = P_i^T, so every
# insertion reaches the estimator through its prefix and m1 is a sum over
# prefixes. It is the PREFIX MULTISET that decides the verdict.
for g in ("T", "O"):
    toks, _ = draws[g]
    pre = sum((_prefix_sum(tk, ROT) for tk in toks), Matrix([0, 0, 0]))
    assert pre / (3 * len(toks)) == _taylor(
        _estimator_R2(draws[g][1], chans[g])[1], gam, 1), g

# ... which is why the verdict is a fact about the DRAW, not the protocol.
# Over every one of the 2^12 representative choices for T the z-share
# cancels; over all 2^24 for O it never does, in either compilation.
print(f"\n  {'draw':6s} {'mode':5s} {'#words':>6s} {'distinct m1':>12s}"
      f" {'(m1)_z over ALL 2^n representative choices':>46s}")
print("  " + "-" * 82)
for g, n in (("T", 12), ("O", 24)):
    for mode in ("bfs", "dij"):
        reach = _mz_reachable(g, mode)
        zs = sorted({m[2] for m in reach}, key=float)
        hit = sum(1 for m in reach if m[2] == 0)
        assert (hit == len(reach)) == (g == "T"), (g, mode)
        rng = f"{zs[0]}" if len(zs) == 1 else f"{zs[0]} ... {zs[-1]}"
        print(f"  {g:6s} {mode:5s} {n:>6d} {len(reach):>12d}"
              f" {rng + (' (always 0)' if hit == len(reach) else ' (never 0)'):>46s}")

# Transcription guard: the verdicts above are booleans and exact
# rationals, so pair them with a value-for-value agreement against a float
# recomputation of the same channels -- the shadow study's arithmetic.
worst = 0.0
for gv in (0.01, 0.05):
    for g in ("T", "O"):
        toks, Rs = draws[g]
        Rf = [np.array(M, dtype=float) for M in Rs]
        ROTf = {n: np.array(ROT[n], dtype=float) for n in NOISE_GATES}
        cf = []
        for tk in toks:
            Tn = np.diag([np.sqrt(1 - gv), np.sqrt(1 - gv), 1 - gv])
            tn = np.array([0.0, 0.0, gv])
            T, t = np.eye(3), np.zeros(3)
            for base, dag in reversed(tk):
                R = ROTf[base].T if dag else ROTf[base]
                T, t = Tn @ (R @ T), Tn @ (R @ t) + tn
            cf.append((T, t))
        Mf = np.mean([R.T @ T for R, (T, _) in zip(Rf, cf)], axis=0) / 3
        mf = np.mean([R.T @ t for R, (_, t) in zip(Rf, cf)], axis=0) / 3
        Me, me = _estimator_R2(Rs, chans[g])
        worst = max(worst,
                    np.abs(np.array(Me.subs(gam, gv).evalf(), dtype=float) - Mf).max(),
                    np.abs(np.array(me.subs(gam, gv).evalf(),
                                    dtype=float).ravel() - mf).max())
assert worst < 1e-12, worst
print(f"\n  transcription guard: the exact channels reproduce a float recomputation")
print(f"  of the same words at gamma = 0.01 and 0.05, max |difference| = {worst:.1e}")

print("[ok] the twirled-native Z0 residual is SECOND order in the per-gate damping")
print("     and the projective one is linear -- for every state, and for every")
print("     dilation strength, not at the study's seven gammas and one test vector.")
print("     It needs BOTH, and each half is shown necessary above: R1 over the SAME")
print("     T words is linear, and R2 over the O draw is linear again at (1-z)/6 --")
print("     for all 2^24 choices of representative. What the twirled-native average")
print("     buys is a SCALAR M1; what the T words add is a z-balanced prefix multiset")

  draw / protocol                        M1 diag                 (m1)_z  Z0 residual
  ----------------------------------------------------------------------------------------
  R1 projective  O draw / octahedron         yes                   1/12  linear
  R1 projective  T draw / icosahedron         NO     -sqrt(5)/40 - 1/24  linear
  R2 native      T draw / bare               yes                      0  SECOND ORDER
  R2 native      T draw / dilation delta     yes                      0  SECOND ORDER
  R2 native      O draw / bare (control)     yes                   1/18  linear
  R2 native      O draw / dilation delta     yes -delta/36 + sqrt(1 - delta)/36 + 1/36  linear

  d/dgamma of the |0>-calibrated Z0 residual, exact and for a generic state:
    R1 projective  O draw / octahedron     1/4 - z/4
    R1 projective  T draw / icosahedron    z/8 + sqrt(5)*(3*y/40 + 3*z/40 - 3/40) - 1/8
    R2 native      T draw / bare           0
    R2 native      T draw / dilation delta 0
    R2 n


  the dilation moves both constants and neither verdict. At delta = 1/20
  the Z0 gamma^2 coefficient is (4 sqrt(95) - 19)(z - 1)/244, which is
  -0.08191466 at z = 0, and the X0 slope gains an x:
    -15*sqrt(95)*x/244 + 295*x/488 - 19/122 + 2*sqrt(95)/61

  on the dilated T row M1 is DIAGONAL but not scalar -- transverse entries
    13*delta/72 - 11*sqrt(1 - delta)/72 - 13/72
  against a zz-entry delta/9 - 2*sqrt(1 - delta)/9 - 1/9, apart by
  5 sqrt(1-delta)(1 - sqrt(1-delta))/72 -- and its (m1)_z is 0 identically in
  delta (transverse (1-delta)/18). The bare O row's (m1)_z is 1/18, the dilated
  O row's (1 - delta + sqrt(1-delta))/36, whose linear Z0 coefficient
  (1-z)(1 - delta + sqrt(1-delta))/(4(1 - delta + 2 sqrt(1-delta))) reads
  0.165957 at delta = 1/20 on the maximally mixed state: linear again

  draw   mode  #words  distinct m1     (m1)_z over ALL 2^n representative choices
  ----------------------------------------------------------------------------------
  T      bf

### 1d. The alignment, met here and priced in §3

R1's fixed alignment $A$ is the identity exactly when $\hat z$ is already a vertex — the
octahedron, and only it. For every other solid $A$ is not even an element of the covariance
group, so it is genuinely an extra fixed rotation the projective route appends to every drawn
word — and §3 will show it is
*inexact* over every thesis gate set, for every vertex choice. That inexactness is where R1
hides the same field extension R2 pays as Decker's nested radicals: same magic, two hiding
places.

In [16]:
# === The alignment per solid (check_alignment) ===

for solid in ("octahedron", "cube", "icosahedron", "dodecahedron"):
    s = load_vertices(solid)
    R = load_rotations(COVARIANCE[solid])
    A, v = alignment(s)
    if solid == "octahedron":
        d_I = np.abs(A - np.eye(3)).max()        # measured 0.0 exactly
        assert d_I < 1e-12, f"{solid}: A != I (max |dev| = {d_I:.2e})"
        print(f"  {solid:14s} v0 = zhat already: A = I")
    else:
        gap = min(np.linalg.norm(A - Rg) for Rg in R)
        assert gap > 1e-3, f"{solid}: A is a covariance-group element?!"
        print(f"  {solid:14s} A: v0 = {np.round(v, 4)} -> zhat;"
              f"  min ||A - R_g|| = {gap:.3f}  (not in G)")
print("[ok] A = I iff octahedron; otherwise A is no covariance-group element -- and")
print("     no thesis gate set synthesizes it exactly (vertex-by-vertex, section 3)")

  octahedron     v0 = zhat already: A = I
  cube           A: v0 = [-0.5774 -0.5774  0.5774] -> zhat;  min ||A - R_g|| = 1.300  (not in G)
  icosahedron    A: v0 = [0.5257 0.     0.8507] -> zhat;  min ||A - R_g|| = 0.773  (not in G)
  dodecahedron   A: v0 = [0.     0.3568 0.9342] -> zhat;  min ||A - R_g|| = 0.513  (not in G)
[ok] A = I iff octahedron; otherwise A is no covariance-group element -- and
     no thesis gate set synthesizes it exactly (vertex-by-vertex, section 3)


## 2. The two jobs of randomness — the reframing, second application

Finding 4. The draw's randomness does two jobs at once, and they are *independent properties of
which maps you average*:

- **Realize.** R1 has no ancilla, so the POVM itself must emerge from the draw: outcome
  $(g, b)$ lands the snapshot on $3b\,R_g^\top v_0$, and the drawn set realizes the POVM iff that
  coin hits every vertex uniformly — a $V/2$-way coin over coset representatives. Needs
  transitivity on the vertex axes.
- **Twirl.** The estimator channel collapses to one scalar iff the drawn set acts irreducibly —
  Schur's hypothesis, nothing about vertices at all.

The first check below shows the coin reproduces the POVM's *effects* — the operators, not
merely the statistics — exactly, solid by solid; the second shows the minimal irreducible draw
$T$ twirling every solid while its orbit *fails to realize* the dodecahedron (it covers 6 of the
10 vertex axes: a perfectly unbiased measurement of the wrong POVM). R1's one draw must do both
jobs, so its bill is the larger of two independent bars.

In [17]:
# === The coin realizes the POVM itself (check_coset_coin) ===

solids_sym = symbolic_solids()
for solid in SOLIDS:
    verts = solids_sym[solid]
    V = len(verts)
    paired = all(any(sp.simplify(v + w) == Matrix([0, 0, 0]) for w in verts)
                 for v in verts)
    if not paired:
        assert solid == "tetrahedron"
        print(f"  {solid:14s} indecomposable -- no coin realization; Naimark forced")
        continue
    effects = [Rational(2, V) * state_from_bloch(v) for v in verts]
    assert sp.simplify(sum(effects, sp.zeros(2, 2)) - sp.eye(2)) == sp.zeros(2, 2)
    # the coin: pick one of V/2 axes w.p. 2/V, measure the {v, -v} basis
    axes = []
    for v in verts:
        if not any(sp.simplify(v + u) == Matrix([0, 0, 0])
                   or sp.simplify(v - u) == Matrix([0, 0, 0]) for u in axes):
            axes.append(v)
    coin = [Rational(2, V) * state_from_bloch(sgn * v)
            for v in axes for sgn in (1, -1)]
    remaining = list(effects)    # effect-for-effect multiset equality
    for C in coin:
        hit = next(j for j, E in enumerate(remaining)
                   if sp.simplify(C - E) == sp.zeros(2, 2))
        remaining.pop(hit)
    assert not remaining
    print(f"  {solid:14s} {V // 2}-way coin + projective readout"
          f" = the same {V} effects, exactly")
print("[ok] the coin realizes the POVM itself (the effects, not merely the statistics)")

  tetrahedron    indecomposable -- no coin realization; Naimark forced
  octahedron     3-way coin + projective readout = the same 6 effects, exactly


  cube           4-way coin + projective readout = the same 8 effects, exactly


  icosahedron    6-way coin + projective readout = the same 12 effects, exactly


  dodecahedron   10-way coin + projective readout = the same 20 effects, exactly
[ok] the coin realizes the POVM itself (the effects, not merely the statistics)


In [18]:
# === The minimal twirl, and where realization fails (check_minimal_twirl) ===

R_T = load_rotations("T")
kappa_R1 = T_NOISE[2, 2]
print(f"R1 with the 12-rotation T draw (|2T| = 24 elements, all Clifford):")
for solid in ("octahedron", "cube", "icosahedron", "dodecahedron"):
    s = load_vertices(solid)
    hits = orbit_counts(s, R_T)
    # the twirl works regardless: Schur needs only irreducibility, which
    # T has -- exactly depolarizing at the same kappa = T_zz, zero offset
    M, o = channel_R1(s, R_T, T_NOISE, t_NOISE)
    d_M = np.abs(M - kappa_R1 * np.eye(3)).max()      # measured 2.2e-16
    assert d_M < 1e-10, f"{solid}: T draw not depol at T_zz (max |dev| = {d_M:.2e})"
    assert np.allclose(o, 0, atol=1e-9)
    if solid == "dodecahedron":
        # ... but the REALIZATION fails: the T-orbit of v0 covers only
        # 6 of the 10 vertex axes -- 8 vertices are never measured, so
        # this is a (perfectly unbiased) 6-axis measurement, NOT the
        # dodecahedral POVM. R1's one draw must do both jobs, so the
        # dodecahedron forces the full 2I: transitivity on 10 axes needs
        # order divisible by 10 (T's 12 fails), leaving D5 as the only
        # proper candidate -- and its invariant C5 axis line makes it
        # reducible, so it cannot twirl.
        assert sorted(set(hits.tolist())) == [0, 2] and int((hits > 0).sum()) == 12
        print(f"  {solid:14s} twirl OK (kappa = T_zz) -- but orbit covers 6/10 axes:")
        print(f"  {'':14s} NOT the dodecahedral POVM; realization forces the full 2I draw")
        continue
    assert hits.min() == hits.max() == 24 // len(s), hits
    print(f"  {solid:14s} orbit uniform x{hits[0]}; depolarizing at kappa = {M[0, 0]:.6f}")
print("[ok] g ~ Unif(T) realizes AND twirls octahedron/cube/icosahedron at kappa = T_zz;")
print("     the dodecahedron is the sole solid whose projective route needs 2I")

R1 with the 12-rotation T draw (|2T| = 24 elements, all Clifford):
  octahedron     orbit uniform x4; depolarizing at kappa = 0.620000
  cube           orbit uniform x3; depolarizing at kappa = 0.620000
  icosahedron    orbit uniform x2; depolarizing at kappa = 0.620000
  dodecahedron   twirl OK (kappa = T_zz) -- but orbit covers 6/10 axes:
                 NOT the dodecahedral POVM; realization forces the full 2I draw
[ok] g ~ Unif(T) realizes AND twirls octahedron/cube/icosahedron at kappa = T_zz;
     the dodecahedron is the sole solid whose projective route needs 2I


### The sweep: exhaustive over every finite subgroup of $SO(3)$

The order-counting argument gets a brute-force replacement: *every* subgroup of $O$ and of $I$
— complete lattices, found by pair closure and verified complete in situ — tested independently
for the two jobs on all four decomposable solids. The sweep is exhaustive over every finite
subgroup of $SO(3)$, not merely over the covariance groups: a draw that realizes permutes the
vertex set, hence already lies inside the solid's rotation group.

Three things fall out, all printed below: the twirl bar is flat at $T$ while the realize bar
climbs $3 \to 4 \to 12 \to 60$, crossing at the icosahedron — so it is the *twirl* that forces
the octahedron's and the cube's draws up to order 12 (both realize already, at orders 3 and 4),
both bars bind at the icosahedron, and realization alone binds for the dodecahedron — never
"the $T$ draw happens to also realize three of them"; the protocol's twirl test
reproduces Schur *exactly* (twirling $=$ irreducible, subgroup for subgroup); and the two sets
**nest**, the nesting inverting exactly at the icosahedron, where they coincide. The
dodecahedron block makes the five-inscribed-cubes argument quantitative — no proper subgroup
reaches more than 6 of its 10 axes, so realization alone convicts it.

(The float sweep decides by tolerance three times over — lattice grid, orbit argmin, twirl
`allclose`. Its exact companion `check_exact_two_bars` lifts all three mechanisms to canonical
form and quantifies the twirl over every $(T, t)$; it is not staged here — the final cell's
`main()` runs it.)

In [19]:
# === Sweep machinery (lifted from core + twojobs) ===

def subgroup_lattice(R):
    """All subgroups of the rotation group R, as frozensets of indices.

    Pair closure finds them: close every generator pair {g, h} under
    multiplication. Completeness is then verified in situ, with no input
    from the classification: adjoining any single element to any subgroup
    found must land back in the family (induction on generating sets does
    the rest), so the returned lattice is provably the whole one.
    """
    n = len(R)
    idx = {rot_key(Rg): i for i, Rg in enumerate(R)}
    mul = np.array([[idx[rot_key(Ri @ Rj)] for Rj in R] for Ri in R])

    def close(gens):
        S = set(gens)
        while True:
            new = set(mul[np.ix_(sorted(S), sorted(S))].ravel().tolist()) - S
            if not new:
                return frozenset(S)
            S |= new

    subs = {close((i, j)) for i in range(n) for j in range(i, n)}
    assert all(close((*S, g)) in subs
               for S in subs for g in range(n) if g not in S), "lattice incomplete"
    return subs


_LATTICE = {}


def lattice(g):
    """subgroup_lattice(load_rotations(g)), memoized (the sweep reuses it)."""
    if g not in _LATTICE:
        _LATTICE[g] = subgroup_lattice(load_rotations(g))
    return _LATTICE[g]


def subgroup_kind(R, S):
    """Structural name of a rotation subgroup, from its element orders.

    Enough of the classification to label the sweep: cyclic when some
    element has the group's own order, Klein V at order 4 otherwise, the
    three polyhedral groups by (order, top element order), dihedral for the
    rest. Element orders are integers read off a float power, and a
    rotation's powers stay well away from the identity until they hit it.
    """
    n = len(S)
    if n == 1:
        return "1"
    orders = []
    for i in sorted(S):
        k, M = 1, R[i]
        while not np.allclose(M, np.eye(3), atol=1e-9):
            M, k = M @ R[i], k + 1
        orders.append(k)
    top = max(orders)
    if top == n:
        return f"C_{n}"
    if n == 4:
        return "V"
    if (n, top) in ((12, 3), (24, 4), (60, 5)):
        return {12: "T", 24: "O", 60: "I"}[n]
    return f"D_{n // 2}"


def by_order(R, subs):
    """{order: (count, sorted structural names)} of a set of subgroups."""
    out = {}
    for S in sorted(subs, key=lambda S: (len(S), sorted(S))):
        cnt, names = out.get(len(S), (0, set()))
        out[len(S)] = (cnt + 1, names | {subgroup_kind(R, S)})
    return {k: (c, sorted(ns)) for k, (c, ns) in sorted(out.items())}


def two_bars(solid):
    """The two bars for one solid, over the complete subgroup lattice.

    REALIZE (the coin reproduces the POVM: the seed's orbit is the whole
    vertex set, uniformly) and TWIRL (the estimator channel is exactly
    depolarizing) are tested independently on every subgroup of the solid's
    covariance group. The minimal draw is the smallest group clearing both,
    i.e. the smallest member of the intersection; that this equals the
    larger of the two bars is asserted, not assumed (see below). `binds`
    names which bar sets it, `draw_groups` the subgroups that attain it --
    five at the icosahedron, one everywhere else.

    The sweep is exhaustive over every finite subgroup of SO(3), not merely
    over the covariance group: if G's orbit of the seed is the whole vertex
    set then G permutes that set, hence sits inside the solid's rotation
    group -- so a draw outside the lattice cannot realize at all.
    """
    g = COVARIANCE[solid]
    R = load_rotations(g)
    subs = lattice(g)
    s = load_vertices(solid)
    irr, realize, twirl = set(), set(), set()
    reach = {}
    for S in subs:
        RS = R[sorted(S)]
        # irreducibility is one number: <chi, chi> = mean tr(R)^2 = 1
        if np.isclose(np.mean([np.trace(R[i]) ** 2 for i in S]), 1.0):
            irr.add(S)
        hits = orbit_counts(s, RS)
        reach[S] = int((hits > 0).sum()) // 2         # vertex axes reached
        if hits.min() == hits.max():
            realize.add(S)
        M, o = channel_R1(s, RS, T_NOISE, t_NOISE)
        # a gap, not a boundary: accepted deviations reach 4.4e-16 over both
        # lattices, the closest miss is 2.5e-2
        if (np.abs(M - T_NOISE[2, 2] * np.eye(3)).max() < 1e-9
                and np.allclose(o, 0, atol=1e-9)):
            twirl.add(S)
    bar_r, bar_t = min(map(len, realize)), min(map(len, twirl))
    # The draw must clear BOTH bars at once, so it is the smallest member of
    # the intersection; max(bar_r, bar_t) is only a lower bound for that,
    # since nothing forces a minimal twirling group to also realize. The
    # bound is attained because the two sets NEST (check_subgroup_sweep
    # asserts the nesting: whichever set is contained in the other supplies
    # the minimum). The icosahedron is where that has teeth -- realize and
    # twirl are the same five T-conjugates, not merely the same order; had
    # the two picked different T's, both bars would still read 12 while the
    # smallest group clearing both jumped to 60.
    both = realize & twirl
    draw = min(map(len, both))
    assert draw == max(bar_r, bar_t), (solid, draw, bar_r, bar_t)
    return {
        "group": g, "rotations": R, "subgroups": subs, "axes": len(s) // 2,
        "irr": irr, "realize": realize, "twirl": twirl, "reach": reach,
        "bar_realize": bar_r, "bar_twirl": bar_t, "draw": draw,
        "binds": ("both" if bar_r == bar_t else
                  "realize" if bar_r > bar_t else "twirl"),
        "min_realize": sorted({subgroup_kind(R, S) for S in realize
                               if len(S) == bar_r}),
        "min_twirl": sorted({subgroup_kind(R, S) for S in twirl
                             if len(S) == bar_t}),
        "n_min_realize": sum(1 for S in realize if len(S) == bar_r),
        "draw_groups": {S for S in both if len(S) == draw},
        "ceiling": max(reach[S] for S in subs if len(S) < len(R)),
    }


def twirl_bar(solid):
    """Smallest order in the lattice whose R2 draw twirls this solid exactly.

    The twirl bar in its route-free form: R2 is defined for all five solids
    (R1 is not), so this is the one bar the tetrahedron also has.
    """
    g = COVARIANCE[solid]
    R, s = load_rotations(g), load_vertices(solid)
    kappa = np.trace(T_NOISE) / 3
    ok = set()
    for S in lattice(g):
        M, o = channel_R2(s, R[sorted(S)], T_NOISE, t_NOISE)
        # same classifier as two_bars: accepted deviations reach 8.9e-16 over
        # all five lattices, closest miss 1.1e-3
        if (np.abs(M - kappa * np.eye(3)).max() < 1e-9
                and np.allclose(o, 0, atol=1e-9)):
            ok.add(S)
    return min(map(len, ok)), ok


def _fmt_orders(d):
    """by_order output -> '3(x4 C_3), 12 (T)'."""
    return ", ".join(f"{o}(x{c} {'/'.join(ns)})" if c > 1 else f"{o} ({'/'.join(ns)})"
                     for o, (c, ns) in d.items())


def check_subgroup_sweep():
    # Finding 4, exhaustively and on BOTH sides. The order-counting argument
    # ("transitivity on 10 axes needs order divisible by 10 -> only D5 ->
    # reducible") gets a brute-force replacement: every subgroup of O and of
    # I, tested independently for the two jobs -- realize (uniform vertex
    # hits) and twirl (estimator channel exactly depolarizing under the
    # generic probe) -- on all four decomposable solids.
    #
    # Deliberately NOT reported: how BADLY a subgroup that fails to twirl
    # fails, i.e. the size of the residual anisotropy ||M - T_zz I||. That
    # number is not a property of the group. Rerun under a second probe and
    # the failures reorder completely -- on the dodecahedron the order-5 and
    # order-10 subgroups go from the worst reducible draws to the best. Only
    # the zero/nonzero dichotomy is real, and `twirl == irr` is exactly it.
    #
    # subgroup_kind reads element orders off a float matrix power, and its
    # NAMES reach two printed tables (the two bar cells of the ledger and of
    # the sweep). Pin the whole census per lattice: a misclassification is then
    # loud here instead of silently relabelling a cell -- V as C_4 costs the
    # cube's realize bar its second name and changes nothing else in the file.
    census = {"O": {"1": 1, "C_2": 9, "C_3": 4, "C_4": 3, "D_3": 4, "D_4": 3,
                    "T": 1, "V": 4, "O": 1},
              "I": {"1": 1, "C_2": 15, "C_3": 10, "C_5": 6, "D_3": 10, "D_5": 6,
                    "T": 5, "V": 5, "I": 1}}
    keys_T = frozenset(rot_key(Rg) for Rg in load_rotations("T"))
    for g, n_subs, iso in (("O", 30, "S_4"), ("I", 59, "A_5")):
        R = load_rotations(g)
        subs = lattice(g)
        assert len(subs) == n_subs, (g, len(subs))
        kinds = sorted(subgroup_kind(R, S) for S in subs)
        assert {k: len(list(v)) for k, v in itertools.groupby(kinds)} == census[g], g
        print(f"  the lattice of {g}: {len(subs)} subgroups -- "
              + _fmt_orders(by_order(R, subs)))
        # irreducibility is one number per subgroup: <chi,chi> = mean tr^2 = 1
        irr = {S for S in subs
               if np.isclose(np.mean([np.trace(R[i]) ** 2 for i in S]), 1.0)}
        assert {len(S) for S in irr} == {12, len(R)}
        for S in irr:                    # every order-12 one is a copy of T
            assert len(S) == len(R) or any(
                frozenset(rot_key(R[h] @ R[i] @ R[h].T) for i in S) == keys_T
                for h in range(len(R)))
        print(f"  {'':16s} irreducible: {_fmt_orders(by_order(R, irr))}"
              f" -- every order-12 one verified conjugate to T")
    print("  (completeness self-verified in situ; 30 and 59 are the known subgroup")
    print("   counts of S_4 and A_5, so each census doubles as a free cross-check)")

    print(f"\n  {'solid':14s} {'realizing subgroups':38s} {'twirling subgroups':18s}"
          f" {'bars':9s} {'binds':8s} minimal draw")
    print("  " + "-" * 104)
    # counts AND names: the name at the realize bar is a printed cell, so it is
    # pinned with the order counts rather than discarded from by_order's output.
    expect = {"octahedron": ({3: (4, ["C_3"]), 6: (4, ["D_3"]), 12: (1, ["T"]),
                              24: (1, ["O"])}, "twirl", 12),
              "cube": ({4: (4, ["C_4", "V"]), 8: (3, ["D_4"]), 12: (1, ["T"]),
                        24: (1, ["O"])}, "twirl", 12),
              "icosahedron": ({12: (5, ["T"]), 60: (1, ["I"])}, "both", 12),
              "dodecahedron": ({60: (1, ["I"])}, "realize", 60)}
    bars = {}
    for solid in ("octahedron", "cube", "icosahedron", "dodecahedron"):
        b = bars[solid] = two_bars(solid)
        R, subs = b["rotations"], b["subgroups"]
        irr = {S for S in subs
               if np.isclose(np.mean([np.trace(R[i]) ** 2 for i in S]), 1.0)}
        assert b["twirl"] == irr, solid          # the protocol test IS Schur
        counts, binds, draw = expect[solid]
        assert by_order(R, b["realize"]) == counts, solid
        assert b["min_twirl"] == ["T"], solid    # the other printed name cell
        assert b["binds"] == binds and b["draw"] == draw, solid
        print(f"  {solid:14s} {_fmt_orders(by_order(R, b['realize'])):38s}"
              f" {_fmt_orders(by_order(R, b['twirl'])):18s}"
              f" {b['bar_realize']:>2d} vs{b['bar_twirl']:>3d}"
              f" {b['binds']:>8s}"
              f" {'/'.join(b['min_realize'] if binds == 'realize' else b['min_twirl'])}"
              f" ({draw})")

    # The bars do not merely cross in size: the two SETS nest, and the
    # nesting inverts exactly at the icosahedron, where they coincide.
    for solid, rel in (("octahedron", "twirl<realize"), ("cube", "twirl<realize"),
                       ("icosahedron", "equal"), ("dodecahedron", "realize<twirl")):
        b = bars[solid]
        if rel == "equal":
            assert b["realize"] == b["twirl"], solid
        elif rel == "twirl<realize":
            assert b["twirl"] < b["realize"], solid
        else:
            assert b["realize"] < b["twirl"], solid
    print("\n  the two sets NEST, and the nesting inverts at the icosahedron:")
    print("    octahedron, cube   twirl   < realize   -- every twirling draw realizes")
    print("    icosahedron        twirl   = realize   -- the same six subgroups")
    print("    dodecahedron       realize < twirl     -- every realizing draw twirls")

    # The dodecahedron's ceiling, and the five-inscribed-cubes argument made
    # quantitative.
    b = bars["dodecahedron"]
    assert b["ceiling"] == 6, b["ceiling"]
    R, s = b["rotations"], load_vertices("dodecahedron")
    _, v0 = alignment(s)
    # The inscribed-cube identification is a COUNT of distinct orbit points and
    # a pairwise |cos| -- both identity questions, both decided on the rounding
    # grid below.  Exact companions run alongside and must agree.
    Kd = solid_field("dodecahedron")
    Rd = [to_field(M, Kd) for M in exact_rotations("I")]
    _, v0e = exact_alignment(exact_vertices("dodecahedron", Kd), Kd)
    assert np.abs(np.array([float(Kd.to_sympy(c)) for c in v0e]) - v0).max() < 1e-12
    split = {}
    for S in (S for S in b["subgroups"] if len(S) == 12):
        r = b["reach"][S]
        split[r] = split.get(r, 0) + 1
        if r == 4:                        # the orbit IS an inscribed cube
            orbit = np.unique(np.round([Rg.T @ v0 for Rg in R[sorted(S)]], 9), axis=0)
            gram = np.abs(orbit @ orbit.T)
            # the deviation here is not float noise but the 9-decimal round the
            # orbit dedup above needs: it caps |gram - 1/3| at 3e-9 by
            # construction, and 2.3e-10 is measured
            d_gram = np.abs(gram[np.triu_indices(4, 1)] - 1 / 3).max()
            assert d_gram < 1e-8, \
                f"orbit is no inscribed cube (max |dev| = {d_gram:.2e})"
            oe = exact_orbit_directions([Rd[i] for i in sorted(S)], v0e, Kd)
            assert len(oe) == len(orbit) == 4, (len(oe), len(orbit))
            assert all(sum((p[k] * q[k] for k in range(3)), Kd.zero)
                       in (Kd.one / 3, -Kd.one / 3)
                       for j, p in enumerate(oe) for q in oe[j + 1:])
    assert split == {4: 2, 6: 3}, split
    print(f"\n  dodecahedron: no proper subgroup reaches more than {b['ceiling']}/10 vertex axes")
    print("    the five T's split 2 + 3: two send v0 around the 4 body diagonals of an")
    print("    inscribed cube (pairwise |cos| = 1/3, verified), three around the other 6")
    print("    -- v0 lies on exactly 2 of I's 5 inscribed cubes. Realization ALONE")
    print("    convicts the dodecahedron; the twirl is not even needed.")

    # The icosahedron's near-misses: which of the twelve C_5's and D_5's fall
    # one axis short, which axis each misses, and why the other two do not.
    b = bars["icosahedron"]
    R, s = b["rotations"], load_vertices("icosahedron")
    near = [S for S in b["subgroups"] if b["reach"][S] == 5]
    assert {len(S) for S in near} == {5, 10} and len(near) == 10
    for S in near:
        RS = R[sorted(S)]
        hits = orbit_counts(s, RS)
        # the 72-degree element: trace 1 + 2 cos 72 = tau, the largest any
        # non-identity rotation of a subgroup of I attains
        C5 = max((M for M in RS if not np.allclose(M, np.eye(3), atol=1e-9)),
                 key=np.trace)
        w, V = np.linalg.eig(C5)
        ax = np.real(V[:, np.argmin(np.abs(w - 1))])
        assert all(min(np.linalg.norm(s[k] - ax), np.linalg.norm(s[k] + ax)) < 1e-9
                   for k in np.flatnonzero(hits == 0))
    print("\n  icosahedron: ten of the twelve C_5's and D_5's reach 5 of the 6 axes --")
    print("    the axis each misses is the one its five-fold rotation fixes. The other")
    print("    two are seed-aligned and never leave v0's own axis, reaching 1. Neither")
    print("    is the ceiling: the five order-12 T-conjugates reach all 6 and realize")

    print("\n[ok] exhaustive over all 30 + 59 subgroups, hence over EVERY finite subgroup")
    print("     of SO(3) (a draw that realizes permutes the vertex set, so it lies")
    print("     inside the solid's rotation group): the twirl bar is T for all four,")
    print("     the realize bar climbs 3 -> 4 -> 12 -> 60, and they cross at the")
    print("     icosahedron -- the order-counting argument, brute-forced and generalized")
    return bars


# bind the returned bars: their repr is a page of arrays
_ = check_subgroup_sweep()

  the lattice of O: 30 subgroups -- 1 (1), 2(x9 C_2), 3(x4 C_3), 4(x7 C_4/V), 6(x4 D_3), 8(x3 D_4), 12 (T), 24 (O)
                   irreducible: 12 (T), 24 (O) -- every order-12 one verified conjugate to T


  the lattice of I: 59 subgroups -- 1 (1), 2(x15 C_2), 3(x10 C_3), 4(x5 V), 5(x6 C_5), 6(x10 D_3), 10(x6 D_5), 12(x5 T), 60 (I)
                   irreducible: 12(x5 T), 60 (I) -- every order-12 one verified conjugate to T
  (completeness self-verified in situ; 30 and 59 are the known subgroup
   counts of S_4 and A_5, so each census doubles as a free cross-check)

  solid          realizing subgroups                    twirling subgroups bars      binds    minimal draw
  --------------------------------------------------------------------------------------------------------
  octahedron     3(x4 C_3), 6(x4 D_3), 12 (T), 24 (O)   12 (T), 24 (O)      3 vs 12    twirl T (12)
  cube           4(x4 C_4/V), 8(x3 D_4), 12 (T), 24 (O) 12 (T), 24 (O)      4 vs 12    twirl T (12)
  icosahedron    12(x5 T), 60 (I)                       12(x5 T), 60 (I)   12 vs 12     both T (12)
  dodecahedron   60 (I)                                 12(x5 T), 60 (I)   60 vs 12  realize I (60)

  the two sets NE


  dodecahedron: no proper subgroup reaches more than 6/10 vertex axes
    the five T's split 2 + 3: two send v0 around the 4 body diagonals of an
    inscribed cube (pairwise |cos| = 1/3, verified), three around the other 6
    -- v0 lies on exactly 2 of I's 5 inscribed cubes. Realization ALONE
    convicts the dodecahedron; the twirl is not even needed.

  icosahedron: ten of the twelve C_5's and D_5's reach 5 of the 6 axes --
    the axis each misses is the one its five-fold rotation fixes. The other
    two are seed-aligned and never leave v0's own axis, reaching 1. Neither
    is the ceiling: the five order-12 T-conjugates reach all 6 and realize

[ok] exhaustive over all 30 + 59 subgroups, hence over EVERY finite subgroup
     of SO(3) (a draw that realizes permutes the vertex set, so it lies
     inside the solid's rotation group): the twirl bar is T for all four,
     the realize bar climbs 3 -> 4 -> 12 -> 60, and they cross at the
     icosahedron -- the order-counting argumen

In [20]:
# === R2's bar: 2T is the UNIVERSAL minimal twirl (check_universal_twirl) ===

R_T = load_rotations("T")
kappa = np.trace(T_NOISE) / 3
for solid in SOLIDS:
    s = load_vertices(solid)
    M, o = channel_R2(s, R_T, T_NOISE, t_NOISE)
    d_M = np.abs(M - kappa * np.eye(3)).max()         # measured 7.8e-16
    assert d_M < 1e-10, \
        f"{solid}: T draw not depol at tr(T)/3 (max |dev| = {d_M:.2e})"
    assert np.allclose(o, 0, atol=1e-9)
print(f"[ok] R2 with g ~ Unif(T): exactly depolarizing at kappa = tr(T)/3 = {kappa:.6f}")
print("     for ALL FIVE solids, tetrahedron and dodecahedron included:")
print("     2T is the UNIVERSAL minimal twirl -- irreducibility is all R2 needs")

# ... and it is minimal in the strong sense: sweep each solid's own
# lattice and nothing below order 12 twirls anything. The bar is FLAT at
# T across all five, which is why the ledger's twirl row is constant --
# a theorem in disguise, since the finite subgroups of SO(3) are cyclic,
# dihedral, T, O, I and only the last three are irreducible.
print()
for solid in SOLIDS:
    R = load_rotations(COVARIANCE[solid])
    bar, ok = twirl_bar(solid)
    irr = {S for S in lattice(COVARIANCE[solid])
           if np.isclose(np.mean([np.trace(R[i]) ** 2 for i in S]), 1.0)}
    assert ok == irr and bar == 12, (solid, bar)
    print(f"  {solid:14s} R2 twirls for {_fmt_orders(by_order(R, ok))}"
          f" -- bar at {bar}")
print("[ok] the twirl bar is FLAT at T (order 12) for every solid, in both")
print("     protocols: nothing smaller is irreducible, so nothing smaller twirls")

[ok] R2 with g ~ Unif(T): exactly depolarizing at kappa = tr(T)/3 = 0.720000
     for ALL FIVE solids, tetrahedron and dodecahedron included:
     2T is the UNIVERSAL minimal twirl -- irreducibility is all R2 needs

  tetrahedron    R2 twirls for 12 (T) -- bar at 12
  octahedron     R2 twirls for 12 (T), 24 (O) -- bar at 12
  cube           R2 twirls for 12 (T), 24 (O) -- bar at 12
  icosahedron    R2 twirls for 12(x5 T), 60 (I) -- bar at 12
  dodecahedron   R2 twirls for 12(x5 T), 60 (I) -- bar at 12
[ok] the twirl bar is FLAT at T (order 12) for every solid, in both
     protocols: nothing smaller is irreducible, so nothing smaller twirls


### The witness: a coin that realizes and twirls nothing

Of the four coins only the octahedron's is closed under multiplication — the cyclic $C_3$ about
the body diagonal $(1,1,1)$, i.e. $\{\mathrm{Id}, F, F^\dagger\}$ as rotations, not just as
words. It realizes the octahedral POVM *exactly* and twirls *nothing*, and the failure has a
closed form for arbitrary noise: the coin preserves the entire readout row of $T$ and merely
cycles it (a circulant), where an irreducible group destroys everything in that row but its
diagonal entry. Schur's "irreducibly" made necessary on the reader's own object — and the
commutant dimensions $\frac1{|G|}\sum_g (\operatorname{tr}R_g)^2$ say why: three scalars to
calibrate for $C_3$, one for $T$, $O$, $I$.

Closure is an identity question, so the verdict is decided twice — on the rounding grid and by
canonical field equality — and the symbolic channel identity is bridged back to `channel_R1` on
every (subgroup, solid) pair of both lattices.

In [21]:
# === Coin machinery (lifted from core + twojobs) ===

def best_circuits(atlas):
    """Min-(magic, depth) dij circuit per SO(3) rotation, over the +-q pair.

    Each rotation appears twice in the binary group (as U and -U); the
    cheaper min-magic (Dijkstra) synthesis represents it. Returns
    {rot_key: (magic, depth, sequence)}.
    """
    best = {}
    for U, seq, depth, magic in zip(atlas["unitaries"], atlas["dij_sequences"],
                                    atlas["dij_depths"], atlas["dij_magic_costs"]):
        k = rot_key(rotation_from_unitary(U))
        cand = (int(magic), int(depth), str(seq))
        if k not in best or cand[:2] < best[k][:2]:
            best[k] = cand
    return best


def coset_representatives(s, R, circuits):
    """One cheapest atlas circuit per vertex axis of the solid.

    The g's whose snapshot hits a given axis {n, -n} (i.e. R_g^T v0 = +-n)
    form a coset of the stabilizer of v0; any representative realizes that
    axis, so the min-(magic, depth) one prices it. Asserts the cosets
    partition the group evenly (= vertex-transitivity). Returns one
    (magic, depth, sequence) triple per axis.
    """
    _, v = alignment(s)
    axes = []                       # one representative vertex per axis
    for n in s:
        if not any(np.allclose(n, -m, atol=1e-9) for m in axes):
            axes.append(n)
    reps, sizes = [], []
    for n in axes:
        members = [g for g, Rg in enumerate(R)
                   if np.allclose(Rg.T @ v, n, atol=1e-9)
                   or np.allclose(Rg.T @ v, -n, atol=1e-9)]
        assert members, "group not transitive on the vertex axes"
        sizes.append(len(members))
        reps.append(min(circuits[rot_key(R[g])] for g in members))
    assert len(set(sizes)) == 1 and sizes[0] * len(axes) == len(R), \
        "cosets do not partition the group evenly"
    return reps


def coin_rotations(solid):
    """The coin itself: the min-(magic, depth) representative per vertex axis.

    coset_representatives prices the axes; this returns the rotations behind
    those prices, so the coin can be asked whether it is a GROUP.
    """
    g = COVARIANCE[solid]
    R, s = load_rotations(g), load_vertices(solid)
    circuits = best_circuits(load_atlas(g))
    _, v = alignment(s)
    axes = []
    for n in s:
        if not any(np.allclose(n, -m, atol=1e-9) for m in axes):
            axes.append(n)
    out = []
    for n in axes:
        members = [i for i, Rg in enumerate(R)
                   if np.allclose(Rg.T @ v, n, atol=1e-9)
                   or np.allclose(Rg.T @ v, -n, atol=1e-9)]
        # same lexicographic min as coset_representatives, so this returns the
        # rotations behind exactly the words Table D.1 prints
        out.append(R[min(members, key=lambda i: circuits[rot_key(R[i])])])
    return np.array(out)


def exact_coin(solid):
    """coin_rotations(solid) over the solid's field, and its INDEX list.

    The witness printed in Appendix F.3.3.2 -- only the octahedron's coin is a
    group -- is a closure verdict, and closure is an identity question: it asks
    which element a product IS.  The float version answers it on rot_key's
    9-decimal grid.  Here the axes, the cosets and (in exact_coin_is_group) the
    closure itself are decided by field equality instead.

    What deliberately stays float is the min over circuit COSTS: that selects
    WHICH representative of a coset to take -- a choice, not an identity -- and
    it keys on rot_key exactly as the float coin does, so both return the same
    atlas index by construction.  The returned indices let the caller check the
    two coins agree element for element rather than merely both being coins.
    """
    g = COVARIANCE[solid]
    K = solid_field(solid)
    R = [to_field(M, K) for M in exact_rotations(g)]
    s = exact_vertices(solid, K)
    circuits, Rf = best_circuits(load_atlas(g)), load_rotations(g)
    _, v = exact_alignment(s, K)
    axes = []
    for n in s:
        if not any([-c for c in n] == m for m in axes):
            axes.append(n)
    idx = []
    for n in axes:
        neg = [-c for c in n]
        members = [i for i, Rg in enumerate(R) if _mv(_tr(Rg), v, K) in (n, neg)]
        assert members, (solid, "group not transitive on the vertex axes")
        idx.append(min(members, key=lambda i: circuits[rot_key(Rf[i])]))
    return [R[i] for i in idx], idx, K


def exact_coin_is_group(C, K):
    """Is the coin closed under multiplication?  Canonical keys, no grid.

    Field elements are canonical AND hashable, which is exactly what a set
    membership test needs and what simplify(a-b)==0 cannot supply.
    """
    keys = {tuple(x for row in M for x in row) for M in C}
    return all(tuple(x for row in _mm(A, B, K) for x in row) in keys
               for A in C for B in C)

In [22]:
# === The C_3 witness (check_coin_group) ===

# The witness: of the four coins only the OCTAHEDRON's is closed under
# multiplication, so it realizes the POVM exactly while twirling nothing
# -- Schur's "irreducibly" made necessary on the reader's own object.
#
# The verdict is decided TWICE: once on rot_key's rounding grid, once by
# canonical field equality (exact_coin / exact_coin_is_group), and the two
# are required to agree.  Closure asks which element a product IS, so it is
# an identity computation, so it is decided in canonical form as well;
# the float path stays because it is what the rest of this check keys on.
groups = {}
for solid in ("octahedron", "cube", "icosahedron", "dodecahedron"):
    C = coin_rotations(solid)
    keys = {rot_key(M) for M in C}
    closed = all(rot_key(A @ B) in keys for A in C for B in C)
    Ce, idx, K = exact_coin(solid)
    assert np.abs(np.array([[[float(K.to_sympy(e)) for e in row] for row in M]
                            for M in Ce]) - C).max() < 1e-12, solid
    assert [rot_key(load_rotations(COVARIANCE[solid])[i]) for i in idx] \
        == [rot_key(M) for M in C], solid    # the SAME coset representatives
    closed_exact = exact_coin_is_group(Ce, K)
    assert closed_exact == closed, solid
    hits = orbit_counts(load_vertices(solid), C)
    assert list(hits) == exact_orbit_counts(
        exact_vertices(solid, K), Ce, K), solid
    assert hits.min() == hits.max(), solid       # every coin realizes
    groups[solid] = closed
    print(f"  {solid:14s} {len(C):>2d}-word coin, realizes;"
          f" closed under multiplication: {'YES -- a group' if closed else 'no'}"
          f"  [exact: {'group' if closed_exact else 'not closed'}]")
assert groups == {"octahedron": True, "cube": False,
                  "icosahedron": False, "dodecahedron": False}

# dim End_G(R^3) = trace of the averaging projector X -> E_g R^T X R,
# which is (1/|G|) sum tr(R)^2: three scalars for C_3, one for T/O/I.
R_F = bloch_matrix(atlas_gates()["F"])
C3 = [sp.eye(3), R_F, R_F * R_F]
assert sp.expand(R_F**3 - sp.eye(3)) == sp.zeros(3, 3)
# ... and the octahedron's coin IS that C_3, as a set of matrices rather
# than as a set of words -- the identification the sentence above asserts.
Ke = solid_field("octahedron")
assert {tuple(x for row in M for x in row) for M in exact_coin("octahedron")[0]} \
    == {tuple(x for row in to_field(M, Ke) for x in row) for M in C3}
assert R_F * Matrix([1, 1, 1]) == Matrix([1, 1, 1])   # the (1,1,1) axis
print("  the octahedron's coin IS {I, R_F, R_F^2} as matrices, not just as words,")
print("  and R_F fixes (1,1,1) -- the 120-degree turn about the body diagonal")
dims = {}
for name, Rs in (("C_3", C3), ("T", exact_rotations("T")),
                 ("O", exact_rotations("O")), ("I", exact_rotations("I"))):
    dims[name] = sp.simplify(sum(sp.trace(M) ** 2 for M in Rs) / len(Rs))
assert dims == {"C_3": 3, "T": 1, "O": 1, "I": 1}, dims
print("\n  dim End_G(R^3) = (1/|G|) sum tr(R)^2:  "
      + ",  ".join(f"{k} -> {v}" for k, v in dims.items()))
print("  (the commutant of a reducible draw is 3-dimensional -- three scalars to")
print("   calibrate, not one -- while T, O and I each buy Schur's single scalar)")

# The channel identity, SYMBOLICALLY and for ARBITRARY noise: six free
# symbols, no probe matrix anywhere. Over an irreducible group Schur
# returns (v.w) Id_3 for any seed; over the coin the readout row of T
# survives entire and merely cycles.
v, w = Matrix(sp.symbols("v_1:4", real=True)), Matrix(sp.symbols("w_1:4", real=True))
for name, Rs in (("T", exact_rotations("T")), ("O", exact_rotations("O")),
                 ("I", exact_rotations("I"))):
    M, off = rank_one_twirl(Rs, v, w)
    assert sp.simplify(M - v.dot(w) * sp.eye(3)) == sp.zeros(3, 3), name
    assert sp.simplify(off) == sp.zeros(3, 1), name
print("\n  [exact, arbitrary v and w] over T, O, I:  M = (v.w) Id_3,  offset = 0")
print("     and v.w = (A^T zhat).(A^T T^T zhat) = zhat^T T^T zhat = T_zz, the")
print("     alignment cancelling because A is a rotation -- exact or not")

a, b, c, tz = sp.symbols("T_zx T_zy T_zz t_z", real=True)
M3, off3 = rank_one_twirl(C3, Matrix([0, 0, 1]), Matrix([a, b, c]))
M3, off3 = sp.simplify(M3), sp.simplify(tz * off3)
assert M3 == Matrix([[c, a, b], [b, c, a], [a, b, c]]), M3
assert off3 == tz * Matrix([1, 1, 1]), off3
print("\n  [exact, arbitrary noise] over the coin C_3 = <R_F>, seed v = zhat:")
print(f"     M = circ(T_zz, T_zx, T_zy) = {M3.tolist()}")
print(f"     offset = t_z (1,1,1) = {off3.T.tolist()}")
print("     -- C_3 preserves the entire readout row of T and merely cycles it,")
print("     where T destroys everything in that row but its diagonal entry.")
print("     Conjugation-averaging preserves the trace, so both channels have")
print("     trace 3 T_zz: the coin's diagonal is already right and only the")
print("     off-diagonal circulant survives. The offset is t_z times the axis")
print("     C_3 fixes, and dies for T because the solid is centered.")

# The bridge: that reduction IS channel_R1, on every subgroup of both
# lattices and all four solids -- so the symbolic claim above is a claim
# about the protocol, not about a formula resembling it.
worst = 0.0
for solid in ("octahedron", "cube", "icosahedron", "dodecahedron"):
    s = load_vertices(solid)
    R = load_rotations(COVARIANCE[solid])
    A, v0 = alignment(s)
    zhat = np.array([0.0, 0.0, 1.0])
    # both this and the T_zz check below, which consumes the very same
    # A^T zhat, measure 1.1e-16
    d_v0 = np.abs(A.T @ zhat - v0).max()
    assert d_v0 < 1e-12, f"{solid}: A^T zhat != v0 (max |dev| = {d_v0:.2e})"
    vv, ww = A.T @ zhat, A.T @ T_NOISE.T @ zhat
    d_zz = abs(vv @ ww - T_NOISE[2, 2])
    assert d_zz < 1e-12, f"{solid}: v.w != T_zz (|dev| = {d_zz:.2e})"
    for S in lattice(COVARIANCE[solid]):
        RS = R[sorted(S)]
        M, o = channel_R1(s, RS, T_NOISE, t_NOISE)
        M2, o2 = rank_one_twirl(RS, vv, ww)
        worst = max(worst, np.abs(M - M2).max(),
                    np.abs(o - t_NOISE[2] * o2).max())
assert worst < 1e-12, worst
print("\n  reduction verified against channel_R1 on all 178 (subgroup, solid)")
print(f"  pairs of both lattices: max |difference| = {worst:.1e}")

# ... and the numeric instance the module's own probe produces, kept as
# corroboration of the symbolic identity rather than as the claim.
C = coin_rotations("octahedron")
M, o = channel_R1(load_vertices("octahedron"), C, T_NOISE, t_NOISE)
row = T_NOISE[2]                      # (T_zx, T_zy, T_zz), cycled
d_M = np.abs(M - np.array([np.roll(row, k + 1) for k in range(3)])).max()
d_o = np.abs(o - t_NOISE[2] * np.ones(3)).max()
assert d_M < 1e-12 and d_o < 1e-12, (d_M, d_o)
print("\n  on the module's generic probe the coin reads")
print(f"     M = {np.round(M, 4).tolist()}, offset = {np.round(o, 4).tolist()}")
print("[ok] the octahedron's coin is a group and does exactly one of the two jobs;")
print("     it is the counterexample that makes Schur's hypothesis necessary")

  octahedron      3-word coin, realizes; closed under multiplication: YES -- a group  [exact: group]


  cube            4-word coin, realizes; closed under multiplication: no  [exact: not closed]


  icosahedron     6-word coin, realizes; closed under multiplication: no  [exact: not closed]


  dodecahedron   10-word coin, realizes; closed under multiplication: no  [exact: not closed]
  the octahedron's coin IS {I, R_F, R_F^2} as matrices, not just as words,
  and R_F fixes (1,1,1) -- the 120-degree turn about the body diagonal

  dim End_G(R^3) = (1/|G|) sum tr(R)^2:  C_3 -> 3,  T -> 1,  O -> 1,  I -> 1
  (the commutant of a reducible draw is 3-dimensional -- three scalars to
   calibrate, not one -- while T, O and I each buy Schur's single scalar)



  [exact, arbitrary v and w] over T, O, I:  M = (v.w) Id_3,  offset = 0
     and v.w = (A^T zhat).(A^T T^T zhat) = zhat^T T^T zhat = T_zz, the
     alignment cancelling because A is a rotation -- exact or not

  [exact, arbitrary noise] over the coin C_3 = <R_F>, seed v = zhat:
     M = circ(T_zz, T_zx, T_zy) = [[T_zz, T_zx, T_zy], [T_zy, T_zz, T_zx], [T_zx, T_zy, T_zz]]
     offset = t_z (1,1,1) = [[t_z, t_z, t_z]]
     -- C_3 preserves the entire readout row of T and merely cycles it,
     where T destroys everything in that row but its diagonal entry.
     Conjugation-averaging preserves the trace, so both channels have
     trace 3 T_zz: the coin's diagonal is already right and only the
     off-diagonal circulant survives. The offset is t_z times the axis
     C_3 fixes, and dies for T because the solid is centered.

  reduction verified against channel_R1 on all 178 (subgroup, solid)
  pairs of both lattices: max |difference| = 5.6e-16

  on the module's generic probe the coin r

### What the draw costs: atlas resources

The bars priced in atlas words. The $2T$ draw is free — all 24 elements at BFS depth $\le 2$,
magic $0$, and still free inside the bigger gate sets — and the coin is cheap everywhere. What
costs is the *draw*, which must clear both bars: since the twirl bar is $T$ for every solid and
$2T$ is all-Clifford, **every $\Phi$ in R1's ledger is charged to realization** — the
dodecahedron alone pays any, and the golden gate is strictly necessary in exactly one case.

In [23]:
# === Pricing the draw and the coin (check_atlas_resources) ===

# the 2T draw is free: 24 Clifford words of depth <= 2
atlas_T = load_atlas("T")
assert int(atlas_T["bfs_depths"].max()) == 2
assert int(atlas_T["dij_magic_costs"].max()) == 0
print("[ok] 2T: all 24 elements at BFS depth <= 2, magic 0 (gate set <X, Z, F>)")

# ... and stays free inside the bigger gate sets
keys_T = {rot_key(Rg) for Rg in load_rotations("T")}
for g in ("O", "I"):
    atlas = load_atlas(g)
    idx = [i for i, U in enumerate(atlas["unitaries"])
           if rot_key(rotation_from_unitary(U)) in keys_T]
    assert len(idx) == 24
    assert int(atlas["dij_magic_costs"][idx].max()) == 0
    assert int(atlas["dij_depths"][idx].max()) <= 2
    print(f"[ok] the 2T subgroup inside 2{g}: 24 elements, dij depth <= 2, magic 0"
          f" -- the twirl draw is Phi-free in the 2{g} gate set")

# the coin, priced per axis by its cheapest coset representative
claims = {"octahedron": (1, 0), "cube": (1, 0),
          "icosahedron": (2, 0), "dodecahedron": (2, 1)}
print(f"\n  {'solid':14s} {'axes':>4s} {'max depth':>9s} {'max Phi':>7s}  coin representatives (dij)")
print("  " + "-" * 86)
for solid, (d_max, m_max) in claims.items():
    s = load_vertices(solid)
    g = COVARIANCE[solid]
    reps = coset_representatives(s, load_rotations(g),
                                 best_circuits(load_atlas(g)))
    magics = [r[0] for r in reps]
    depths = [r[1] for r in reps]
    seqs = [r[2] if r[2] else "I" for r in reps]
    assert max(depths) <= d_max and max(magics) <= m_max, (solid, reps)
    if solid == "dodecahedron":
        # the per-axis minima pinned exactly: four axes cost a Phi even in
        # their cheapest coset representative, six are free -- mean 0.4
        # per shot, the floor check_flip_completion attains
        assert sorted(magics) == [0] * 6 + [1] * 4, magics
    print(f"  {solid:14s} {len(reps):>4d} {max(depths):>9d} {max(magics):>7d}"
          f"  {', '.join(seqs)}")
print("[ok] coin: octahedron/cube depth <= 1, icosahedron <= 2 (all 0 Phi);")
print("     dodecahedron <= 2 with <= 1 Phi -- the coin is cheap everywhere. What")
print("     costs is the DRAW, which must clear both bars (next check)")

# the dodecahedron's bill: the full 2I draw -- and it is charged to
# REALIZATION, not to the twirl
atlas_I = load_atlas("I")
magic = atlas_I["dij_magic_costs"]
assert int(atlas_I["dij_depths"].max()) <= 4 and int(magic.max()) <= 1
assert int(magic.sum()) == 96 and np.isclose(magic.mean(), 0.8)
print("[ok] full 2I draw (the dodecahedron's REALIZATION bill): dij depth <= 4 AND")
print(f"     <= 1 Phi simultaneously; mean {magic.mean():.1f} Phi ="
      f" {int(magic.sum())}/{len(magic)} elements")

# the magic economics, per solid: price the CHEAPEST draw that clears
# both bars. Where several subgroups tie at the bar (the icosahedron's
# five T-conjugates, only one of which is the atlas's own 2T) the
# protocol may pick any, so the bill is the minimum over them.
print()
for solid in ("octahedron", "cube", "icosahedron", "dodecahedron"):
    b = two_bars(solid)
    R = b["rotations"]
    circuits = best_circuits(load_atlas(b["group"]))
    both = b["draw_groups"]
    bill = min((max(circuits[rot_key(R[i])][0] for i in S),
                max(circuits[rot_key(R[i])][1] for i in S)) for S in both)
    assert bill == ((1, 4) if solid == "dodecahedron" else (0, 2)), (solid, bill)
    print(f"  {solid:14s} minimal draw = order {b['draw']}"
          f" ({'/'.join(b['min_twirl' if b['binds'] != 'realize' else 'min_realize'])}),"
          f" {len(both)} candidate(s); cheapest costs"
          f" {bill[0]} Phi at depth <= {bill[1]}")
print("[ok] every Phi in the randomized-projective ledger is charged to REALIZATION:")
print("     the twirl bar is T for all four solids and 2T is all-Clifford, so a draw")
print("     carries magic only where realization pushed it past T -- the dodecahedron")
print("     alone. The golden gate is strictly necessary in exactly one case.")

[ok] 2T: all 24 elements at BFS depth <= 2, magic 0 (gate set <X, Z, F>)
[ok] the 2T subgroup inside 2O: 24 elements, dij depth <= 2, magic 0 -- the twirl draw is Phi-free in the 2O gate set
[ok] the 2T subgroup inside 2I: 24 elements, dij depth <= 2, magic 0 -- the twirl draw is Phi-free in the 2I gate set

  solid          axes max depth max Phi  coin representatives (dij)
  --------------------------------------------------------------------------------------
  octahedron        3         1       0  F†, F, I
  cube              4         1       0  Z, F†, I, F
  icosahedron       6         2       0  F†, X F†, F, X F, I, X
  dodecahedron     10         2       1  Φ, F† Φ, Φ† F†, Φ†, I, Z, F, F X, F†, F† Z
[ok] coin: octahedron/cube depth <= 1, icosahedron <= 2 (all 0 Phi);
     dodecahedron <= 2 with <= 1 Phi -- the coin is cheap everywhere. What
     costs is the DRAW, which must clear both bars (next check)
[ok] full 2I draw (the dodecahedron's REALIZATION bill): dij depth <= 4 AN

*Novelty note.* The ingredients are classical — a 2-design twirl depolarizes (Schur), $2T$ is
the minimal group 2-design in $d = 2$ (Gross–Audenaert–Eisert 2007), and measurement twirling
is standard error-mitigation practice — but the realize/twirl decoupling for the Platonic
POVMs, the axis-correlated scalar $T_{zz}$ vs $\operatorname{tr}T/3$ distinction, and $2T$'s
universal-minimal-twirl status appear to be original to this thesis. The closest adjacent
work, Nguyen et al. (2022), uses Platonic transitivity for a different purpose (simplifying
the canonical dual), and models readout noise on the dilation ancillas rather than
group-averaging it.

## 3. The exactness obstruction — the reframing, third application

Finding 3. The maps a protocol averages must be *compiled*, and compilation carries the gate
set's arithmetic with it. Two independent obstructions, two different invariants:

- **Direction.** A rank-1 POVM realized by *any* protocol over gates with matrix entries in a
  conjugation-closed field $K$ — any ancillas, adaptivity, classical randomness — must have all
  its Bloch vertices in $K_\mathbb{R}^3$. The witness is the rotation-invariant
  $\det[v_a\,v_b\,v_c]$ of a spanning vertex triple, tested against
  $K_\mathbb{R} = \mathbb{Q}(\sqrt2,\sqrt5)$ — the real field of every gate set of the thesis:
  $2T$, Clifford ($2O$), Clifford${}+T$, $2I$, Clifford${}+\Phi$ and their unions all have
  entries in $K = \mathbb{Q}(\sqrt2,\sqrt5,i)$, and the entangling Clifford each is adjoined
  with is free, CNOT's entries being $0$ and $1$. Which
  triple is a choice, and the choice moves the number — sign always, magnitude and even the
  minimal polynomial on some solids — but never the $K_\mathbb{R}$-coset, so any one triple
  decides. (The triples are printed in the published vertex numbering, which is why
  `atlas_vertices` matters.)
- **Weight** (next cell). The direction lemma's escape hatch — a dilation — is closed by the
  ring: an effect's trace is a finite sum of gate-entry products, landing in
  $\mathcal{R}\cap\mathbb{Q} = \mathbb{Z}[1/2]$, while a transitive covariant POVM on $V$ outcomes
  needs $\operatorname{tr}E_k = 2/V$. So $V$ must be a power of two — and *deterministic* is
  three bans, not one: no coin, no discarded branch, a bounded number of rounds, each closing
  one classical way to buy the division the ring will not supply.

Between them the two convict all five solids; the octahedron fails weight alone, which is why
its exactness must be exhibited with a *coin* — randomness spent to buy $1/3$ as a probability,
never as an amplitude.

In [24]:
# === Field instruments (lifted from randomized_core.py) ===

def in_field(x, theta):
    """Is the algebraic number x an element of Q(theta)?

    field_isomorphism embeds Q(x) into Q(theta) iff x lies in Q(theta).
    """
    x = sp.simplify(x)
    if x.is_Rational:
        return True
    return sp.field_isomorphism(sp.AlgebraicNumber(x),
                                sp.AlgebraicNumber(theta)) is not None


def dyadic_order(x, kmax=8):
    """Smallest k with 2^k x an algebraic integer, or None if there is none.

    Every thesis gate has entries in calR = Z[tau, i, sqrt2, 1/2]: algebraic
    integers over a power of two (H and F carry a 1/sqrt2, Phi a 1/2, while
    T's exp(i pi/4) is an algebraic integer outright). calR is a ring, closed
    under conjugation, so a sum or product of gate entries never leaves it
    -- which is what pins a deterministic protocol's branch amplitudes, and
    with them the traces of the effects those branches realize. On the
    rationals R cuts down to Z[1/2], where the test is decidable outright:
    p/q in lowest terms qualifies iff q is a power of two.
    """
    x = sp.nsimplify(sp.simplify(x))
    if x.is_Rational:
        q = sp.denom(x)
        k = sp.multiplicity(2, q) if q % 2 == 0 else 0
        return k if q // 2**k == 1 else None
    y = sp.Symbol("_y")
    for k in range(kmax + 1):
        if sp.minimal_polynomial(2**k * x, y, polys=True).LC() == 1:
            return k
    return None


def det_witness(verts):
    """(1-based indices, det) of the first non-coplanar vertex triple.

    tab:povm-exactness prints both, so both must come from one computation --
    and from atlas_vertices(), since the caption sends the reader to the atlas
    to look the indices up.  The first three vertices span for three solids and
    are coplanar for the octahedron and the icosahedron, whence (1,3,5) and
    (1,2,5) there.
    """
    for trip in itertools.combinations(range(len(verts)), 3):
        d = sp.simplify(Matrix.hstack(*[verts[i] for i in trip]).det())
        if d != 0:
            return tuple(i + 1 for i in trip), sp.radsimp(d)
    raise AssertionError("no spanning triple")            # verts span R^3


def det_invariant(verts):
    """det[v_a v_b v_c] of the first non-coplanar vertex triple.

    A rotation R has det R = 1, so det[Rv_a Rv_b Rv_c] = det[v_a v_b v_c]: the
    value is orientation-free, and every vertex being K_R-rational forces
    it into K_R -- one det outside K_R obstructs the whole solid.

    Which triple is a choice, and it moves the number: the sign flips with the
    triple's handedness, and the magnitude moves too on the icosahedron (two
    values) and the dodecahedron (five, realizing FOUR minimal polynomials).
    What it cannot move is the coset -- check_obstruction() sweeps every
    spanning triple of every solid and finds all their determinants in one
    K_R-multiple class -- which is the computational face of Lemma 5's "same
    square class, so any one triple decides", and the reason a table may print
    a single row per solid at all.
    """
    return det_witness(verts)[1]


# === The direction obstruction (check_obstruction) ===

# ATLAS order, not symbolic_solids() order: the table this feeds prints the
# triple's indices and its caption sends the reader to tab:povm-atlas to
# look them up, so the two numberings' disagreement -- three solids, and
# atlas_vertices' docstring has the account -- has to be resolved HERE, in
# the reader's favour.  It reaches the printed determinant as a sign, and
# only on the tetrahedron and the icosahedron.
solids_sym = {s: atlas_vertices(s) for s in SOLIDS}
x = sp.Symbol("x")
KR = sqrt(2) + sqrt(5)           # primitive element of Q(sqrt2, sqrt5)
expected = {
    "tetrahedron": ((1, 2, 3), 27 * x**2 - 16, sqrt(3), "sqrt(3)"),
    "octahedron": ((1, 3, 5), x - 1, None, "-- (none)"),
    "cube": ((1, 2, 3), 27 * x**2 - 16, sqrt(3), "sqrt(3)"),
    "icosahedron": ((1, 2, 5), 125 * x**4 - 100 * x**2 + 16,
                    sqrt(5 + 2 * sqrt(5)), "sqrt(5+2 sqrt5)"),
    "dodecahedron": ((1, 2, 3), 27 * x**2 - 16, sqrt(3), "sqrt(3)"),
}
# The icosahedron carries two five-fold surds that look unrelated: the
# determinant's sqrt(10 - 2 sqrt5) and the demanded sqrt(5 + 2 sqrt5).  Both
# are the vertex normalizer sqrt(2 + tau) scaled by a unit -- up by tau, down
# by tau -- so all three name one extension of Q(sqrt5), and 2 + tau is
# Lemma 6's second named number.  That is how the lemma convicts the Demands
# column, whose surd it never mentions.  The determinant's own square is the
# lemma's first named number.  tab:povm-exactness prints both identities in a
# midrule sandwich, so both are asserted here before anything is written.
assert sp.simplify(sqrt(5 + 2 * sqrt(5)) - TAU_SYM * sqrt(2 + TAU_SYM)) == 0
assert sp.simplify(sqrt(10 - 2 * sqrt(5))
                   - (2 / TAU_SYM) * sqrt(2 + TAU_SYM)) == 0
assert sp.simplify(det_invariant(solids_sym["icosahedron"])**2
                   - sp.Rational(2, 25) * (5 - sqrt(5))) == 0
print(f"  {'solid':14s} {'(a,b,c)':>9s} {'det[v_a v_b v_c]':>16s}  {'min poly':26s} {'in Q(sqrt2,sqrt5)?':>18s}  demands")
print("  " + "-" * 96)
for solid in SOLIDS:
    trip, d = det_witness(solids_sym[solid])
    mp = sp.minimal_polynomial(d, x)
    exact = in_field(d, KR)
    trip_exp, poly_exp, ext, demand = expected[solid]
    assert trip == trip_exp, (solid, trip)
    assert sp.expand(mp - poly_exp) == 0, (solid, mp)
    assert exact == (solid == "octahedron"), solid
    if ext is not None:          # the demanded extension admits it
        assert in_field(d, sqrt(2) + sqrt(5) + ext), solid
    print(f"  {solid:14s} {str(trip):>9s} {float(d):>+16.6f}  {str(mp):26s} "
          f"{str(exact):>18s}  {demand}")
# Which triple is a choice, and the choice moves the number -- so the table
# printing ONE row per solid needs the choice not to move the VERDICT.
# Lemma 5 says so via square classes; the sweep says so directly, and says
# more: every spanning triple's determinant is a K_R-multiple of the one
# printed, one coset per solid.  (The Lean formalization needs the same
# shape: a lemma stated at the bare surd sqrt3 applies to no solid at all,
# and only its coset form reaches them.)  Cheap at 5s: the
# dodecahedron's 960 spanning triples realize ten determinants, five
# magnitudes and FOUR minimal polynomials, and all ten are K_R x sqrt3.
sweep = {                        # spanning triples, dets, |dets|, min polys
    "tetrahedron": (4, 2, 1, 1),
    "octahedron": (8, 2, 1, 1),
    "cube": (32, 2, 1, 1),
    "icosahedron": (160, 4, 2, 1),
    "dodecahedron": (960, 10, 5, 4),
}
print()
print(f"  {'solid':14s} {'spanning':>9s} {'dets':>5s} {'|dets|':>7s} {'min polys':>10s}"
      f"  one K_R-coset?")
print("  " + "-" * 65)
for solid in SOLIDS:
    verts, ref = solids_sym[solid], det_invariant(solids_sym[solid])
    dets, n = set(), 0
    for trip in itertools.combinations(range(len(verts)), 3):
        d = sp.simplify(Matrix.hstack(*[verts[i] for i in trip]).det())
        if d != 0:
            dets.add(sp.radsimp(d))
            n += 1
    coset = all(in_field(sp.radsimp(d / ref), KR) for d in dets)
    mags = {sp.radsimp(sp.Abs(d)) for d in dets}
    polys = {sp.minimal_polynomial(m, x) for m in mags}
    assert coset, solid
    assert (n, len(dets), len(mags), len(polys)) == sweep[solid], (
        solid, n, len(dets), len(mags), len(polys))
    print(f"  {solid:14s} {n:>9d} {len(dets):>5d} {len(mags):>7d} {len(polys):>10d}"
          f"  {str(coset):>14s}")
print("[ok] one coset per solid: the triple moves the determinant -- sign always,")
print("     magnitude on I and D, and on D the MINIMAL POLYNOMIAL too (four of them)")
print("     -- but never the coset, so any one triple decides.  That is Lemma 5's")
print("     square class, and it is what lets tab:povm-exactness print one row each")
print()
print("[ok] only the octahedral POVM is exact over K_R = Q(sqrt2, sqrt5) -- the real")
print("     field of every thesis gate set; the rest demand the listed extensions")
print("     (Decker's nested radicals in R2; the alignment A in R1 -- same magic,")
print("     two hiding places).  The icosahedron's two surds are one extension under")
print("     three names -- sqrt(5+2 sqrt5) = tau sqrt(2+tau) and sqrt(10-2 sqrt5) =")
print("     (2/tau) sqrt(2+tau), the vertex normalizer scaled up and down by a unit --")
print("     so Lemma 6 bars them by its second named number, the determinant by its first")

  solid            (a,b,c) det[v_a v_b v_c]  min poly                   in Q(sqrt2,sqrt5)?  demands
  ------------------------------------------------------------------------------------------------
  tetrahedron    (1, 2, 3)        -0.769800  27*x**2 - 16                            False  sqrt(3)
  octahedron     (1, 3, 5)        +1.000000  x - 1                                    True  -- (none)
  cube           (1, 2, 3)        -0.769800  27*x**2 - 16                            False  sqrt(3)


  icosahedron    (1, 2, 5)        +0.470228  125*x**4 - 100*x**2 + 16                False  sqrt(5+2 sqrt5)
  dodecahedron   (1, 2, 3)        -0.769800  27*x**2 - 16                            False  sqrt(3)

  solid           spanning  dets  |dets|  min polys  one K_R-coset?
  -----------------------------------------------------------------
  tetrahedron            4     2       1          1            True
  octahedron             8     2       1          1            True
  cube                  32     2       1          1            True


  icosahedron          160     4       2          1            True


  dodecahedron         960    10       5          4            True
[ok] one coset per solid: the triple moves the determinant -- sign always,
     magnitude on I and D, and on D the MINIMAL POLYNOMIAL too (four of them)
     -- but never the coset, so any one triple decides.  That is Lemma 5's
     square class, and it is what lets tab:povm-exactness print one row each

[ok] only the octahedral POVM is exact over K_R = Q(sqrt2, sqrt5) -- the real
     field of every thesis gate set; the rest demand the listed extensions
     (Decker's nested radicals in R2; the alignment A in R1 -- same magic,
     two hiding places).  The icosahedron's two surds are one extension under
     three names -- sqrt(5+2 sqrt5) = tau sqrt(2+tau) and sqrt(10-2 sqrt5) =
     (2/tau) sqrt(2+tau), the vertex normalizer scaled up and down by a unit --
     so Lemma 6 bars them by its second named number, the determinant by its first


### The weight half

Every thesis gate — Cliffords, $T$, $\Phi$ and its conjugate partner, CNOT — has entries in
$\mathcal{R} = \mathbb{Z}[\tau, i, \sqrt2, \tfrac12]$: algebraic integers over a power of two.
$\mathcal{R}$ is a ring closed under conjugation, so no branch amplitude of any deterministic
protocol ever leaves it. The table then convicts by weight where direction is silent and vice
versa — no solid clears both.

In [25]:
# === The weight obstruction (check_weight_obstruction) ===

# The companion to check_obstruction, and the reason the octahedron's
# exactness has to be exhibited with a COIN. That lemma constrains an
# effect's direction (its Bloch vertex); this one constrains its weight.
#
# Fix a protocol whose only randomness is quantum -- unitaries from the
# gate set, ancillas in |0>, computational-basis readout, feedforward
# allowed, but no classical coin, no discarded branch, and a bounded
# number of rounds. Each branch acts on the data qubit as the row
# vector <y|U(. (x) |0...0>), whose entries are sums of products of
# gate entries, hence lie in calR (see dyadic_order). An outcome class
# realizes sum_i |a_i><a_i| over FINITELY many branches, rank 1 only if
# all the |a_i> are parallel, and its trace sum_i ||a_i||^2 is then a
# finite sum, lying in calR n R = the reals of calR. A transitive covariant
# POVM on V outcomes has tr E_k = 2/V, a rational, and calR n Q = Z[1/2]
# -- so V must be a power of two.
#
# Unlike the vertex test this one needs no rotation invariance: weights
# are orientation-blind to begin with, so a cleverer pose cannot save a
# solid, Decker's or ours. What CAN save one is a classical operation on
# the weight, and there are exactly three -- the three bans above. A
# coin buys the octahedron's 1/3 as a bias, which is what the coin of
# check_coset_coin spends; a discard buys it as 1/4 over 3/4; an
# unbounded retry buys it as the series sum_{n>=0}(1/4)^(n+1). All
# three deliver a PROBABILITY, never an amplitude, and
# weight_obstruction_escapes.py exhibits the latter two.
H = (1 / sqrt(2)) * Matrix([[1, 1], [1, -1]])
S = Matrix([[1, 0], [0, sI]])
gates = dict(atlas_gates())                      # X, Z, F, Phi
gates.update({
    "H": H, "S": S,
    "T": Matrix([[1, 0], [0, (1 + sI) / sqrt(2)]]),
    "Phi*": Rational(1, 2) * Matrix([[-SIG_SYM - sI * TAU_SYM, 1],
                                     [-1, -SIG_SYM + sI * TAU_SYM]]),
    "CNOT": Matrix(4, 4, lambda i, j: int((i, j) in
                                          ((0, 0), (1, 1), (2, 3), (3, 2)))),
})
for name, U in sorted(gates.items()):
    assert sp.simplify(U * U.conjugate().T - sp.eye(U.rows)) == sp.zeros(U.rows)
    orders = [dyadic_order(e) for e in U]
    assert all(k is not None for k in orders), name
    print(f"  {name:5s} entries in calR at 2^-{max(orders)}")
# closure is a ring axiom, but spot-check it on every two-letter word
for A in gates.values():
    for B in gates.values():
        if A.rows == B.rows:
            assert all(dyadic_order(e) is not None for e in A * B)
print("  every two-letter word stays in calR (closure under x, as a ring must)")
print()
KR = sqrt(2) + sqrt(5)
# either numbering will do here, unlike check_obstruction(): weights are
# order-free, and the direction column is a membership BOOLEAN, which the
# coset sweep there shows the triple cannot move (one coset per solid, so
# either all its determinants are in K_R or none are).  Nothing is printed
# with a vertex index, so nothing has to be the reader's.
solids_sym = symbolic_solids()
weight_ok, direction_ok = set(), set()
print(f"  {'solid':14s} {'V':>3s}  {'tr E_k':>7s}  {'weight in Z[1/2]?':>18s}"
      f"  {'vertices in K_R^3?':>19s}")
print("  " + "-" * 70)
for solid in SOLIDS:
    verts = solids_sym[solid]
    V = len(verts)
    w = Rational(2, V)
    effects = [w * state_from_bloch(v) for v in verts]
    assert sp.simplify(sum(effects, sp.zeros(2, 2)) - sp.eye(2)) == sp.zeros(2, 2)
    assert all(sp.simplify(sp.trace(E) - w) == 0 for E in effects)
    if dyadic_order(w) is not None:
        weight_ok.add(solid)
    if in_field(det_invariant(verts), KR):
        direction_ok.add(solid)
    print(f"  {solid:14s} {V:3d}  {str(w):>7s}  {str(solid in weight_ok):>18s}"
          f"  {str(solid in direction_ok):>19s}")
assert weight_ok == {"tetrahedron", "cube"}, weight_ok
assert direction_ok == {"octahedron"}, direction_ok
assert not (weight_ok & direction_ok)            # no solid clears both
assert dyadic_order(Rational(1, 3)) is None
print("[ok] two independent obstructions, and between them they convict all five:")
print("     the tetrahedron and cube fail on direction (sqrt3), the icosahedron and")
print("     dodecahedron on both, the octahedron on weight alone -- 2/V is in Z[1/2]")
print("     only for V a power of two. So NO deterministic dilation over any thesis")
print("     gate set realizes any Platonic solid POVM exactly, in any orientation --")
print("     deterministic meaning no coin, no discarded branch, and a bounded number")
print("     of rounds. The octahedron's 1/3 is bought classically, through one of the")
print("     three: a coin's bias, a 1/4-over-3/4 discard, or an unbounded retry's")
print("     series (weight_obstruction_escapes.py exhibits the last two)")

  CNOT  entries in calR at 2^-0
  F     entries in calR at 2^-1
  H     entries in calR at 2^-1
  Phi   entries in calR at 2^-1
  Phi*  entries in calR at 2^-1


  S     entries in calR at 2^-0
  T     entries in calR at 2^-0
  X     entries in calR at 2^-0
  Z     entries in calR at 2^-0


  every two-letter word stays in calR (closure under x, as a ring must)

  solid            V   tr E_k   weight in Z[1/2]?   vertices in K_R^3?
  ----------------------------------------------------------------------
  tetrahedron      4      1/2                True                False
  octahedron       6      1/3               False                 True


  cube             8      1/4                True                False


  icosahedron     12      1/6               False                False


  dodecahedron    20     1/10               False                False
[ok] two independent obstructions, and between them they convict all five:
     the tetrahedron and cube fail on direction (sqrt3), the icosahedron and
     dodecahedron on both, the octahedron on weight alone -- 2/V is in Z[1/2]
     only for V a power of two. So NO deterministic dilation over any thesis
     gate set realizes any Platonic solid POVM exactly, in any orientation --
     deterministic meaning no coin, no discarded branch, and a bounded number
     of rounds. The octahedron's 1/3 is bought classically, through one of the
     three: a coin's bias, a 1/4-over-3/4 discard, or an unbounded retry's
     series (weight_obstruction_escapes.py exhibits the last two)


### Three corollaries

- **No exact alignment, for any vertex.** The direction lemma is total vertex by vertex: a
  $K_\mathbb{R}$-rational rotation taking *any* vertex to $\pm\hat z$ would put that vertex in
  $K_\mathbb{R}^3$ — so R1's alignment is inexact for every vertex choice, not just the
  conventional $v_0$.
- **Each inexact solid sits on the axes of a magic gate.** $X/Z$ land on the octahedron, $F$ on
  tetrahedron, cube and dodecahedron, and $\Phi$'s rotation axis *is* an icosahedron vertex in
  the published pose — and the eigenstates of $F$ and $\Phi$ are precisely their gate sets'
  magic states, so the field extension a solid demands is the magic of the gate it sits on:
  *the measurement inherits the magic*, and in atlas orientation a Platonic POVM is exactly
  implementable iff its vertices are the Pauli axes.
- **The octahedron has nothing to hide — and the protocols still do not merge there.**
  $A = \mathrm{Id}$, so the projective route on the octahedron *is* random Pauli measurement,
  and the coin realization proves no radicals are *forced* on any route — Decker's octahedral
  dilation still writes $\sqrt{(3\pm\sqrt3)/18}$ into its $U_A$, but that is the
  construction's choice, not the geometry's demand. What vanishes at the octahedron is the
  obstruction, not the distinction: twirled-native keeps its dilation, ancillas and all, and
  still reads $\operatorname{tr}T/3$ where the projective route reads $T_{zz}$.

In [26]:
# === No exact alignment anywhere (check_no_exact_alignment) ===

# finding 3, sharpened to the alignment: if a rotation W with entries
# in K_R took ANY vertex v to +-zhat, then v = +-W^T zhat -- a row of
# W -- would lie in K_R^3. So a vertex with a coordinate outside K_R
# admits no K_R-rational alignment, hence (by finding 3) no exact
# circuit over any thesis gate set; checking every vertex closes the
# loophole of aligning a cleverer vertex than our v0
KR = sqrt(2) + sqrt(5)
solids_sym = symbolic_solids()
cache = {}

def member(c):
    c = sp.simplify(c)
    if c not in cache:
        cache[c] = in_field(c, KR)
    return cache[c]

for solid in SOLIDS:
    if solid == "octahedron":
        continue
    verts = solids_sym[solid]
    assert all(not all(member(c) for c in v) for v in verts), solid
    print(f"  {solid:14s} every one of its {len(verts)} vertices has a coordinate outside K_R")
print("[ok] no vertex of an inexact solid lies in K_R^3, so no K_R-rational rotation")
print("     -- hence no exact circuit over any thesis gate set -- aligns any vertex")
print("     to zhat: R1's alignment is inexact for every vertex choice, not just v0")

# === The gate axes (check_gate_axes) ===

solids_sym = symbolic_solids()
landing = {"X": {"octahedron"}, "Z": {"octahedron"},
           "F": {"tetrahedron", "cube", "dodecahedron"},
           "Phi": {"icosahedron"}}
for name, U in atlas_gates().items():
    n = bloch_axis(U)
    hits = {s for s in SOLIDS
            if on_solid(n, solids_sym[s]) or on_solid(-n, solids_sym[s])}
    assert hits == landing[name], (name, hits)
    print(f"  {name:4s} axis = {tuple(sp.nsimplify(c) for c in n)}   lies on: {sorted(hits)}")
print("[ok] X/Z -> octahedron, F -> tetrahedron+cube+dodecahedron, Phi -> icosahedron:")
print("     each inexact solid sits on the eigen-axes of a magic gate and inherits its magic;")
print("     Phi's rotation axis IS an icosahedron vertex, in the atlas orientation")

# === The octahedron IS the Pauli bases (check_octahedron_exact) ===

verts = symbolic_solids()["octahedron"]
E = [Rational(1, 3) * state_from_bloch(v) for v in verts]
assert sp.simplify(sum(E, sp.zeros(2, 2)) - sp.eye(2)) == sp.zeros(2, 2)
axes = {(1, 0, 0), (-1, 0, 0), (0, 1, 0), (0, -1, 0), (0, 0, 1), (0, 0, -1)}
assert {tuple(int(c) for c in v) for v in verts} == axes
print("[ok] the octahedron IS the three Pauli bases (sum E_k = I, vertices = +-axes):")
print("     its PROJECTIVE route is literally randomized-Pauli measurement (A = Id),")
print("     and what vanishes here is the obstruction, not the distinction --")
print("     twirled-native keeps its dilation, ancillas and all, and still reads")
print("     tr T/3 where the projective route reads T_zz")

  tetrahedron    every one of its 4 vertices has a coordinate outside K_R
  cube           every one of its 8 vertices has a coordinate outside K_R


  icosahedron    every one of its 12 vertices has a coordinate outside K_R


  dodecahedron   every one of its 20 vertices has a coordinate outside K_R
[ok] no vertex of an inexact solid lies in K_R^3, so no K_R-rational rotation
     -- hence no exact circuit over any thesis gate set -- aligns any vertex
     to zhat: R1's alignment is inexact for every vertex choice, not just v0


  X    axis = (1, 0, 0)   lies on: ['octahedron']


  Z    axis = (0, 0, 1)   lies on: ['octahedron']


  F    axis = (sqrt(3)/3, sqrt(3)/3, sqrt(3)/3)   lies on: ['cube', 'dodecahedron', 'tetrahedron']


  Phi  axis = (0, -sqrt(sqrt(5)/10 + 1/2), -sqrt(1/2 - sqrt(5)/10))   lies on: ['icosahedron']
[ok] X/Z -> octahedron, F -> tetrahedron+cube+dodecahedron, Phi -> icosahedron:
     each inexact solid sits on the eigen-axes of a magic gate and inherits its magic;
     Phi's rotation axis IS an icosahedron vertex, in the atlas orientation
[ok] the octahedron IS the three Pauli bases (sum E_k = I, vertices = +-axes):
     its PROJECTIVE route is literally randomized-Pauli measurement (A = Id),
     and what vanishes here is the obstruction, not the distinction --
     twirled-native keeps its dilation, ancillas and all, and still reads
     tr T/3 where the projective route reads T_zz


### The reorientation: Decker's pose against ours

Decker's circuits park each solid's cyclic symmetry axis on $\hat z$ (the size of his Fourier
block is that axis's order); the published pose parks the Pauli axes there. The two differ by a
fixed reorientation per solid — one representative of a coset of the rotation group — and the
question is whether that correction comes free. It does not, by a third field smaller than
$K_\mathbb{R}$: a gate's Bloch action is quadratic in its entries, so the $1/\sqrt2$ of $H$ and
$F$ pairs off and every rotation an *atlas-generated* gate set realizes has $SO(3)$ entries in
$\mathbb{Q}(\sqrt5)$ — *atlas-generated* meaning every **single-qubit** gate an atlas word up
to phase: $2T$, Clifford, $2I$, Clifford$+\Phi$ and their unions, i.e. every thesis gate set
with no $T$ in it. (The scope matters: CNOT is in every thesis gate set and is no atlas word.)
All five reorientations leave that field, so none is an atlas word.
Adjoining $T$ moves the line exactly twice — $\mathrm{Bloch}(T) = R_z(45^\circ)$, the tetrahedron's
and cube's correction, the $T^\dagger$ the circuit figures draw — while the other three stay
barred over every thesis gate set. And the gap between the fields is a trap worth naming:
$K_\mathbb{R}$-exactness is *necessary, never sufficient* — $T$ clears §3's field and ring
tests side by side and is still no atlas word, which is exactly what the glue cell springs.

And yet **the pose costs the R2 estimator nothing**, and that is a theorem rather than two
poses that happened to agree: running `exact_channel_R2` verbatim over the polynomial ring
$\mathbb{Q}(\sqrt5)[C, T, t]$,

$$M(sC) = \frac{\operatorname{tr}(C^\top C\,T)}{3}\,\mathrm{Id}_3, \qquad \mathrm{off}(sC) = 0$$

identically in an *arbitrary* $3\times3$ matrix $C$ and the noise — the measured vertices reach
the channel only through $\sum_k \hat n_k\hat n_k^\top$ and $\sum_k \hat n_k$, and a pose moves
neither. The proviso: the pose costs nothing only *provided the belief moves with the gate*. Each coset
member $m = hR_0$ induces its own labelling of the unreoriented run, priced by the two-list
channel at exactly $\kappa = \operatorname{tr}(mT)/3$ — a family centred on zero, with RMS
$1/3$ noiselessly (the $\operatorname{tr}(m)/3$ law; at the probe the noisy second moment is
$|T|_F^2/27$) — so no single member is ever "the" per-solid figure.

This cell distills each half once (the full `check_reorientation_obstruction` — all five
solids, both noise settings, the tail shift — runs inside the final cell's `main()`).

In [27]:
# === Reorientations + the pose theorem (lifted from core + field) ===

def _Rz(c, s): return Matrix([[c, -s, 0], [s, c, 0], [0, 0, 1]])


def _Ry(c, s): return Matrix([[c, 0, s], [0, 1, 0], [-s, 0, c]])


# One reorientation per solid, carrying DECKER's pose onto the atlas's: the
# (Fourier block size, R, field demand) of Table D.3. R is one representative
# of a coset of the solid's rotation group, so there are |G| of them -- 12, 24,
# 24, 60, 60. Shared by check_reorientation_obstruction (which prices R) and
# check_decker_outcome_order (which derives Decker's pose from his circuits and
# checks these R against it); defined once so the two cannot drift apart.
# The REPRESENTATIVE is load-bearing, not just the coset: tab:decker-labels
# prints each R's induced outcome permutation and the circuit figures draw
# U_R = R^-1, so every entry below must stay the literal factor product
# Table D.3 prints -- a coset-mate passes every set-level check here while
# owing a different permutation.
# (The block sits directly above REORIENT so lift_assign("REORIENT") carries
# it into the walkthrough by its own comment-walking rule, not as a lift
# neighbor's lexical baggage.)
REORIENT = {
    "tetrahedron": (2, _Rz(sqrt(2) / 2, sqrt(2) / 2), "sqrt2"),
    "octahedron": (3, _Rz(sqrt(2) / 2, sqrt(2) / 2)
                   * _Ry(sqrt(3) / 3, sqrt(6) / 3) * _Rz(-1, 0), "sqrt3"),
    "cube": (4, _Rz(sqrt(2) / 2, sqrt(2) / 2), "sqrt2"),
    "icosahedron": (3, _Ry(1 / (TAU_SYM * sqrt(3)), -TAU_SYM / sqrt(3)),
                    "sqrt3"),
    "dodecahedron": (5, _Ry(TAU_SYM / sqrt(TAU_SYM + 2),
                            1 / sqrt(TAU_SYM + 2)), "tau/sqrt(tau+2)"),
}


# Radial denominators -- the same table as povm_properties's _RADIAL: every
# solid's vertices are an integer-ish tuple over Q(sqrt5) over a single radius.
# Only exact_reposed_twirl_R2 uses them, and it is quadratic in the vertices,
# so clearing the radius costs it nothing and drops all five solids out of
# their own degree-4 fields into one degree-2 one.  Measured over the five:
# 60 s -> 22 s.
_RADIAL = {"tetrahedron": sqrt(3), "octahedron": Rational(1), "cube": sqrt(3),
           "icosahedron": sqrt(2 + TAU_SYM), "dodecahedron": sqrt(3)}

_POSE_FIELD = sp.QQ.algebraic_field(sqrt(5))

_POSE_RING = {}


def _pose_ring():
    """Q(sqrt5)[B, T, t]: 21 free symbols -- a 3x3 re-pose and a generic noise.

    Memoized like exact_lattice(): one ring serves all five solids, and
    building it is the sweep's only fixed cost.
    """
    if not _POSE_RING:
        names = ([f"B{i}{j}" for i in range(3) for j in range(3)]
                 + [f"T{i}{j}" for i in range(3) for j in range(3)]
                 + [f"t{k}" for k in range(3)])
        P = _POSE_FIELD.poly_ring(*names)
        g = [P.convert(sp.Symbol(n)) for n in names]
        _POSE_RING.update(
            P=P,
            B=[[g[3 * i + j] for j in range(3)] for i in range(3)],
            T=[[g[9 + 3 * i + j] for j in range(3)] for i in range(3)],
            t=g[18:])
    return _POSE_RING["P"], _POSE_RING["B"], _POSE_RING["T"], _POSE_RING["t"]


def exact_reposed_twirl_R2(solid, draw=None):
    """Is R2's scalar (tr T/3) at EVERY pose of the measured POVM, or only at
    the one pose check_reorientation_obstruction happens to try?

    That check asserts the twirled-native channel is (tr T/3) Id_3 with zero
    offset once the measured POVM is carried to Decker's pose.  It is a
    corollary; this is the theorem it is a corollary of.  For an ARBITRARY 3x3
    matrix C -- orthogonality is never used, so there is no parametrisation of
    SO(3) to get right and no corner of it to miss --

        M(sC) = (tr(C^T C T) / 3) Id_3,        off(sC) = 0,

    identically in C and in the noise (T, t).  A pose is a rotation, C^T C is
    then Id_3, and the theorem collapses to exactly what that assertion probes.

    Why the measured vertices cannot reach the scalar: they enter only through
    sum_k n_k n_k^T, which is (V/3) Id_3 for every atlas solid, and
    C^T ((V/3) Id_3) C = (V/3) C^T C; the offset needs sum_k n_k = 0 as well.
    Both moments are pose-covariant -- which is the file's own reason in code,
    that R2's twirl needs the DRAWN group irreducible rather than covariant for
    the measured POVM, so re-posing the POVM cannot touch it.

    No new transcription: exact_channel_R2 is run VERBATIM, its coefficient
    ring K taken to be a polynomial ring instead of a number field.  The loop
    check_exact_scalars already guards value-for-value against channel_R2 is
    the loop that proves this, and it is generic in K because every mechanism
    it uses -- K.zero, K.one, K.convert, exact division -- is.

    `draw` overrides the drawn group, which is the only hypothesis there is:
    the theorem needs it irreducible, so a reducible draw must fail at every
    pose exactly as it fails at one.

    Returns (verdict, evaluate).  evaluate(C, T, t) is the symbolic channel at
    float arguments: the verdict is a BOOLEAN, and a boolean exact check does
    not test its own transcription, so the caller pairs it
    with a value-for-value comparison against channel_R2.
    """
    P, B, T, t = _pose_ring()
    K, lam = _POSE_FIELD, _RADIAL[solid]
    # Clearing the radius rather than the compositum.  The free
    # matrix B then plays the part of C/lam, so lam itself never has to be
    # representable in Q(sqrt5) -- lam^2 does, and it enters in exactly one
    # place, next to the second moment it belongs to.
    s = [[P.convert(K.from_sympy(sp.expand(lam * c)), K) for c in v]
         for v in atlas_vertices(solid)]
    R = [[[P.convert(K.from_sympy(M[i, j]), K) for j in range(3)] for i in range(3)]
         for M in (exact_rotations(COVARIANCE[solid]) if draw is None else draw)]
    sB = [[sum((B[a][i] * n[a] for a in range(3)), P.zero) for i in range(3)]
          for n in s]
    M, off = exact_channel_R2(sB, R, T, t, P)          # the SAME loop, over K[.]

    lam2 = P.convert(K.from_sympy(sp.expand(lam ** 2)), K)
    A = [[lam2 * sum((B[k][i] * B[k][j] for k in range(3)), P.zero)
          for j in range(3)] for i in range(3)]                    # C^T C
    want = sum((sum((A[i][k] * T[k][i] for k in range(3)), P.zero)
                for i in range(3)), P.zero) / P.convert(3)         # tr(C^T C T)/3
    ok = (all(M[i][j] == (want if i == j else P.zero)
              for i in range(3) for j in range(3))
          and all(o == P.zero for o in off))

    def evaluate(C, Tn, tn):
        """The (M, off) this identity predicts, at float (C, T, t)."""
        vals = ([c / float(lam) for c in np.asarray(C, dtype=float).ravel()]
                + list(np.asarray(Tn, dtype=float).ravel())
                + list(np.asarray(tn, dtype=float)))

        def at(p):
            return sum(float(K.to_sympy(c))
                       * math.prod(vals[k] ** e for k, e in enumerate(mono))
                       for mono, c in p.terms())

        return (np.array([[at(M[i][j]) for j in range(3)] for i in range(3)]),
                np.array([at(o) for o in off]))

    return ok, evaluate

In [28]:
# === Distilled: the field lemma, the pose theorem, the coset laws (glue) ===

Q5 = sqrt(5)

# (a) The field lemma. Every Clifford acts as a signed permutation, Phi as a
# golden turn -- all inside Q(sqrt5), a field, hence closed under words. The
# Clifford quantifier needs Clifford GENERATORS, so H and S join the atlas
# four (as in the module's own check), and the pair closure below is the
# receipt that <Bloch(H), Bloch(S)> is the full order-24 Clifford rotation
# group -- not the order-12 group the atlas four alone would generate.
gates = dict(atlas_gates())                      # X, Z, F, Phi
gates.update({"H": (1 / sqrt(2)) * Matrix([[1, 1], [1, -1]]),
              "S": Matrix([[1, 0], [0, sI]])})
bloch = {}
for name, U in sorted(gates.items()):
    Rb = bloch[name] = bloch_matrix(U)
    assert all(in_field(e, Q5) for e in Rb), name
    kind = ("signed permutation" if all(e.is_Integer for e in Rb)
            else "golden half-integers")
    print(f"  {name:4s} Bloch matrix: {kind:22s} -- in Q(sqrt5)")
G24 = {sp.ImmutableMatrix(bloch["H"]), sp.ImmutableMatrix(bloch["S"])}
while True:
    grown = G24 | {sp.ImmutableMatrix(a * b) for a in G24 for b in G24}
    if len(grown) == len(G24):
        break
    G24 = grown
assert len(G24) == 24
assert all(all(e.is_Integer for e in Mb) for Mb in G24)
print("  closure: <Bloch(H), Bloch(S)> = 24 signed permutations -- every"
      " Clifford")
# ... and the trap: T's entries pass section 3's field and ring tests, yet
# Bloch(T) = Rz(45 deg) needs 1/sqrt2 -- field-exact is not atlas.
T_GATE = Matrix([[1, 0], [0, (1 + sI) / sqrt(2)]])
assert not all(in_field(e, Q5) for e in bloch_matrix(T_GATE))
print("  T    Bloch(T) = Rz(45 deg): NOT in Q(sqrt5) -- no atlas word")

# (b) All five reorientations leave Q(sqrt5): none is an atlas word.
print()
for solid in SOLIDS:
    m_fb, R0s, demand = REORIENT[solid]
    assert not all(in_field(e, Q5) for e in R0s), solid
    print(f"  {solid:14s} Fourier block {m_fb}, demands {demand}:"
          f" not in Q(sqrt5)")

# (c) The pose theorem, on the SIC (the smallest orbit), with its float
# transcription guard -- and the reducible negative control.
ok, evaluate = exact_reposed_twirl_R2("tetrahedron")
assert ok
R0 = np.array(REORIENT["tetrahedron"][1], dtype=float)
M, off = channel_R2(load_vertices("tetrahedron") @ R0,
                    load_rotations("T"), T_NOISE, t_NOISE)
Me, offe = evaluate(R0, T_NOISE, t_NOISE)
gap = max(np.abs(Me - M).max(), np.abs(offe - off).max())
# the caller contract: the generic boolean above must pair
# with a value-for-value float bridge -- asserted, as the module's own
# caller asserts it, so a drift here dies instead of printing large
assert gap < 1e-12, gap
print(f"\n  tetrahedron: M(sC) = tr(C^T C T)/3 Id, off = 0, identically in"
      f" (C, T, t);")
print(f"  evaluated at C = Decker's R and the probe vs channel_R2:"
      f" max|diff| = {gap:.1e}")
R_F = bloch_matrix(atlas_gates()["F"])
assert not exact_reposed_twirl_R2("octahedron",
                                  [sp.eye(3), R_F, R_F * R_F])[0]
print("  negative control: the reducible C_3 coin fails the theorem, at"
      " every pose")

# (d) The coset laws, noiseless family tr(m)/3: centred on zero, RMS 1/3.
print(f"\n  {'solid':14s} {'coset':>6s} {'induced kappa range':>21s}"
      f" {'<k>':>9s} {'<k^2>':>7s}")
for solid in SOLIDS:
    R0 = np.array(REORIENT[solid][1], dtype=float)
    rots = load_rotations(COVARIANCE[solid])
    kaps = np.array([np.trace(h @ R0) / 3 for h in rots])
    assert abs(kaps.mean()) < 1e-12
    assert abs((kaps ** 2).mean() - 1 / 9) < 1e-12
    print(f"  {solid:14s} {len(rots):>6d}  [{kaps.min():+7.4f},"
          f" {kaps.max():+7.4f}] {kaps.mean():>9.1e}"
          f" {(kaps ** 2).mean():>7.4f}")

# ... and one member fed through the two-list channel itself: believe m d_k,
# measure d_k, and the average returns exactly tr(m T)/3 Id, zero offset.
R0 = np.array(REORIENT["icosahedron"][1], dtype=float)
rots = load_rotations("I")
s_d = load_vertices("icosahedron") @ R0
memb = rots[5] @ R0
M, off = channel_R2(s_d @ memb.T, rots, T_NOISE, t_NOISE, s_actual=s_d)
kap = np.trace(memb @ T_NOISE) / 3
assert np.abs(M - kap * np.eye(3)).max() < 1e-10
assert np.abs(off).max() < 1e-9
print(f"\n  coset member #5 (icosahedron), two-list channel: kappa ="
      f" {kap:+.6f},")
print(f"  max|M - kappa Id| = {np.abs(M - kap * np.eye(3)).max():.1e},"
      f"  max|off| = {np.abs(off).max():.1e}")

  F    Bloch matrix: signed permutation     -- in Q(sqrt5)
  H    Bloch matrix: signed permutation     -- in Q(sqrt5)
  Phi  Bloch matrix: golden half-integers   -- in Q(sqrt5)
  S    Bloch matrix: signed permutation     -- in Q(sqrt5)
  X    Bloch matrix: signed permutation     -- in Q(sqrt5)
  Z    Bloch matrix: signed permutation     -- in Q(sqrt5)
  closure: <Bloch(H), Bloch(S)> = 24 signed permutations -- every Clifford
  T    Bloch(T) = Rz(45 deg): NOT in Q(sqrt5) -- no atlas word

  tetrahedron    Fourier block 2, demands sqrt2: not in Q(sqrt5)
  octahedron     Fourier block 3, demands sqrt3: not in Q(sqrt5)
  cube           Fourier block 4, demands sqrt2: not in Q(sqrt5)
  icosahedron    Fourier block 3, demands sqrt3: not in Q(sqrt5)


  dodecahedron   Fourier block 5, demands tau/sqrt(tau+2): not in Q(sqrt5)



  tetrahedron: M(sC) = tr(C^T C T)/3 Id, off = 0, identically in (C, T, t);
  evaluated at C = Decker's R and the probe vs channel_R2: max|diff| = 4.4e-16
  negative control: the reducible C_3 coin fails the theorem, at every pose

  solid           coset   induced kappa range       <k>   <k^2>
  tetrahedron        12  [-0.3333, +0.8047]  -1.9e-17  0.1111
  octahedron         24  [-0.3285, +0.7003]   2.8e-17  0.1111
  cube               24  [-0.3333, +0.8047]  -9.5e-17  0.1111
  icosahedron        60  [-0.3333, +0.8697]   4.2e-18  0.1111
  dodecahedron       60  [-0.3333, +0.9004]   1.3e-17  0.1111

  coset member #5 (icosahedron), two-list channel: kappa = +0.554049,
  max|M - kappa Id| = 8.9e-16,  max|off| = 5.6e-18


## 3 (cont.) Decker's outcome order — the reframing, fourth application

Finding 5. A reorientation carries vertices to vertices but not indices to indices, so a second
correction stands between Decker's circuits and our vertex list: a relabelling of outcomes.
Pricing it needs a datum the rotation never sees — his vertex list **in his outcome order** —
so the five circuits are rebuilt from his own formulas. The order is not ours to choose: he
fixes it twice, as the columns of his printed $M$ and as the numbered vertices of his figures,
so the rebuild is *self-certifying* — it must reproduce his printed columns value for value, in
his order, anchored on his stated vertex 1, and a wrong Fourier convention dies loudly instead
of returning a plausible permutation.

Then the reframing again: skipping the relabelling means the estimator *believes* one list of
maps while the device performs another — and for an irreducible draw any fixed mismatch twirls
to exactly $\frac{\operatorname{tr}(BT)}{V}\,\mathrm{Id}$ with zero offset,
$B = \sum_k b_ka_k^\top$ pairing belief against device. No bias — no relabelling can tilt a
twirled estimator — but the estimator shrinks by the overlap $\kappa$, for a $1/\kappa^2$ shot
premium the table prices per solid, from benign to catastrophic. That the premium is *exactly*
$1/\kappa^2$ for every state is a second fact, checked below: each coordinate Pauli's
single-shot second moment stays $3/\kappa^2$, which needs the atlas pose's coordinate
half-turns, not just irreducibility. The anchors close the story with labels that have no geometry behind them:
the antipodal belief, every SIC derangement, and the exhaustive scramble law over all $V!$
bijections — where, noiselessly, $\kappa$ can vanish
outright and the estimator is killed rather than taxed.

In [29]:
# === Decker's circuits, rebuilt from his formulas (lifted from randomized_decker.py) ===

# Decker's parameters, verbatim from Secs. 6-10, unrescaled (|a|^2 + |b|^2 = 1).
_DA3, _DB3 = math.sqrt((3 + math.sqrt(3)) / 6), math.sqrt((3 - math.sqrt(3)) / 6)

_DP = math.sqrt(75 + 30 * math.sqrt(5)) / 30

_DM = math.sqrt(75 - 30 * math.sqrt(5)) / 30

_DAD, _DBD = math.sqrt(0.5 + _DP), math.sqrt(0.5 - _DP)

_DGD, _DDD = math.sqrt(0.5 + _DM), math.sqrt(0.5 - _DM)   # Sec. 9 (dodecahedron)

_DGI, _DDI = math.sqrt(0.5 - _DM), math.sqrt(0.5 + _DM)   # Sec. 10 (exchanged)

# sign of the exponent in F_m; +1 only for the cube (see the note above)
FOURIER_SIGN = {"tetrahedron": -1, "octahedron": -1, "cube": +1,
                "icosahedron": -1, "dodecahedron": -1}

# "vertex 1 is given by the vector ..." -- Secs. 6, 7, 8 and Secs. 9, 10
_ANCHOR_TOC = (math.sqrt(2 / 3), 0.0, 1 / math.sqrt(3))

_ANCHOR_ID = (math.sqrt((10 - 2 * math.sqrt(5)) / 15), 0.0,
              math.sqrt((5 + 2 * math.sqrt(5)) / 15))

DECKER_ANCHOR = {"tetrahedron": _ANCHOR_TOC, "octahedron": _ANCHOR_TOC,
                 "cube": _ANCHOR_TOC, "icosahedron": _ANCHOR_ID,
                 "dodecahedron": _ANCHOR_ID}


def decker_fourier(m, sign):
    """F_m = sqrt(1/m) (omega^{jk}), omega = exp(sign . 2 pi i/m)."""
    w = np.exp(sign * 2j * np.pi / m)
    return np.array([[w ** (j * k) for k in range(m)]
                     for j in range(m)]) / np.sqrt(m)


def _pad_block(F, dim):
    """F (+) I_{dim-m}: the padding that embeds an m-orbit into a register."""
    out = np.eye(dim, dtype=complex)
    out[:len(F), :len(F)] = F
    return out


def _cnot(nq, ctrl, targ):
    """Q^dag on nq qubits, as every Appendix D figure draws it."""
    P = np.zeros((2 ** nq, 2 ** nq))
    for i in range(2 ** nq):
        b = [(i >> (nq - 1 - p)) & 1 for p in range(nq)]
        if b[ctrl]:
            b[targ] ^= 1
        P[sum(v << (nq - 1 - p) for p, v in enumerate(b)), i] = 1
    return P


def bloch_of_ket(psi):
    """Bloch vector of the (unnormalized, non-zero) state vector psi."""
    rho = np.outer(np.asarray(psi, dtype=complex), np.conj(psi))
    rho = rho / np.trace(rho).real
    return np.array([2 * rho[0, 1].real, -2 * rho[0, 1].imag,
                     (rho[0, 0] - rho[1, 1]).real])


def decker_columns(solid):
    """The POVM vectors as Decker PRINTS them, in his own vertex numbering."""
    if solid == "tetrahedron":                                # Sec. 6, Line (2)
        return [(_DA3, _DB3), (_DA3, -_DB3),
                (_DB3, _DA3 * 1j), (_DB3, -_DA3 * 1j)]
    if solid == "cube":                                       # Sec. 7
        return [(_DA3, _DB3), (_DA3, _DB3 * 1j),
                (_DA3, -_DB3), (_DA3, -_DB3 * 1j),
                (_DB3, -_DA3), (_DB3, -_DA3 * 1j),
                (_DB3, _DA3), (_DB3, _DA3 * 1j)]
    m = 5 if solid == "dodecahedron" else 3
    om = np.exp(-2j * np.pi / m)
    if solid == "octahedron":                                 # Sec. 8, Eq. (4)
        return ([(_DA3, _DB3 * om ** j) for j in range(m)]
                + [(_DB3, -_DA3 * om ** j) for j in range(m)])
    g, d = (_DGI, _DDI) if solid == "icosahedron" else (_DGD, _DDD)
    return ([(_DAD, _DBD * om ** j) for j in range(m)]        # Secs. 9, 10,
            + [(_DBD, -_DAD * om ** j) for j in range(m)]     # Lines (6), (8)
            + [(g, d * om ** j) for j in range(m)]
            + [(d, -g * om ** j) for j in range(m)])


def decker_circuit(solid):
    """Rows of Mtilde^dag . iota -- the circuit exactly as Appendix D draws it.

    Row k is the (conjugated) POVM vector of computational outcome k; rows of
    norm zero are the outcomes the padding leaves unreachable.
    """
    s = FOURIER_SIGN[solid]
    if solid == "tetrahedron":                    # Fig. D.1
        a, b = math.sqrt((3 + math.sqrt(3)) / 12), math.sqrt((3 - math.sqrt(3)) / 12)
        UA, nq, na = np.sqrt(2) * np.array([[a, b], [b, -a]]), 2, 1
        head = (np.kron(np.eye(2), decker_fourier(2, s))
                @ np.diag([1, 1, 1, 1j]) @ np.kron(UA, np.eye(2)))
    elif solid == "octahedron":                   # Fig. D.2
        a, b = math.sqrt((3 + math.sqrt(3)) / 18), math.sqrt((3 - math.sqrt(3)) / 18)
        UA, nq, na = np.sqrt(3) * np.array([[a, b], [b, -a]]), 3, 1
        head = np.kron(UA, _pad_block(decker_fourier(3, s), 4).conj().T)
    elif solid == "cube":                         # Fig. D.3
        a, b = math.sqrt((3 + math.sqrt(3)) / 24), math.sqrt((3 - math.sqrt(3)) / 24)
        UA, nq, na = 2 * np.array([[a, b], [b, -a]]), 3, 1
        head = np.kron(UA, decker_fourier(4, s).conj().T)
    else:                                         # Figs. D.4, D.5
        icos = solid == "icosahedron"
        sc = math.sqrt(1 / 6) if icos else math.sqrt(1 / 10)
        g, d = (_DGI, _DDI) if icos else (_DGD, _DDD)
        a, b, g, d = _DAD * sc, _DBD * sc, g * sc, d * sc
        A = (np.sqrt(3) if icos else np.sqrt(5)) * np.array(
            [[a, b, g, d], [b, -a, d, -g], [g, -d, -a, b], [d, g, -b, -a]])
        d_iso = np.abs(A.conj().T @ A - np.eye(4)).max()
        assert d_iso < 1e-12, (solid, d_iso)
        nq, na = (4, 2) if icos else (5, 2)
        Fb = (_pad_block(decker_fourier(3, s), 4) if icos
              else _pad_block(decker_fourier(5, s), 8))
        head = np.kron(A.conj().T, Fb.conj().T)
    # Q^dag: target the last ancilla-register bit, control the data wire. This
    # is Decker's Q ("fixes the first row, maps the (m+2)nd to the second") for
    # every solid, and the CNOT each Appendix D caption spells out in kets.
    iota = np.zeros((2 ** nq, 2))
    iota[0, 0] = iota[1, 1] = 1
    return head @ _cnot(nq, ctrl=nq - 1, targ=na - 1).T @ iota


def decker_vertices(solid):
    """(V, 3) Bloch vertices in DECKER's outcome order, the live indices, and W.

    W (the circuit rows) comes back with them so that a caller wanting both the
    vertices and the rows they came from need not rebuild the circuit -- the
    kron, the 2^nq CNOT and the isometry check are not cheap at the dodecahedron.
    """
    W = decker_circuit(solid)
    live = [k for k in range(len(W)) if np.linalg.norm(W[k]) > 1e-9]
    return np.array([bloch_of_ket(np.conj(W[k])) for k in live]), live, W

In [30]:
# === His outcome order, and what mislabelling costs (check_decker_outcome_order) ===

KEPT = {"tetrahedron": 1, "octahedron": 0, "cube": 0,
        "icosahedron": 0, "dodecahedron": 0}
LIVE = {"tetrahedron": 4, "octahedron": 6, "cube": 8,
        "icosahedron": 12, "dodecahedron": 20}
verdict, kappas = {}, {}

def second_moment(b, a, R_sm, T_sm, t_sm):
    """(c0, w) of the calibrated single-site SECOND moment under R2 --
    belief list b, device list a, draw R_sm, measurement-side noise (T, t).

    The calibrated single-shot estimate of sigma_alpha is 3 (R_g^T b_k)_alpha
    / kappa, drawn with probability (1/(|G| V))(1 + a_k . (T R_g r + t)), so
    E[o_alpha^2] = (9/kappa^2)(c0_alpha + w_alpha . r) with

        c0_alpha = (1/(V|G|)) sum_g sum_k (R_g^T b_k)_alpha^2 (1 + a_k . t),
        w_alpha  = (1/(V|G|)) sum_g sum_k (R_g^T b_k)_alpha^2  R_g^T T^T a_k.

    Matched and noiseless the moment is 3 per site, so the premium is
    exactly 1/kappa^2 for every state iff c0 = 1/3 and w = 0.  Returns c0
    as a (3,) and w as a (3, 3), alpha down the rows.
    """
    U_sm = np.einsum("gia,ki->gka", R_sm, b) ** 2      # (R_g^T b_k)_alpha^2
    W_sm = np.einsum("gib,ki->gkb", R_sm, a @ T_sm)    # (R_g^T T^T a_k)_beta
    norm_sm = len(b) * len(R_sm)
    return (np.einsum("gka,k->a", U_sm, 1 + a @ t_sm) / norm_sm,
            np.einsum("gka,gkb->ab", U_sm, W_sm) / norm_sm)

sm_worst, d_ico = {}, None
print(f"  {'solid':14s} {'F_m':>10s} {'live':>7s} {'his columns':>12s}"
      f" {'anchor':>7s} {'order kept':>11s} {'kappa':>10s} {'premium':>10s}")
print("  " + "-" * 88)
for solid in SOLIDS:
    d, live, W = decker_vertices(solid)
    cols = decker_columns(solid)
    n = load_vertices(solid)
    V = len(n)
    assert len(live) == len(cols) == V == LIVE[solid], solid
    assert live == sorted(live)              # padding never reorders
    # (i) his printed columns, value for value, in his order -- and (ii)
    # his stated anchor for vertex 1. This is what makes the check
    # self-certifying: a wrong Fourier convention dies here.
    for j in range(V):
        d_col = np.abs(d[j] - bloch_of_ket(cols[j])).max()
        assert d_col < 1e-10, (solid, j, d_col)
    d_anchor = np.abs(d[0] - DECKER_ANCHOR[solid]).max()
    assert d_anchor < 1e-10, (solid, d_anchor)
    # it really is a POVM, and the solid really is congruent to the atlas's
    d_pov = np.abs(sum(np.outer(np.conj(r), r) for r in W) - np.eye(2)).max()
    assert d_pov < 1e-12, (solid, d_pov)
    d_w = max(abs(np.linalg.norm(W[k]) ** 2 - 2 / V) for k in live)
    assert d_w < 1e-12, (solid, d_w)
    R = np.array(REORIENT[solid][1], dtype=float)
    rots = load_rotations(COVARIANCE[solid])

    def induced(RR):
        """The permutation RR induces on the atlas vertex list, with MARGIN.

        What is decided here is DISCRETE -- a permutation -- so a bare
        tolerance is the wrong guarantee: one wrong entry does not perturb
        the kappa below, it jumps it to a different value.  Measured
        over every match this check makes (all 12+24+24+60+60 coset members
        times every vertex, not only the accepted ones): worst accepted
        distance 7.5e-16, closest runner-up 0.7136 -- the solid's own
        minimum inter-vertex distance.  So the assertion is on the MARGIN,
        match inside 1e-9 and runner-up beyond 0.1, which leaves ~7e8 of
        slack on the reject side.  It also subsumes a rounding-grid vertex
        key: sorted(induced(R)) == range(V) is the same claim -- Table
        D.3's R carries THIS circuit's solid onto the atlas -- decided by
        separation rather than by an 8-decimal grid.
        """
        out = []
        for x in d:
            dist = np.linalg.norm(n - RR @ x, axis=1)
            near = np.sort(dist)[:2]
            assert near[0] < 1e-9 and near[1] > 0.1, (solid, near)
            out.append(int(np.argmin(dist)))
        return out
    # ...and Table D.3's R, derived independently there, carries THIS
    # circuit's solid onto the atlas -- the two derivations meet here
    perm = induced(R)
    assert sorted(perm) == list(range(V)), solid
    # D.2's other mismatch, priced there in words: run U_R but skip the
    # relabelling, so outcome j fires as atlas vertex perm[j] and is read
    # as atlas vertex j. The octahedron's key sends every vertex onto a
    # perpendicular one, so that kappa is exactly 0 -- killed rather than
    # taxed -- while the tetrahedron's key is the identity (its drawn R is
    # the one coset member that keeps outcome order), so its skip is free.
    skip = [float(n[j] @ n[perm[j]]) for j in range(V)]
    if solid == "octahedron":
        assert max(abs(x) for x in skip) < 1e-12, skip
    if solid == "tetrahedron":
        assert perm == list(range(V)), perm
    kept = sum(1 for g in rots if induced(g @ R) == list(range(V)))
    # the rotation group acts faithfully on vertices, so distinct coset
    # members induce distinct permutations: at most one can preserve order
    assert kept <= 1 and kept == KEPT[solid], (solid, kept)
    # kappa: the overlap of the believed list (the atlas, index for index)
    # with the actual one (his). Theorem: a fixed misspecification reaches
    # the estimator as this one scalar, so the bill is a 1/kappa^2 shot
    # premium and never a bias (thesis F.3.2).
    kappa = float(np.mean([n[k] @ d[k] for k in range(V)]))
    assert -1 <= kappa <= 1
    # ...and the PAIRING is fed through the two-list channel itself --
    # believe the atlas list, measure his -- which is what makes this
    # overlap the estimator-channel MULTIPLIER rather than a cosine
    # table: kappa Id noiseless, tr(B T)/V Id at the probe, offset 0.
    # (Watched: a belief/device swap fires the noisy assert on four of
    # the five -- the cube's B = n^T d is symmetric and stays blind.)
    M2, off2 = channel_R2(n, rots, np.eye(3), np.zeros(3), s_actual=d)
    assert np.abs(M2 - kappa * np.eye(3)).max() < 1e-10, solid
    assert np.allclose(off2, 0, atol=1e-9), solid
    kap_n = np.trace(n.T @ d @ T_NOISE) / V
    M2, off2 = channel_R2(n, rots, T_NOISE, t_NOISE, s_actual=d)
    assert np.abs(M2 - kap_n * np.eye(3)).max() < 1e-10, solid
    assert np.allclose(off2, 0, atol=1e-9), solid
    # F.3.2's premium sentence -- a fixed misspecification "multiplies the
    # single-shot second moment of a weight-w term by exactly 1/kappa^{2w}"
    # -- claims more than the two-list channel just asserted, which is the
    # FIRST moment. The second is priced per site by second_moment():
    # E[o^2] = (9/kappa^2)(c0 + w . r), and "exactly 1/kappa^2, for every
    # state" is c0 = 1/3 and w = 0 -- the weight-w power then follows from
    # the product dual, site by site. Asserted on the misspecification the
    # appendix prices, believe the atlas list and measure Decker's, under
    # both draws (the covariance group, and the atlas T draw that suffices
    # for the twirl) and both noise settings. 1e-12 against a measured
    # worst of 1.7e-15 on c0 and 1.9e-17 on w over the twenty cases; the
    # negative control after the table misses by 1.3e-2, ten orders away.
    dc0s, dws = [], []
    for R_sm in (rots, load_rotations("T")):
        for T_sm, t_sm in ((np.eye(3), np.zeros(3)), (T_NOISE, t_NOISE)):
            c0_sm, w_sm = second_moment(n, d, R_sm, T_sm, t_sm)
            dc0s.append(np.abs(c0_sm - 1 / 3).max())
            dws.append(np.abs(w_sm).max())
    assert max(dc0s) < 1e-12, (solid, max(dc0s))
    assert max(dws) < 1e-12, (solid, max(dws))
    sm_worst[solid] = (max(dc0s), max(dws))
    if solid == "icosahedron":
        d_ico = d                    # the negative control's device list
    # cross-check against the ROTATION-ONLY coset prices tr(h R)/3
    # (check_reorientation_obstruction's scan). This kappa is a rotation-
    # AND-labelling fact, so containment in that family's range is an
    # observation, not a law -- the antipodal anchor below prices at -1,
    # outside every coset range here -- but all five do land inside, and
    # two land ON an end. The tetrahedron's is the sharp one: its outcome
    # order survives (kept = 1), so its kappa IS a coset member's price,
    # the maximum. The cube reaches the minimum by a different route --
    # its eight per-vertex overlaps are not the tetrahedral angle at all
    # but a 4/4 split between -(1+sqrt2)/3 and (sqrt2-1)/3, and only
    # their MEAN is -1/3, the sqrt2's cancelling.
    cos_k = [np.trace(h @ R) / 3 for h in rots]
    assert min(cos_k) - 1e-12 < kappa < max(cos_k) + 1e-12, solid
    if solid == "tetrahedron":
        assert abs(kappa - max(cos_k)) < 1e-12
    if solid == "cube":
        assert abs(kappa - min(cos_k)) < 1e-12
        per = np.sort([n[k] @ d[k] for k in range(V)])
        assert np.abs(per[:4] + (1 + math.sqrt(2)) / 3).max() < 1e-12
        assert np.abs(per[4:] - (math.sqrt(2) - 1) / 3).max() < 1e-12
    verdict[solid], kappas[solid] = perm, kappa
    sign = "+" if FOURIER_SIGN[solid] > 0 else "-"
    print(f"  {solid:14s} {f'exp({sign}2pi i/m)':>10s} {len(live):3d}/{2 ** int(np.log2(len(W))):<3d}"
          f" {'reproduced':>12s} {'ok':>7s} {kept:>6d}/{len(rots):<4d}"
          f" {kappa:>+10.6f} {1 / kappa ** 2:>9.2f}x")
assert abs(kappas["cube"] + 1 / 3) < 1e-12                # exactly -1/3
# the tetrahedron's is (1 + sqrt2)/3 exactly: its surviving member is
# R_z(45 deg), so its price is the coherent-error law (1 + 2 cos theta)/3
# read at theta = 45 deg -- the identity tying this table to the
# reorientation coset scan
assert abs(kappas["tetrahedron"] - (1 + math.sqrt(2)) / 3) < 1e-12
assert min(kappas, key=lambda s: abs(kappas[s])) == "dodecahedron"
# F.3.2's prose literals, "weight-one premia from 1.54x to 10621.86x",
# pinned as the strings the sentence prints, through the :.2f the table
# rounds by. Table D.4 is generated from these same kappas
# (randomized_fragments), so the TABLE cannot drift from them; the SENTENCE
# can, and this is its pin -- with its "from ... to": the two are the
# range's ends (the dodecahedron's minimum |kappa| is the line above).
assert f"{1 / kappas['tetrahedron'] ** 2:.2f}" == "1.54", kappas["tetrahedron"]
assert f"{1 / kappas['dodecahedron'] ** 2:.2f}" == "10621.86", kappas["dodecahedron"]
assert max(kappas, key=lambda s: abs(kappas[s])) == "tetrahedron"
print("[ok] all five circuits reproduce Decker's printed vertex lists value for")
print("     value IN HIS OUTCOME ORDER, each anchored on his stated vertex 1.")
print("     The cube alone needs omega = +i (Sec. 7 conjugates Sec. 4's F_m);")
print("     under Sec. 4's convention it yields the same POVM misordered.")
print("[ok] outcome order survives for the TETRAHEDRON alone -- 1 of its 12")
print("     reorientations, the T-dagger already drawn in Figures 4.1 and D.1.")
print("     For the other four it is 0 of 24, 24, 60, 60, so each owes the")
print("     fixed permutation below, applied in classical post-processing:")
print()
for solid in SOLIDS:
    print(f"  {solid:14s} {verdict[solid]}")
print()
print("[ok] and skipping it is not free. Running his circuit against the atlas")
print("     list is unbiased -- no relabelling can tilt a twirled estimator --")
print("     but it shrinks by kappa, for a 1/kappa^2 shot premium: 1.54x, 9.65x,")
print("     9.00x (kappa = -1/3 exactly), 73.60x, and 10621.86x. The three")
print("     failure modes each get an exemplar, one per covariance group: the")
print("     tetrahedron pays a premium and nothing more; the cube flips the")
print("     sign; and the dodecahedron's kappa = -0.0097 leaves 1/kappa ill-")
print("     conditioned and the channel indistinguishable from heavy")
print("     depolarizing noise -- a labelling bug an experimenter would blame")
print("     on the hardware. The modes nest rather than partition (the")
print("     dodecahedron is negative too), so that is the worst mode each")
print("     reaches; the octahedron and icosahedron are the interpolation.")
print("     The pair worth reading together is the cube against the")
print("     octahedron: |kappa| = 0.3333 vs 0.3220 -- near-equal by coincidence,")
print("     not identity -- so 9.00x vs 9.65x, the same bill with opposite")
print("     signs. The sign flip costs nothing in shots; the whole hazard is a")
print("     negative estimator-channel factor discarded as an artefact.")
# The premium's exactness is a property of the ANCHORED draw, not of kappa
# -- the negative control for the second-moment pins in the loop. c0 = 1/3
# needs only an irreducible draw (Schur puts every (R_g^T b_k)_alpha^2 at
# 1/3 on average) and a centred device list (its t-term is mean(a) . t);
# w is a CUBIC moment, E_g[(R_g^T b)_alpha^2 (R_g^T T^T a)_beta], and it
# dies because T, O and I in the atlas pose all contain the three
# coordinate half-turns, each monomial being odd in an axis one of them
# negates. Conjugate T by an h in I \ T and the draw is still a group,
# still irreducible, still inside I -- c0 stays 1/3 -- but its half-turns
# are about h's axes, and on Decker's icosahedron list w comes back at
# 0.0128 (0.0100 at the probe): a second moment that reads the state, which
# no single premium prices. With the lists agreeing the same conjugate
# draw gives w = 0 again, the antipodal list's own odd moments vanishing,
# so what the pins exclude is the PAIRING of a mismatch with an un-anchored
# draw. 1e-3 is a gap, not a boundary: fourteen orders above the accepted
# 1.9e-17, one below the measured 1.3e-2.
rots_T, rots_I = load_rotations("T"), load_rotations("I")
keys_T = {rot_key(g) for g in rots_T}
h_conj = next(g for g in rots_I if rot_key(g) not in keys_T)
T_conj = np.array([h_conj @ g @ h_conj.T for g in rots_T])
keys_c = {rot_key(g) for g in T_conj}
assert len(keys_c) == 12 and keys_c != keys_T           # a DIFFERENT copy of T
assert keys_c <= {rot_key(g) for g in rots_I}           # ... inside I
assert all(rot_key(g1 @ g2) in keys_c for g1 in T_conj for g2 in T_conj)
half_turns = [np.diag(v) for v in ([1., -1, -1], [-1., 1, -1], [-1., -1, 1])]
for g_name in ("T", "O", "I"):
    assert all(any(np.allclose(g, P, atol=1e-12, rtol=0) for g in load_rotations(g_name))
               for P in half_turns), g_name
assert not any(np.allclose(g, P, atol=1e-12, rtol=0) for g in T_conj for P in half_turns)
n_ico = load_vertices("icosahedron")
c0_c, w_c = second_moment(n_ico, d_ico, T_conj, np.eye(3), np.zeros(3))
assert np.abs(c0_c - 1 / 3).max() < 1e-12               # c0 does not see it
assert np.abs(w_c).max() > 1e-3, np.abs(w_c).max()       # w does: 0.0128
_, w_cp = second_moment(n_ico, d_ico, T_conj, T_NOISE, t_NOISE)
# by value, not > 1e-3: w = 0 holds for any fixed map, so this is the only
# assert that sees how T enters w (transposed 0.0114, dropped 0.0128)
assert abs(np.abs(w_cp).max() - 0.0100) < 5e-4, np.abs(w_cp).max()
_, w_cm = second_moment(n_ico, n_ico, T_conj, np.eye(3), np.zeros(3))
assert np.abs(w_cm).max() < 1e-12                        # no mismatch, no w
print()
print("[ok] and the premium is EXACT, for every state: the calibrated single-site")
print("     second moment is (9/kappa^2)(c0 + w . r) with c0 = 1/3 and w = 0 on all")
print("     five solids, under the covariance draw and the T draw, at T = Id and at")
print(f"     the probe -- worst |c0 - 1/3| = {max(v[0] for v in sm_worst.values()):.1e},"
      f" worst |w| = {max(v[1] for v in sm_worst.values()):.1e}. It is the")
print("     anchored draw's doing, not kappa's: a T-conjugate inside I whose")
print(f"     half-turns are not the coordinate ones leaves |w| = {np.abs(w_c).max():.4f} on")
print("     Decker's icosahedron list (c0 still 1/3), a second moment that reads")
print("     the state.")
# Labels with no geometry behind them: two closed-form anchors and one
# exact law, all zero-RNG, each anchor run in BOTH noise settings -- at
# T = Id the two-list channel is belief/device-symmetric (tr B is), so
# only the T_NOISE legs, priced independently at tr(B T)/V, can catch a
# swapped wiring (the six 4-cycle derangements do; the antipodal
# anchor's B = -(V/3) Id is symmetric and stays blind). Antipodal
# relabel: believe the antipode list and the two-list channel is -Id
# exactly (kappa = -1), -tr(T)/3 Id under noise. SIC derangement: the
# tetrahedron's Gram is constant -1/3 off the diagonal, so ANY
# fixed-point-free relabelling prices at kappa = -1/3 exactly -- run
# over all 9 of them, not a representative; T_NOISE lifts the
# degeneracy to nine DISTINCT prices (asserted). Scramble law,
# exhaustive over ALL V! bijections: E[kappa] = 0 and E[kappa^2] =
# 1/(3(V-1)), so a scrambled label is worse on a BIGGER POVM. Worse in
# KIND, too -- noiselessly, and this is the part that does not survive
# being read as a premium: 1/(3(V-1)) is a second moment, and at T = Id
# E[1/kappa^2] does not exist, because kappa vanishes OUTRIGHT on a
# large minority of bijections -- 8 of 24 at V = 4, 200 of 720 at
# V = 6, where it is the single commonest outcome. There the noiseless
# estimator channel is the ZERO map; a noise map revives it, the kill
# being tr(B) = 0 while tr(B T) survives -- asserted below on one
# killed bijection, taxed ~186x at the probe. Among the rest the median
# premium is 9x at V = 4 but 36x at V = 6, against the 9 and 15 that
# reading 3(V-1) as typical would predict -- right at V = 4 by
# coincidence, wrong by 2.4x at V = 6. The unprinted residue beside
# Table D.4, and what it says is that a scrambled label is a different
# failure from a mispriced one.
s6, rot6 = load_vertices("octahedron"), load_rotations("O")
M, off = channel_R2(-s6, rot6, np.eye(3), np.zeros(3), s_actual=s6)
d_M = np.abs(M + np.eye(3)).max()            # measured 2.1e-33
assert d_M < 1e-10, f"antipodal relabel not -Id (max |dev| = {d_M:.2e})"
assert np.allclose(off, 0, atol=1e-9)
M, off = channel_R2(-s6, rot6, T_NOISE, t_NOISE, s_actual=s6)
d_M = np.abs(M + (np.trace(T_NOISE) / 3) * np.eye(3)).max()
assert d_M < 1e-10, f"noisy antipodal relabel not -tr(T)/3 Id ({d_M:.2e})"
assert np.allclose(off, 0, atol=1e-9)
s4, rot4 = load_vertices("tetrahedron"), load_rotations("T")
noisy = []
for p in itertools.permutations(range(4)):
    if any(p[k] == k for k in range(4)):
        continue
    M, off = channel_R2(s4[list(p)], rot4, np.eye(3), np.zeros(3),
                        s_actual=s4)
    d_M = np.abs(M + np.eye(3) / 3).max()    # worst over all 9: 5.6e-17
    assert d_M < 1e-10, f"SIC derangement {p} off -Id/3 ({d_M:.2e})"
    assert np.allclose(off, 0, atol=1e-9), p
    kap = np.trace(s4[list(p)].T @ s4 @ T_NOISE) / 4
    M, off = channel_R2(s4[list(p)], rot4, T_NOISE, t_NOISE, s_actual=s4)
    d_M = np.abs(M - kap * np.eye(3)).max()
    assert d_M < 1e-10, f"noisy SIC derangement {p} off tr(BT)/4 ({d_M:.2e})"
    assert np.allclose(off, 0, atol=1e-9), p
    noisy.append(round(float(kap), 12))
assert len(set(noisy)) == 9                  # the degeneracy lifts whole
# the kill is noiseless: a single-fixed-point bijection has kappa = 0
p0 = (0, 2, 3, 1)
kap0 = np.trace(s4[list(p0)].T @ s4 @ T_NOISE) / 4
M, off = channel_R2(s4[list(p0)], rot4, np.eye(3), np.zeros(3),
                    s_actual=s4)
assert np.abs(M).max() < 1e-10               # the ZERO map, at T = Id
M, off = channel_R2(s4[list(p0)], rot4, T_NOISE, t_NOISE, s_actual=s4)
assert abs(kap0) > 0.05                      # revived: taxed, not killed
assert np.abs(M - kap0 * np.eye(3)).max() < 1e-10, p0
assert np.allclose(off, 0, atol=1e-9), p0
scramble = {}
for s, law, n_zero in ((s4, 1 / 9, 8), (s6, 1 / 15, 200)):
    V = len(s)
    kk = np.array([np.mean([s[p[k]] @ s[k] for k in range(V)])
                   for p in itertools.permutations(range(V))])
    d_mean, d_msq = abs(kk.mean()), abs((kk ** 2).mean() - law)
    assert d_mean < 1e-12, (V, d_mean)
    assert d_msq < 1e-12, (V, d_msq)
    # the killed bijections are a COUNT, so this asserts on the margin the
    # way induced() does, not on a tolerance: exact zeros on one side, the
    # nearest live |kappa| (1/3 at V = 4, 1/6 at V = 6) on the other
    zero = np.abs(kk) < 1e-12
    assert int(zero.sum()) == n_zero, (V, int(zero.sum()))
    assert np.abs(kk[~zero]).min() > 0.1, V
    scramble[V] = (100 * zero.mean(), np.median(1 / kk[~zero] ** 2))
print()
print("[ok] anchors, closed form, both noise settings: believe the antipodes")
print("     and the channel is -Id exactly (kappa = -1; -tr(T)/3 Id at the")
print("     probe); on the SIC EVERY derangement prices at -1/3 exactly, all 9")
print("     of them, the Gram being constant off-diagonal -- a degeneracy the")
print("     probe noise lifts to nine DISTINCT prices, asserted. And")
print("     over ALL V! label bijections, E[kappa] = 0 and E[kappa^2] =")
print("     1/(3(V-1)) -- exhaustive at V = 4 (1/9) and V = 6 (1/15), so a")
print("     scramble is worse on a bigger POVM. But that second moment is not")
print("     a premium and must not be read as one: kappa is exactly ZERO for")
print(f"     {scramble[4][0]:.1f}% of the tetrahedron's bijections and"
      f" {scramble[6][0]:.1f}% of the")
print("     octahedron's -- at T = Id the estimator is killed outright, not")
print("     taxed, and E[1/kappa^2] infinite. A noise map revives the kill")
print("     into a steep tax -- tr(B T) survives where tr(B) = 0, one such")
print(f"     bijection asserted at a {1 / kap0 ** 2:.0f}x premium at the probe. Among the")
print(f"     survivors the median premium is {scramble[4][1]:.0f}x at V = 4 --"
      f" inverting 1/(3(V-1))")
print(f"     happens to say 9 there too -- but {scramble[6][1]:.0f}x at V = 6,"
      f" 2.4x above the 15 it")
print("     predicts. A scrambled label is a different failure from a")
print("     mispriced one.")

  solid                 F_m    live  his columns  anchor  order kept      kappa    premium
  ----------------------------------------------------------------------------------------
  tetrahedron    exp(-2pi i/m)   4/4     reproduced      ok      1/12    +0.804738      1.54x


  octahedron     exp(-2pi i/m)   6/8     reproduced      ok      0/24    +0.321975      9.65x
  cube           exp(+2pi i/m)   8/8     reproduced      ok      0/24    -0.333333      9.00x
  icosahedron    exp(-2pi i/m)  12/16    reproduced      ok      0/60    +0.116566     73.60x
  dodecahedron   exp(-2pi i/m)  20/32    reproduced      ok      0/60    -0.009703  10621.86x
[ok] all five circuits reproduce Decker's printed vertex lists value for
     value IN HIS OUTCOME ORDER, each anchored on his stated vertex 1.
     The cube alone needs omega = +i (Sec. 7 conjugates Sec. 4's F_m);
     under Sec. 4's convention it yields the same POVM misordered.
[ok] outcome order survives for the TETRAHEDRON alone -- 1 of its 12
     reorientations, the T-dagger already drawn in Figures 4.1 and D.1.
     For the other four it is 0 of 24, 24, 60, 60, so each owes the
     fixed permutation below, applied in classical post-processing:

  tetrahedron    [0, 1, 2, 3]
  octahedron     [4, 2, 0, 5, 3, 1

### The tail weight: what the correction buys

The mislabelling premium said what skipping the corrections *costs*; this functional says what
performing them *buys*. The single-Pauli estimate's fourth moment is
$27\,\langle w_x^4 + w_y^4 + w_z^4\rangle$ over the swept snapshot directions — asserted below
to be the exact protocol average at every pose, draw (the minimal $T$ draw included), Pauli and
state: state-free, axis-free, draw-blind, a property of the *posed vertex set* alone. Second
moments cannot see any of it — a union of rotated copies of a 2-design is still a 2-design, so
the variance is $3$ either way — which is why the pose surfaces in the fourth moment or
nowhere. Two SOS
identities pin its range, floor exactly the eight cube directions, ceiling exactly the six
Pauli axes — so the published pose is the tetrahedron's and cube's global optimum and the
octahedron's global pessimum, and the unreoriented (Decker-pose) numbers of Appendix D.2 follow
exactly, the 5-designs immovable at the sphere's own value — and in Decker's pose the cube's
famously light tails do not merely thicken against the octahedron's but *reverse*: Appendix D.2's
point that the cube's advantage is a property of its atlas orientation, demonstrated. The same
functional prices Appendix
F.3.3's $T$-draw orbit POVMs of the dodecahedron — the inscribed-cube eight against the golden
twelve, $\sigma^4 + \tau^4 = 7$ behind the split.

In [31]:
# === The tail-weight functional (check_tail_weight) ===

x, y, z = sp.symbols("x y z")
q_sym = x**4 + y**4 + z**4
S_sym = x**2 + y**2 + z**2
# the range is two SOS identities, not a search: on the sphere S = 1,
#   q - 1/3 = (x^2 - 1/3)^2 + (y^2 - 1/3)^2 + (z^2 - 1/3)^2 >= 0,
#   1 - q   = 2 (x^2 y^2 + y^2 z^2 + z^2 x^2)               >= 0,
# with equality iff every square vanishes: all w_a^2 = 1/3 (the eight
# cube directions) at the floor, at most one nonzero coordinate (the
# six Pauli axes) at the ceiling. Asserted as polynomial identities off
# the sphere too, the slack carrying its (S - 1) multiplier:
assert sp.expand(q_sym - Rational(1, 3)
                 - sum((w**2 - Rational(1, 3))**2 for w in (x, y, z))
                 - Rational(2, 3) * (S_sym - 1)) == 0
assert sp.expand(1 - q_sym - 2 * (x**2 * y**2 + y**2 * z**2 + z**2 * x**2)
                 + (S_sym - 1) * (S_sym + 1)) == 0

def q(w):
    return (np.asarray(w, float) ** 4).sum(axis=-1)

def tail(verts):
    return 27 * float(q(verts).mean())

def moment4(sl, rots, r):
    # E[o_a^4], a = x, y, z: readout probability times the fourth
    # power of the snapshot coordinate, averaged exactly over
    # draw x outcome. This is the R2 estimator with the belief moving
    # with the gate (snapshots and probabilities off the same posed
    # list) -- the sharper flavor, since R1's b^4 = 1 kills its state
    # term before any averaging; R1's orbit sweeps are priced in the
    # dodecahedron block below.
    V = len(sl)
    out = np.zeros(3)
    for Rg in rots:
        p = (1 + (sl @ (Rg @ r))) / V
        out += p @ (81 * (sl @ Rg) ** 4)
    return out / len(rots)

def _rz(t):
    c, s = math.cos(t), math.sin(t)
    return np.array([[c, -s, 0.0], [s, c, 0.0], [0.0, 0.0, 1.0]])

def _ry(t):
    c, s = math.cos(t), math.sin(t)
    return np.array([[c, 0.0, s], [0.0, 1.0, 0.0], [-s, 0.0, c]])

C_GEN = _rz(0.3) @ _ry(0.7) @ _rz(1.1)     # a fixed generic pose probe
R_GEN = np.array([0.24, -0.4, 0.56])       # a fixed generic state probe
ATLAS_TAILS = {"tetrahedron": 9.0, "octahedron": 27.0, "cube": 9.0,
               "icosahedron": 81 / 5, "dodecahedron": 81 / 5}
DECKER_TAILS = {"tetrahedron": 15.0, "octahedron": 12.0, "cube": 15.0,
                "icosahedron": 81 / 5, "dodecahedron": 81 / 5}
rows = {}
for solid in SOLIDS:
    n = load_vertices(solid)
    R = np.array(REORIENT[solid][1], dtype=float)
    # Decker's vertex SET is R^T (atlas set): check_decker_outcome_order
    # has already certified, vertex for vertex, that Table D.3's R
    # carries his rebuilt circuit's solid onto the atlas, and a tail
    # weight is order-blind, so the reorientation transpose stands in
    # for the circuits here (they are not cheap at the dodecahedron).
    d = n @ R
    gen = n @ C_GEN
    draws = ("T",) if COVARIANCE[solid] == "T" else ("T", COVARIANCE[solid])
    for verts in (n, d, gen):
        for g in draws:                      # the minimal twirl included
            for r in (np.zeros(3), R_GEN):
                m = moment4(verts, load_rotations(g), r)
                # measured worst over all 100 calls: axis spread
                # 3.6e-15, deviation from the functional 2.5e-14
                assert m.max() - m.min() < 1e-10, (solid, g)
                assert abs(m.mean() - tail(verts)) < 1e-10, (solid, g)
        assert 9 - 1e-9 <= tail(verts) <= 27 + 1e-9, solid
    rows[solid] = (tail(n), tail(d), tail(gen))
    assert abs(tail(n) - ATLAS_TAILS[solid]) < 1e-10, solid
    assert abs(tail(d) - DECKER_TAILS[solid]) < 1e-10, solid
# the generic pose lands the three cubic-axis solids strictly interior
# (measured 16.115 / 16.327 / 16.115): the extremes are POSE facts
for solid in ("tetrahedron", "octahedron", "cube"):
    assert 9.5 < rows[solid][2] < 26.5, solid
# the tetrahedron prices as the cube in EVERY common pose: q is even
# and the cube's eight vertices are the tetrahedron's four with their
# antipodes, so the two vertex averages coincide identically
assert all(abs(a - b) < 1e-12
           for a, b in zip(rows["tetrahedron"], rows["cube"]))
# equality cases, attained vertex by vertex (measured 2.2e-16): the
# atlas tetrahedron and cube sit at the SOS floor (every coordinate
# +-1/sqrt3 -- the global optimum, and the only attaining points), the
# atlas octahedron at the ceiling (the Pauli axes -- the pessimum)
for solid, val in (("tetrahedron", 1 / 3), ("cube", 1 / 3),
                   ("octahedron", 1.0)):
    assert np.abs(q(load_vertices(solid)) - val).max() < 1e-12, solid
# the icosahedral closed forms behind the immovable 81/5. Atlas
# icosahedron: EVERY vertex has q = 3/5 exactly, (1 + tau^4)/(1 +
# tau^2)^2 = 3/5 -- so even a sub-orbit sweep of it reads 81/5. Atlas
# dodecahedron: q takes exactly two values, 1/3 on its cube eight and
# 7/9 on its twelve (0, +-sigma, +-tau)/sqrt3 vertices -- the pretty
# identity is sigma^4 + tau^4 = 7 -- and 27 x the vertex-weighted mean
# is (8 x 9 + 12 x 21)/20 = 81/5. In Decker's pose neither list is
# q-constant any more (spread asserted), yet both means hold at 3/5:
# a rotated 5-design is a 5-design, the second collapse mechanism.
assert sp.simplify((1 + TAU_SYM**4) / (1 + TAU_SYM**2)**2
                   - Rational(3, 5)) == 0
assert sp.simplify(SIG_SYM**4 + TAU_SYM**4 - 7) == 0
assert np.abs(q(load_vertices("icosahedron")) - 3 / 5).max() < 1e-12
qd = np.sort(q(load_vertices("dodecahedron")))
assert np.abs(qd[:8] - 1 / 3).max() < 1e-12
assert np.abs(qd[8:] - 7 / 9).max() < 1e-12
for solid in ("icosahedron", "dodecahedron"):
    dd = load_vertices(solid) @ np.array(REORIENT[solid][1], dtype=float)
    assert q(dd).max() - q(dd).min() > 0.05, solid
    assert abs(q(dd).mean() - 3 / 5) < 1e-12, solid
# the first collapse mechanism, isolated: every T and O rotation
# preserves q pointwise (measured 1.1e-15 over all 24 on a posed set),
# where a generic I rotation moves a generic direction's q by ~0.3 --
# the icosahedral draws really do need the design argument
d_oct = load_vertices("octahedron") @ np.array(
    REORIENT["octahedron"][1], dtype=float)
dev_O = max(np.abs(q(d_oct @ g) - q(d_oct)).max()
            for g in load_rotations("O"))
assert dev_O < 1e-12, dev_O
u = np.array([0.36, 0.48, 0.80])             # a generic unit direction
dev_I = max(abs(q(g.T @ u) - q(u)) for g in load_rotations("I"))
assert dev_I > 0.05, dev_I

# the octahedron's own pose landscape: vertices +- the columns of C, so
# the functional is f(C) = 9 sum_ac C_ac^4. The pose-minimum is 11, at
# the frame whose columns are three of the signed permutations of
# (1/3, 2/3, 2/3) -- EXACT there, per column 1/81 + 16/81 + 16/81 =
# 11/27. Those 24 permutations ARE the O-orbit of the direction, but
# they sit on 12 axes and no octahedron: the orbit names the pool the
# three columns are drawn from, never the pose. Minimality is PROVED
# below; the deterministic Euler-grid descent that follows (coarse
# 40 x 20 x 40, three shrink rounds, zero RNG) lands at 11 + 1.0e-7 and
# is kept as corroboration, and for its maximum: the coarse grid
# contains the identity, so it recovers the atlas pose's 27 -- the
# global pessimum as a pose-landscape fact. Decker's pose sits at
# exactly 12: nearly attaining what no pose of the octahedron beats.
C_STAR = np.array([[-1, 2, 2], [-2, 1, -2], [-2, -2, 1]]) / 3.0
assert np.abs(C_STAR @ C_STAR.T - np.eye(3)).max() < 1e-15
assert abs(np.linalg.det(C_STAR) - 1) < 1e-12
M_STAR = Matrix([[-1, 2, 2], [-2, 1, -2], [-2, -2, 1]]) / 3
assert 9 * sum(e**4 for e in M_STAR) == 11   # exact, in rationals
assert abs(9 * float((C_STAR ** 4).sum()) - 11) < 1e-12

# MINIMALITY, proved -- the grid below only corroborates it. In
# axis-angle R = cI + s K_n + (1 - c) n n^T, P = sum_ij R_ij^4 is even
# in s (so s^2 -> 1 - c^2 is lossless) and symmetric in u_i = n_i^2, so
# P = P(c, e_2, e_3) at e_1 = 1, where dP/de_3 = 12 (1 - c)^3 (4c + 3)
# is free of e_3: P is AFFINE in it, so at fixed (c, e_2) the minimum
# sits at an endpoint of the feasible e_3-interval -- and no case split
# is owed, an affine function bottoming out at an endpoint even where
# its slope vanishes (c = 1, c = -3/4). Lagrange on {u >= 0, e_1 = 1,
# e_2 fixed} gives u_j u_k = lam + mu (1 - u_i); times u_i, every u_i
# is a root of mu t^2 - (lam + mu) t + e_3, so an endpoint has two u_i
# equal or (the constraint binding) one u_i = 0. Both families come out
# at 11/9 below: tail weight 9 x 11/9 = 11, attained at C_STAR above.
cc, ss = sp.symbols("c s", real=True)
nn = Matrix(sp.symbols("n1 n2 n3", real=True))
Kn = Matrix([[0, -nn[2], nn[1]], [nn[2], 0, -nn[0]], [-nn[1], nn[0], 0]])
Rax = cc * sp.eye(3) + ss * Kn + (1 - cc) * (nn * nn.T)
Pax = sp.Poly(sp.expand(sum(e ** 4 for e in Rax)), ss).as_expr().subs(
    {ss ** 4: (1 - cc ** 2) ** 2, ss ** 2: 1 - cc ** 2})
uu = sp.symbols("u1 u2 u3", nonnegative=True)
for w, v in zip(nn, uu):
    Pax = sp.expand(Pax.subs(w ** 6, v ** 3).subs(w ** 4, v ** 2)
                    .subs(w ** 2, v))
sym, rem, _ = sp.symmetrize(Pax, uu, formal=True)
Pe = sp.expand(sym.subs(sp.Symbol("s1"), 1))     # n is a unit vector
assert rem == 0 and sp.degree(Pe, sp.Symbol("s3")) == 1
assert sp.expand(sp.diff(Pe, sp.Symbol("s3"))
                 - 12 * (1 - cc) ** 3 * (4 * cc + 3)) == 0
aa = sp.symbols("a", real=True)

def _rect_min(E, hi):
    """Exact min of the polynomial E(c, a) over [-1, 1] x [0, hi]: the
    four corners, the interior critical points, and each edge's own."""
    box = {cc: (-1, 1), aa: (0, hi)}
    vals = [E.subs({cc: p, aa: q}) for p in box[cc] for q in box[aa]]
    for so in sp.solve([E.diff(cc), E.diff(aa)], [cc, aa], dict=True):
        if (len(so) == 2 and all(v.is_real for v in so.values())
                and all(box[k][0] <= v <= box[k][1]
                        for k, v in so.items())):
            vals.append(E.subs(so))
    for var, oth in ((cc, aa), (aa, cc)):
        for v0 in box[oth]:
            f = E.subs(oth, v0)
            vals += [f.subs(var, r) for r in sp.solve(f.diff(var), var)
                     if r.is_real and box[var][0] <= r <= box[var][1]]
    return min(sp.nsimplify(v) for v in vals)

for uvec, hi in (((aa, aa, 1 - 2 * aa), Rational(1, 2)),  # two u_i equal
                 ((aa, 1 - aa, 0), 1)):                   # one u_i = 0
    assert _rect_min(sp.expand(Pax.subs(dict(zip(uu, uvec)))),
                     hi) == Rational(11, 9), uvec

def _stack(ts, kind):
    c, s, z = np.cos(ts), np.sin(ts), np.zeros_like(ts)
    rows_ = ([[c, -s, z], [s, c, z], [z, z, z + 1]] if kind == "z"
             else [[c, z, s], [z, z + 1, z], [-s, z, c]])
    return np.stack([np.stack(r, -1) for r in rows_], -2)

al = np.linspace(0, 2 * np.pi, 40, endpoint=False)
be = np.linspace(0, np.pi, 20)
ga = np.linspace(0, 2 * np.pi, 40, endpoint=False)
f_max = None
for _ in range(4):
    C = np.einsum("aij,bjk,ckl->abcil",
                  _stack(al, "z"), _stack(be, "y"), _stack(ga, "z"))
    f = 9 * (C ** 4).sum((-2, -1))
    if f_max is None:
        f_max = float(f.max())
    i, j, k = np.unravel_index(np.argmin(f), f.shape)
    da, db, dg = al[1] - al[0], be[1] - be[0], ga[1] - ga[0]
    al = np.linspace(al[i] - da, al[i] + da, 21)
    be = np.linspace(be[j] - db, be[j] + db, 21)
    ga = np.linspace(ga[k] - dg, ga[k] + dg, 21)
f_min = float(f.min())
assert 11 - 1e-6 < f_min < 11 + 1e-5         # measured 11 + 1.0e-7
assert abs(f_max - 27) < 1e-9                # Euler (0, 0, 0) is in-grid
f_dec = 9 * float((np.array(REORIENT["octahedron"][1], dtype=float)
                   ** 4).sum())
assert abs(f_dec - 12) < 1e-12               # measured 2.1e-15
assert abs(f_dec - rows["octahedron"][1]) < 1e-12
assert f_min < f_dec < rows["octahedron"][0]

# Appendix F.3.3's footnote prices the T-draw's two dodecahedral orbit
# POVMs at E[o^4] = 9 and 21 against the full solid's 81/5 -- the same
# functional, read off the two q-values above. The alignment vertex
# sits on the twelve-vertex family: its T-orbit is that family whole
# (12 of the 20 vertices, 6 of the 10 axes, antipodally closed), q =
# 7/9 throughout, tail 27 x 7/9 = 21. Any remaining vertex is on the
# inscribed cube: T-orbit of size 4, antipodes completing the
# eight-vertex POVM, q = 1/3, tail 9. R1's swept set IS the orbit, so
# these are the R1 flavor of the identity, trivially state-free.
s20, rT = load_vertices("dodecahedron"), load_rotations("T")
_, v0 = alignment(s20)
sw = np.array([g.T @ v0 for g in rT])
m = 81 * (sw ** 4).mean(axis=0)
assert m.max() - m.min() < 1e-10 and abs(m.mean() - 21) < 1e-10
keys20 = {rot_key(v) for v in s20}
orb = {rot_key(w) for w in sw}
assert len(orb) == 12 and orb <= keys20      # 12 of 20, hit once each
assert {rot_key(-w) for w in sw} == orb      # antipodally closed
v1 = next(v for v in s20 if rot_key(v) not in orb)
sw1 = np.array([g.T @ v1 for g in rT])
m1 = 81 * (sw1 ** 4).mean(axis=0)
assert m1.max() - m1.min() < 1e-10 and abs(m1.mean() - 9) < 1e-10
orb1 = {rot_key(w) for w in sw1}
povm1 = orb1 | {rot_key(-w) for w in sw1}
assert len(orb1) == 4 and len(povm1) == 8    # the inscribed cube
assert povm1 <= keys20 and len(orb | povm1) == 20
assert (8 * 9 + 12 * 21) / 20 == 81 / 5      # the families' mean is F.1's

print(f"  {'solid':14s} {'atlas':>8s} {'decker':>8s} {'generic':>9s}")
print("  " + "-" * 44)
for solid in SOLIDS:
    a, dd, gg = rows[solid]
    print(f"  {solid:14s} {a:8.3f} {dd:8.3f} {gg:9.4f}")
print()
print("[ok] the tail weight -- the single-Pauli estimate's fourth moment --")
print("     is 27<w_x^4 + w_y^4 + w_z^4> over the swept directions, asserted")
print("     as the exact protocol average at every pose, draw (T included),")
print("     Pauli and state: state-free, axis-free, draw-blind. Two SOS")
print("     identities put it in [9, 27], floor exactly the eight cube")
print("     directions, ceiling exactly the six Pauli axes -- the atlas pose")
print("     is the tetrahedron's and cube's GLOBAL optimum and the")
print("     octahedron's global pessimum, and D.2's unreoriented numbers are")
print("     pinned: 9 -> 15 (tetrahedron and cube, equal in every common")
print("     pose), 27 -> 12 (octahedron), the 5-designs immovable at 81/5.")
print("[ok] the octahedron's own pose landscape: minimum 11 -- exact at the")
print("     frame of signed (1/3, 2/3, 2/3) permutations, and PROVED")
print("     minimal: sum_ij R_ij^4 is affine in e_3(n^2), so Lagrange")
print("     forces two endpoint families and both return 11/9. The grid")
print(f"     descent corroborates at {f_min:.7f}, so Decker's 12 nearly")
print("     attains what no pose of it beats; the grid's maximum is the")
print("     atlas pose's own 27.")
print("[ok] and F.3.3's T-draw orbit POVMs price by the same functional: the")
print("     dodecahedron's q is 1/3 on its cube eight and 7/9 on its twelve")
print("     (sigma^4 + tau^4 = 7), so the two realized POVMs read 9 and 21,")
print("     and their vertex-weighted mean is the full solid's 81/5.")

  solid             atlas   decker   generic
  --------------------------------------------
  tetrahedron       9.000   15.000   16.1154
  octahedron       27.000   12.000   16.3269
  cube              9.000   15.000   16.1154
  icosahedron      16.200   16.200   16.2000
  dodecahedron     16.200   16.200   16.2000

[ok] the tail weight -- the single-Pauli estimate's fourth moment --
     is 27<w_x^4 + w_y^4 + w_z^4> over the swept directions, asserted
     as the exact protocol average at every pose, draw (T included),
     Pauli and state: state-free, axis-free, draw-blind. Two SOS
     identities put it in [9, 27], floor exactly the eight cube
     directions, ceiling exactly the six Pauli axes -- the atlas pose
     is the tetrahedron's and cube's GLOBAL optimum and the
     octahedron's global pessimum, and D.2's unreoriented numbers are
     pinned: 9 -> 15 (tetrahedron and cube, equal in every common
     pose), 27 -> 12 (octahedron), the 5-designs immovable at 81/5.
[ok] the oc

## 4. The implementation ledger

The suite's Section 4 collects the bill: native (Naimark) ancilla counts against the projective
route's atlas circuits, the field demand each solid's exactness obstruction names, and the draw
that clears both bars. Decker's native circuits are Naimark dilations — a register of dimension
$\ge V$, prepared by inter-orbit unitaries whose amplitudes carry exactly the radicals the
lemma forces; that bound makes their ancilla counts minimal *within* the dilation route, so on
their own terms they are not beatable. The projective route beats them by *leaving* those terms
— one qubit, a coin, Clifford-cheap coset circuits — wherever antipodality permits it to exist.

(Run as a script — never from this notebook — the entry point also emits this ledger and five
more fragments as LaTeX into `code/data/`; every printed number is recomputed from the same
loaders and primitives the checks used. The thesis's Table 5.2 is this same ledger transposed —
solids as columns, the master spec sheet `spec_sheet()` derives from these primitives — so the
two shapes are one table.)

In [32]:
# === Section 4: the implementation ledger (print_ledger) ===

demand = {"tetrahedron": "sqrt3", "octahedron": "-- (none)", "cube": "sqrt3",
          "icosahedron": "sqrt(5+2 sqrt5)", "dodecahedron": "sqrt3"}
draw = {"octahedron": "2T", "cube": "2T", "icosahedron": "2T",
        "dodecahedron": "2I"}
per_shot = {"octahedron": "depth <= 2, 0 Phi, A = I",
            "cube": "depth <= 2, 0 Phi, + A",
            "icosahedron": "depth <= 2, 0 Phi, + A",
            "dodecahedron": "depth <= 4, <= 1 Phi (mean 0.8), + A"}
ancillas = {"tetrahedron": 1, "octahedron": 2, "cube": 2,
            "icosahedron": 3, "dodecahedron": 4}
print(f"  {'POVM (V)':18s} {'native anc.':>11s}  {'field demand':>15s}  {'proj. draw':>10s}  projective per-shot")
print("  " + "-" * 92)
for solid in SOLIDS:
    V = len(load_vertices(solid))
    anc = math.ceil(math.log2(V)) - 1    # Naimark register dim >= V
    assert anc == ancillas[solid]
    print(f"  {solid:12s} ({V:2d})  {anc:>10d}  {demand[solid]:>15s}"
          f"  {draw.get(solid, '--'):>10s}  {per_shot.get(solid, '-- (indecomposable)')}")
print("  (native ancillas: a Naimark register of dim >= V needs ceil(log2 V) qubits")
print("   incl. the system -- minimal within the dilation route, whose radicals are")
print("   forced by finding 3. twirled-native = native + the same 2T draw on every")
print("   row: 0 Phi, depth <= 2. estimator-channel factors: T_zz (randomized-")
print("   projective) vs tr(T)/3 (twirled-native); native instead ASSUMES")
print("   depolarizing and corrects a calibrated scalar, with the known blind spot.)")

  POVM (V)           native anc.     field demand  proj. draw  projective per-shot
  --------------------------------------------------------------------------------------------
  tetrahedron  ( 4)           1            sqrt3          --  -- (indecomposable)
  octahedron   ( 6)           2        -- (none)          2T  depth <= 2, 0 Phi, A = I
  cube         ( 8)           2            sqrt3          2T  depth <= 2, 0 Phi, + A
  icosahedron  (12)           3  sqrt(5+2 sqrt5)          2T  depth <= 2, 0 Phi, + A
  dodecahedron (20)           4            sqrt3          2I  depth <= 4, <= 1 Phi (mean 0.8), + A
  (native ancillas: a Naimark register of dim >= V needs ceil(log2 V) qubits
   incl. the system -- minimal within the dilation route, whose radicals are
   forced by finding 3. twirled-native = native + the same 2T draw on every
   row: 0 Phi, depth <= 2. estimator-channel factors: T_zz (randomized-
   projective) vs tr(T)/3 (twirled-native); native instead ASSUMES
   depolarizing

### The three corners

Disambiguated and priced, the closing trade-off is a triangle, not a dichotomy:

| corner | circuit | calibration | SIC? | price |
|---|---|---|---|---|
| **native** | Decker dilation, no random gates | *assumes* depolarizing; fits one scalar $\hat\eta$ from $\vert0\rangle$ shots | yes | structurally blind to any channel fixing $\vert0\rangle$ |
| **twirled-native** | native + a $2T$ draw | depolarizing is a *theorem*; factor $\operatorname{tr}T/3$ | yes | the dilation's ancillas, plus a depth-$\le2$ all-Clifford draw |
| **randomized-projective** | one qubit: $U_g$, $A$, measure $Z$ | depolarizing is a theorem; factor $T_{zz}$ | no | the SIC forfeit, the alignment $A$, and — for the dodecahedron alone — the golden gate per shot |

Two sentences carry the whole picture. **The SIC is not the price of the twirl — the ancilla
is:** twirled-native keeps the tetrahedron and twirls it, paying only the dilation it already
owed. **The magic cannot be dodged, only relocated:** the projective route saves the ancillas
and pays $A$; the native route pays the radicals; the octahedron is the one POVM whose
*projective* corner has no field bill at all — $A = \mathrm{Id}$, literally random Pauli
measurements (its native corner still cannot be exact: the weight obstruction stands in every
pose, and Decker's octahedral circuit carries a $\sqrt3$ from his orientation). The other two
bills survive even there: the native corner keeps its blind spot, and the twirled-native
corner its ancillas and its own factor — the corners never collapse; their common obstruction
just vanishes at that one point.

## 5. Remark: the unitary-design ladder

The twirl needs only a unitary 2-design, and $2T$ is the minimal *group* one in $d = 2$ — but
the three binary groups climb higher: exact 2-/3-/5-designs, read off frame potentials against
the Catalan numbers (for a group the potential is an integer, so failing a level overshoots by
at least 1; $2I$ meets $t = 5$ exactly and first fails at $t = 6$, the degree of the icosahedral
invariant). The glue line re-derives the *spherical* design strengths of the vertex sets by
float moment averaging and pins them to `povm_properties.py`'s `EXPECTED_DESIGN` — the one
place the repo fixes the ladder's values. A pin, not a corroboration: the exact number-field
derivation behind those five integers runs inside that script's own `main()`, not here — two
mechanisms sharing nothing, meeting at one hardcoded dict.

In [33]:
# === Frame potentials + spherical strengths (lifted) ===

def frame_potential(U, t):
    """Frame potential F_t = mean over pairs of |tr(U_a+ U_b)|^(2t).

    Equals the Haar value (the Catalan number C_t) iff the set is a unitary
    t-design; for a group the value is an integer, so failing a level
    overshoots the Catalan number by at least 1.
    """
    tr = np.einsum("aij,bij->ab", np.conj(U), U)
    return float((np.abs(tr) ** (2 * t)).mean())


def design_strength(s, t_max=8):
    """Spherical t-design strength of a unit-vector set.

    t is the largest degree at which the vertex average of every monomial
    matches its average over the sphere. Derived here rather than read off
    Table 2.2 so that the ledger has a single generator; the values agree
    with povm_properties.py's independent computation (2, 3, 3, 5, 5).
    """
    def sphere_moment(e):
        if any(k % 2 for k in e):
            return 0.0                      # odd monomials integrate to zero
        num = math.prod(math.prod(range(k - 1, 0, -2)) or 1 for k in e)
        return num / (math.prod(range(sum(e) + 1, 0, -2)) or 1)

    for t in range(1, t_max + 1):
        for e in itertools.product(range(t + 1), repeat=3):
            if sum(e) == t and abs(float(np.mean(np.prod(s ** np.array(e), axis=1)))
                                   - sphere_moment(e)) > 1e-9:
                return t - 1
    return t_max


# === The ladder (check_unitary_designs) ===

catalan = {1: 1, 2: 2, 3: 5, 4: 14, 5: 42, 6: 132}
strength = {"T": 2, "O": 3, "I": 5}
for g, t_max in strength.items():
    U = load_atlas(g)["unitaries"]
    vals = {t: frame_potential(U, t) for t in range(1, t_max + 2)}
    for t in range(1, t_max + 1):
        assert abs(vals[t] - catalan[t]) < 1e-9, (g, t, vals[t])
    assert vals[t_max + 1] > catalan[t_max + 1] + 0.5, (g, vals)
    shown = "  ".join(f"F_{t} = {vals[t]:7.3f}" for t in sorted(vals))
    print(f"  2{g}: {shown}   -> exact {t_max}-design, not {t_max + 1}")
print("  (Haar values are the Catalan numbers 1, 2, 5, 14, 42, 132.)")
print("[ok] 2T/2O/2I are exact unitary 2-/3-/5-designs (2I overshoots: it meets t = 5")
print("     exactly and first fails at t = 6, the degree of the icosahedral invariant);")
print("     the twirl needs only a 2-design, and 2T is the minimal group one in d = 2")


# glue: the SPHERICAL strengths of the vertex sets, pinned to the ladder's
# single source (import is write-free; povm_properties' own exact derivation
# asserts against the same dict when THAT script runs, not here)
from povm_properties import EXPECTED_DESIGN

got = {solid: design_strength(load_vertices(solid)) for solid in SOLIDS}
assert got == EXPECTED_DESIGN, got
print("spherical t-designs:  "
      + "  ".join(f"{s} {t}" for s, t in got.items()))
print("pinned to povm_properties.EXPECTED_DESIGN (the ladder's single source)")

  2T: F_1 =   1.000  F_2 =   2.000  F_3 =   6.000   -> exact 2-design, not 3
  2O: F_1 =   1.000  F_2 =   2.000  F_3 =   5.000  F_4 =  15.000   -> exact 3-design, not 4
  2I: F_1 =   1.000  F_2 =   2.000  F_3 =   5.000  F_4 =  14.000  F_5 =  42.000  F_6 = 133.000   -> exact 5-design, not 6
  (Haar values are the Catalan numbers 1, 2, 5, 14, 42, 132.)
[ok] 2T/2O/2I are exact unitary 2-/3-/5-designs (2I overshoots: it meets t = 5
     exactly and first fails at t = 6, the degree of the icosahedral invariant);
     the twirl needs only a 2-design, and 2T is the minimal group one in d = 2
spherical t-designs:  tetrahedron 2  octahedron 3  cube 3  icosahedron 5  dodecahedron 5
pinned to povm_properties.EXPECTED_DESIGN (the ladder's single source)


## 6. The receipts

The cells above *are* the modules — lifted mechanically at build time — but a committed
notebook can drift after later module edits. This cell closes the gap:

1. **currency**: the build pinned a sha256 of each of the eight module files *and of the
   builder*; the cell re-hashes them now. Any later edit — lifted or not, code or comment —
   fails here naming the rebuild + re-execute pair, so a stale committed notebook cannot
   execute quietly. Textual identity is the whole anti-drift guarantee; the next item is a
   demonstration on top of it;
2. **behavioral spot-checks**: the notebook's rebuilt primitives against the imported
   production modules, each side fed its own inputs — same arithmetic run twice, so the honest
   residual is $0.0$, not a tolerance;
3. **the full report**: the ledger's `spec_sheet()` re-derived and pinned (read-only — the
   writers live in `write_fragments`, behind the `__main__` guard), then the entry point's
   `main()` end to end. Every check staged above runs again inside it, and every check *not*
   staged here — the exact two-bars companion, the full reorientation check, and the rest —
   runs too. The only thing in the suite this notebook never executes is the fragment writers
   themselves: the LaTeX layer over the rows `spec_sheet` just pinned, which is exactly what
   *writes nothing* buys.

The six thesis fragments are emitted only when `randomized_implementations.py` runs as a
script.

In [34]:
# === Anti-drift: currency, spot-checks, spec_sheet, then main() ===

import hashlib
import io
import sys

import randomized_implementations as ri
import randomized_core as rc
import randomized_scalars as rsc
import randomized_twojobs as rtj
import randomized_field as rfl
import randomized_decker as rdk
import randomized_fragments as rfr

# (1) Currency. The files on disk must be the files this notebook was built
# from -- pinned at build time, byte for byte, builder included. Any later
# edit fails HERE, naming the remedy: textual identity is the anti-drift
# guarantee; the spot-checks below are a demonstration on top of it.
_PINNED = {
    "_build_randomization_walkthrough": "b99617c9031498db1854540cd11025aea7cda961d31a7a413ae666a355b676a6",
    "randomized_core": "675cfb8d31ecea738b28bdfa42a27036b7dfa59d3cc152ab01acc95a9696217f",
    "randomized_decker": "0c421850f8c494e5d57f7b62428a3ad46ddbfed363fc21454eb8819922dcb56c",
    "randomized_field": "b66ab58eff519ec6990c0cf749fd2692f042f8e700a860deaf915f011940d1b0",
    "randomized_fragments": "54feffed6d9ceb33bebf3ca460105b6afdbd4bb194eae04b0559f5696118e40f",
    "randomized_implementations": "e8c25f959a7c5d57e284c0bc4724cbb64995fdbd328dff251c155d0318cb6621",
    "randomized_obstruction": "049ab3250faac1c750396957cdcc6669ad0c5de7588ac601123d858d17dfbd7c",
    "randomized_scalars": "dfe5776579db00b23456894d0cc0321ef634669ee5ed77f396b31b331f6394bf",
    "randomized_twojobs": "3819acdc4934c77a00244c9daefa58a45a6c72d55c97836c92f1c0fe1a7d40ea",
}
for _name in sorted(_PINNED):
    _got = hashlib.sha256(Path(f"{_name}.py").read_bytes()).hexdigest()
    assert _got == _PINNED[_name], (
        f"{_name}.py changed since this notebook was built -- rebuild and "
        "re-execute:\n"
        "  uv run python _build_randomization_walkthrough.py\n"
        "  uv run --with jupyter --with nbconvert jupyter nbconvert "
        "--to notebook --execute --inplace randomization_walkthrough.ipynb")
print(f"currency: {len(_PINNED)} files match their build-time sha256 pins\n")

# (2) Behavioral spot-checks, each side fed its own inputs -- same
# arithmetic run twice, so the honest residual is 0.0, not a tolerance.
assert np.array_equal(T_NOISE, ri.T_NOISE)
assert np.array_equal(t_NOISE, ri.t_NOISE)
s, R = load_vertices("icosahedron"), load_rotations("I")
for f_nb, f_mod in ((channel_R1, ri.channel_R1), (channel_R2, ri.channel_R2)):
    M_nb, o_nb = f_nb(s, R, T_NOISE, t_NOISE)
    M_md, o_md = f_mod(rc.load_vertices("icosahedron"),
                       rc.load_rotations("I"), rc.T_NOISE, rc.t_NOISE)
    assert np.array_equal(M_nb, M_md) and np.array_equal(o_nb, o_md)

# core: alignment, rot_key, the two bars, the ladder, the symbolic layer
A_nb, v_nb = alignment(s)
A_md, v_md = rc.alignment(rc.load_vertices("icosahedron"))
assert np.array_equal(A_nb, A_md) and np.array_equal(v_nb, v_md)
assert rot_key(R[7]) == rc.rot_key(rc.load_rotations("I")[7])
b_nb, b_md = two_bars("cube"), rc.two_bars("cube")
assert all(b_nb[k] == b_md[k] for k in
           ("bar_realize", "bar_twirl", "binds", "draw", "min_realize"))
U_T = load_atlas("T")["unitaries"]
assert frame_potential(U_T, 2) == rc.frame_potential(
    rc.load_atlas("T")["unitaries"], 2)
s_c = load_vertices("cube")
assert design_strength(s_c) == rc.design_strength(rc.load_vertices("cube"))
assert sp.simplify(det_invariant(atlas_vertices("cube"))
                   - rc.det_invariant(rc.atlas_vertices("cube"))) == 0

# field: the exact kit over Q (octahedron) AND over the icosahedron's
# algebraic number field -- compared by domain equality, which `is` could
# only ever decide for the QQ singleton
K = solid_field("octahedron")
assert K == rfl.solid_field("octahedron")
s_e = exact_vertices("octahedron", K)
assert s_e == rfl.exact_vertices("octahedron", K)
R_e = [to_field(M_, K) for M_ in exact_rotations("O")]
R_em = [rfl.to_field(M_, K) for M_ in rfl.exact_rotations("O")]
assert R_e == R_em
assert exact_channel_R2(s_e, R_e, *_probe(K, entry=(0, 0)), K) == \
    rfl.exact_channel_R2(rfl.exact_vertices("octahedron", K), R_em,
                         *rfl._probe(K, entry=(0, 0)), K)
K5 = solid_field("icosahedron")
assert K5 == rfl.solid_field("icosahedron") and K5 != sp.QQ
assert exact_vertices("icosahedron", K5) == \
    rfl.exact_vertices("icosahedron", K5)

# scalars: the word layer;  twojobs: the coin;  decker: the circuits
assert _parse_word("X F†") == rsc._parse_word("X F†")
toks_nb, Rs_nb = exact_draw("T")
toks_md, Rs_md = rsc.exact_draw("T")
assert toks_nb == toks_md and Rs_nb == Rs_md
assert np.array_equal(coin_rotations("octahedron"),
                      rtj.coin_rotations("octahedron"))
d_nb, live_nb, W_nb = decker_vertices("dodecahedron")
d_md, live_md, W_md = rdk.decker_vertices("dodecahedron")
assert np.array_equal(d_nb, d_md) and live_nb == live_md
assert np.array_equal(W_nb, W_md)
print("spot-checks: the rebuilt primitives agree with the modules\n")

# (3) The ledger's own derivation, re-derived and pinned. Read-only: the
# writers live in write_fragments, behind the __main__ guard -- and the
# tripwire in the first cell would refuse them anyway.
_spec = rfr.spec_sheet()
assert set(_spec) == set(SOLIDS)
print(f"spec_sheet: {len(_spec)} ledger rows re-derived; ladder and bar"
      f" agreement pinned\n")

# ... then the entry point's own report, end to end.
class _Tee(io.TextIOBase):
    def __init__(self, *streams):
        self.streams = streams

    def writable(self):
        return True

    def write(self, text):
        for st in self.streams:
            st.write(text)
        return len(text)

    def flush(self):
        for st in self.streams:
            st.flush()


_buf = io.StringIO()
_stdout = sys.stdout
sys.stdout = _Tee(_stdout, _buf)
try:
    ri.main()
finally:
    sys.stdout = _stdout
print(f"\n[ok] blocks in the report above: {_buf.getvalue().count('[ok]')}")

currency: 9 files match their build-time sha256 pins



spot-checks: the rebuilt primitives agree with the modules



spec_sheet: 5 ledger rows re-derived; ladder and bar agreement pinned

=== 0. canonical data ====================================================
[ok] npz vertices match the exact symbolic solids; elements = (1/V)(I + n.sigma)
[ok] symbolic gates match gates.npz projectively (X, Z, F, Phi)
[ok] group_2X unitaries project onto exactly the group_X rotations (+-U pair up)
[ok] T < O and T < I as rotation groups (2T sits inside both 2O and 2I)

=== 1. two protocols, two scalars (findings 1 + 2 + 6) ===================
generic probe noise:  T_zz = 0.620000   tr(T)/3 = 0.720000

solid          grp   R1 (randomized-projective)     R2 (twirled-native)
------------------------------------------------------------------------
tetrahedron    T       undefined (no antipodes)   depol, kappa = 0.720000
octahedron     O        depol, kappa = 0.620000   depol, kappa = 0.720000
cube           O        depol, kappa = 0.620000   depol, kappa = 0.720000
icosahedron    I        depol, kappa = 0.620000   dep

  octahedron     O    Q                 identity in T      identity in T   max|diff| = 5.6e-16


  cube           O    Q(sqrt3)          identity in T      identity in T   max|diff| = 7.8e-16


...

  dodecahedron      5         5        2   tau/sqrt(tau+2)            no          0.720000000         1.0e-15
[ok] every rotation realizable over an ATLAS-GENERATED gate set has
     SO(3) entries in Q(sqrt5); all five reorientations leave it, so
     none is an atlas word. Adjoining T makes exactly two exact --
     the tetrahedron's and the cube's, as T+, the gate the circuit
     figures draw -- while the other three stay barred over every
     thesis gate set by K_R and Lemma 6. Each inexact one is still
     approximable to any accuracy, Clifford+Phi being universal.
     Costs nothing new: a reorientation is fixed and
     g-independent, and no dilation was exact in any pose anyway
[ok] and it costs the ESTIMATOR nothing: R2 returns tr(T)/3 = 0.720000 in
     Decker's pose as in ours, all five solids, offset still zero.
     The pose is a question of naming, not of function -- what needs
     it is the claim that prepending U_g rotates OUR vertices by g
[ok] and that is a THEOREM

  icosahedron        60   [-0.3333, +0.8697]       0.0117        7250.0x        297.1x


  dodecahedron       60   [-0.3333, +0.9004]       0.0249        1614.0x        184.1x
[ok] and the proviso is priced: every coset member's induced labelling
     is exactly depolarizing at kappa = tr(m T)/3, zero offset -- the
     two-list channel, element by element, noiseless AND at the probe
     noise, the setting that tells belief from device. The columns
     above are the NOISELESS family tr(m)/3, centred on ZERO with
     RMS 1/3 (<kappa> = 0, <kappa^2> = 1/9; at the probe the laws
     are 0 and |T|_F^2/27, all four asserted), so applying the
     relabelling WITHOUT the gate turns a free correction into a
     shot premium the RMS reads as 9x. An understatement twice
     over: 1/kappa^2 is convex, the mean premium column running
     18x to 297x, and the noisy family comes far closer to the
     kill -- min |kappa| = 6.1e-05, a 2.7e+08x premium. The kappa
     RANGES bound Table D.4's five (asserted there); premia they
     do not bound: D.4's dodecahedron undercuts every 

  solid             atlas   decker   generic
  --------------------------------------------
  tetrahedron       9.000   15.000   16.1154
  octahedron       27.000   12.000   16.3269
  cube              9.000   15.000   16.1154
  icosahedron      16.200   16.200   16.2000
  dodecahedron     16.200   16.200   16.2000

[ok] the tail weight -- the single-Pauli estimate's fourth moment --
     is 27<w_x^4 + w_y^4 + w_z^4> over the swept directions, asserted
     as the exact protocol average at every pose, draw (T included),
     Pauli and state: state-free, axis-free, draw-blind. Two SOS
     identities put it in [9, 27], floor exactly the eight cube
     directions, ceiling exactly the six Pauli axes -- the atlas pose
     is the tetrahedron's and cube's GLOBAL optimum and the
     octahedron's global pessimum, and D.2's unreoriented numbers are
     pinned: 9 -> 15 (tetrahedron and cube, equal in every common
     pose), 27 -> 12 (octahedron), the 5-designs immovable at 81/5.
[ok] the oc


[ok] blocks in the report above: 48


## Closing notes

**What was shown**, finding by finding:

| finding | where | the one line |
|---|---|---|
| 1 | §1, §1a, §1b | the estimator-channel factor identifies the protocol — $T_{zz}$ vs $\operatorname{tr}T/3$, both *identities* in the noise; and the one mistake priced differently: a constant carried across protocols is a bias, everything else a premium |
| 2 | §1, §1a | the SIC is not the price of the twirl; the ancilla is — R2 twirls the tetrahedron exactly, R1 cannot even be defined for it |
| 3 | §3 | direction ($K_\mathbb{R}$) and weight ($\mathbb{Z}[1/2]$) between them convict all five; the octahedron survives only through the coin, and *deterministic* is three bans, not one |
| 4 | §2 | realize and twirl are independent properties of the drawn set; the bars cross at the icosahedron, the sweep is exhaustive over every finite subgroup of $SO(3)$, and the $C_3$ coin is the witness that Schur's hypothesis is necessary |
| 5 | §3 cont. | Decker's circuits rebuilt in his own outcome order; a skipped relabelling twirls to one overlap $\kappa$ — a $1/\kappa^2$ premium, never a bias — and the tail weight is an exact pose functional with the published pose extremal |
| 6 | §1c | gate noise separates the protocols as an *order* in $\gamma$: the twirled-native $Z_0$ residual is second order on the $2T$ draw — a fact needing both the protocol and the draw's prefix multiset — where every projective row stays linear |

**The through-line, once more.** A protocol is which maps you average: the object handed to the
group average decides the scalar (§1), the drawn set's orbit and its irreducibility decide the
two jobs (§2), the gate field decides which maps exist exactly (§3), a mismatch between
believed and performed maps twirls to one overlap (§3 cont.), and noise correlated with the
draw is precisely where "which maps you average" stops being well-posed — and the failure is
itself exactly priceable (§1c).

**Where this sits in the repo.** The suite backs Section 5.2.3 (the implementation ledger),
Appendix D (exactness, Decker's circuits, outcome order) and Appendix F.3 (the estimator
channels, the two-bars sweep, the $C_3$ witness); the definitional seam is Chapter 4's *Two
Randomized Implementations* subsection, whose dashed-box figure is the twirled-native picture.
The numerical shadow study that consumes the
two channels lives in `shadow_experiments.py`, with its own walkthrough
(`shadow_walkthrough.ipynb`) — the two notebooks meet at the two-protocol distinction and
otherwise divide the labor: the variance landscape, dual optimization and Monte Carlo live
there; the exact scalars, the two jobs, the obstructions and the pricing live here.

To run the production suite end to end (writes the six LaTeX fragments; deterministic):

```
cd code && uv run randomized_implementations.py
```

To regenerate this notebook after editing the builder (never edit the .ipynb directly):

```
cd code && uv run python _build_randomization_walkthrough.py
uv run --with jupyter --with nbconvert jupyter nbconvert --to notebook --execute --inplace randomization_walkthrough.ipynb
```